# RSNA Knee — RGB 2.5D voisins DICOM, 320

**Shard 7/12** (CPU, internet off). **1778 series**, ~10.6 Go.

UID **figés** depuis `data/crop130_rgb25d/manifest_rgb25d_proto_part*.csv` (6 slots v20b, contraste exact). Même recette que le proto 224, seul le resize passe à **320** :

1. crop physique **130 mm** (`PixelSpacing`) ou crop centre 70 %
2. bande profondeur **20–80 %**
3. linspace **24** centres dans la bande
4. pour chaque centre `k`, **`k-1, k, k+1` dans le volume DICOM**
5. resize **320×320**, uint8, percentiles 1–99

Shape `(24, 3, 320, 320)`, canaux R=k-1, G=k, B=k+1.

Fichiers dans `/kaggle/working/` :
- `volumes_crop130_rgb25d_320_part7.zip`
- `manifest_rgb25d_320_part7.csv`
- `RECIPE.json`

Download : `python -X utf8 kaggle_DICOM_exctract/download_rgb25d_320.py` → `data/crop130_rgb25d_320/`.

In [ ]:
SHARD_ID = 7
N_SHARDS = 12

import os, io, gc, time, json, traceback
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
import pydicom

# --- Recette (doit etre reproduite a l'identique au submit) ----------------
CROP_MM = 130.0
CROP_REL = 0.70          # fallback si PixelSpacing absent
BAND = (0.20, 0.80)      # profondeur : on ignore les 20 % de chaque bord
N_SLICES = 24
IMG_SIZE = 320
PLANES = ["Sagittal", "Coronal", "Axial"]
N_WORKERS = 4
# --------------------------------------------------------------------------

KAGGLE = os.path.exists("/kaggle")
OUT_DIR = Path("/kaggle/working") if KAGGLE else Path(".")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("KAGGLE", KAGGLE, "| SHARD", SHARD_ID, "/", N_SHARDS)
print("out", OUT_DIR)


In [ ]:

def find_comp_dir() -> Path:
    if not KAGGLE:
        p = Path("../data/csv")
        if (p / "train_series.csv").exists():
            return p
        raise FileNotFoundError("train_series.csv introuvable en local")
    data = Path("/kaggle/input")
    print("contenu /kaggle/input :", sorted(x.name for x in data.iterdir()))
    for base in [data / "competitions", data]:
        if not base.exists():
            continue
        for p in sorted(base.iterdir()):
            if (p / "train.csv").exists() and (p / "train_series.csv").exists():
                return p
    raise FileNotFoundError("train.csv + train_series.csv introuvables sous /kaggle/input")

COMP_DIR = find_comp_dir()
TRAIN_SERIES_CSV = COMP_DIR / "train_series.csv"
TRAIN_SERIES_DIR = COMP_DIR / "train_series"
if not TRAIN_SERIES_DIR.exists():
    for cand in COMP_DIR.iterdir():
        if cand.is_dir() and (cand / "train_series").exists():
            TRAIN_SERIES_DIR = cand / "train_series"
            break
print("COMP_DIR         ", COMP_DIR)
print("TRAIN_SERIES_CSV ", TRAIN_SERIES_CSV, TRAIN_SERIES_CSV.exists())
print("TRAIN_SERIES_DIR ", TRAIN_SERIES_DIR, TRAIN_SERIES_DIR.exists())


In [ ]:
import json
jobs = json.loads('[{"study":"1.2.826.0.1.3680043.8.498.10004945927472656027199792075652399585","series":"1.2.826.0.1.3680043.8.498.87374594163896635184044513927478147680","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10009639203170750274174707434356622764","series":"1.2.826.0.1.3680043.8.498.18313433830014722992362938871973097711","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10018749256316287542884435387067561101","series":"1.2.826.0.1.3680043.8.498.94288268297326706009195326058014280624","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10021288557171866794624317186288618513","series":"1.2.826.0.1.3680043.8.498.28399217752101799876825340333611472859","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10029168398037940681915299760479434938","series":"1.2.826.0.1.3680043.8.498.77941154390263872263584692780262100897","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10039686090738454718354181337513293981","series":"1.2.826.0.1.3680043.8.498.75650187531194852023871501055652723295","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10059200627175320481920009076697103801","series":"1.2.826.0.1.3680043.8.498.10709481291181856834545021650181350504","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10066945398485042741116657288959323922","series":"1.2.826.0.1.3680043.8.498.26822768456111767807589411073221552859","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10077803906105306230229342168184970097","series":"1.2.826.0.1.3680043.8.498.25907331750467633841871804423069889144","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10087350795575215080051569638065974347","series":"1.2.826.0.1.3680043.8.498.22083524570800997810756116671664380573","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10088707526476055079083082931629995449","series":"1.2.826.0.1.3680043.8.498.35491444433021315625274489410742816825","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149","series":"1.2.826.0.1.3680043.8.498.67495435033389756963800730529101229430","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10096216489317602387766193675918767557","series":"1.2.826.0.1.3680043.8.498.36636715272183714971214450630447386449","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10100718559177860518999368876392117659","series":"1.2.826.0.1.3680043.8.498.48773532541265941245766532225595532276","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10120841513315425967980213160839994882","series":"1.2.826.0.1.3680043.8.498.10689105701012441571090306678235579195","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10123751739767586025744123799948754343","series":"1.2.826.0.1.3680043.8.498.23233651069805614070606540865878503229","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10128954897472795975323353442305671586","series":"1.2.826.0.1.3680043.8.498.72186653999779217286768831822800036101","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10132169498032330070492330913783717864","series":"1.2.826.0.1.3680043.8.498.23318582185131593644573850889259806152","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10135587809070718435691939321937314104","series":"1.2.826.0.1.3680043.8.498.53787212012756349650425557865396727826","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10144735099702632881322032871591573824","series":"1.2.826.0.1.3680043.8.498.10241162724241290501988646649924895142","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10147717839726734017869499799005084725","series":"1.2.826.0.1.3680043.8.498.11660423496805057440817786799305499257","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10149967031679082348901055550415519790","series":"1.2.826.0.1.3680043.8.498.58365219889749827895260020422275070750","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10155611590208164695256621422536643320","series":"1.2.826.0.1.3680043.8.498.10462470055652637110111618026173286727","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10159551371305936155888341171022480536","series":"1.2.826.0.1.3680043.8.498.57572582751178344543602830837717150471","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10162720374291180196973998234794414415","series":"1.2.826.0.1.3680043.8.498.68527550681985449152635877451104473534","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269","series":"1.2.826.0.1.3680043.8.498.10560750949481691706699585852544080976","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10181291283712008544475048457589664787","series":"1.2.826.0.1.3680043.8.498.20901820955413693658696381284317208013","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10189392162056736163704940747328177364","series":"1.2.826.0.1.3680043.8.498.10052052142602846814375429051826138830","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10197515493414564227930855210542221039","series":"1.2.826.0.1.3680043.8.498.12980681846875147350649586386360411702","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10201197020583611347810070328854760801","series":"1.2.826.0.1.3680043.8.498.10924378837370335196075121591337549347","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10205592316907266136938526795826751183","series":"1.2.826.0.1.3680043.8.498.95536512734425776602520192097860235501","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10211034309703220222647821582770542129","series":"1.2.826.0.1.3680043.8.498.12382624143288964745669413836123425766","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10218285921271312931891912812808052375","series":"1.2.826.0.1.3680043.8.498.10278488061269623205313792244369792044","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10239418182819270860819533058081041576","series":"1.2.826.0.1.3680043.8.498.10583953208255984456583770488538610073","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10240237086492917349933491458808329326","series":"1.2.826.0.1.3680043.8.498.11557742576782526485538415707244897156","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10247930792182001003497189021027466833","series":"1.2.826.0.1.3680043.8.498.35608671283339823163310389372564194553","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10252163031705285141538200795896549203","series":"1.2.826.0.1.3680043.8.498.50274783247595581317710304976396918185","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10255067212695306715722782826234429766","series":"1.2.826.0.1.3680043.8.498.80258559168539047134052592979729222310","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10262024906324456863812446480254600405","series":"1.2.826.0.1.3680043.8.498.77870308243951449193536192960049285228","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10271567891603078238024189515805908561","series":"1.2.826.0.1.3680043.8.498.88773960893971809432349013133060938098","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10274956886032850418742861294277331382","series":"1.2.826.0.1.3680043.8.498.14620362972151612241393582775369984795","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10284263494252700630857173843779724077","series":"1.2.826.0.1.3680043.8.498.60337584896692774071317547908359669895","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10299396087685174806129850512856451005","series":"1.2.826.0.1.3680043.8.498.98733876567133671733705168596969228500","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10304242666970626179776896768140580106","series":"1.2.826.0.1.3680043.8.498.49359067841584277580273128692124052324","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10309477218703786877612946261946268132","series":"1.2.826.0.1.3680043.8.498.23816706223453344020224240407904285751","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10312350714164739081126996907250465738","series":"1.2.826.0.1.3680043.8.498.43835505875637030850523826237531274924","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10321461243762453573005633063753741725","series":"1.2.826.0.1.3680043.8.498.77210428091457580811449886040415971239","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10322724865012212544754724404226583315","series":"1.2.826.0.1.3680043.8.498.41731945035369097344055161439283312539","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10326352669696598934883805147731461341","series":"1.2.826.0.1.3680043.8.498.93295672727859736252059817008094548300","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10336221220793312503236159680373829726","series":"1.2.826.0.1.3680043.8.498.83503881961115842501214336878691245141","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10339654093023509742663983374586998753","series":"1.2.826.0.1.3680043.8.498.12388880255344817546094433590949886135","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10351168527688244230248760520733020605","series":"1.2.826.0.1.3680043.8.498.82660432235369063200246536571426554029","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10352011938574988727803498915608467179","series":"1.2.826.0.1.3680043.8.498.71481674289694068040563564652813938771","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10354576637002647863048798607011636355","series":"1.2.826.0.1.3680043.8.498.31190305534499887879578927967668398461","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10371162371589200772388569623043753617","series":"1.2.826.0.1.3680043.8.498.99634594348259466263340155185440986840","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10382074558506016193900889176239337801","series":"1.2.826.0.1.3680043.8.498.20691354159833082824942005460248838148","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10387227472110643414333498953513810066","series":"1.2.826.0.1.3680043.8.498.60966757792454483156872454637373917656","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10391178814963564201551003358372148419","series":"1.2.826.0.1.3680043.8.498.10246041763579315727223026907770171189","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10401813253995823397883215400000450724","series":"1.2.826.0.1.3680043.8.498.72268490913167361974905048615088362113","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10407865232593182855530878478701744409","series":"1.2.826.0.1.3680043.8.498.12474275447357619608811072193870370014","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10415616591430643294142401482460086713","series":"1.2.826.0.1.3680043.8.498.93761036745921182938722262656172838889","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10419958821528108176389641942168280311","series":"1.2.826.0.1.3680043.8.498.61189977904749664748013799164178420681","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10422130685453254363700365908336246863","series":"1.2.826.0.1.3680043.8.498.31569966894343555996835020273730383643","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10425131723456547056365404016184663035","series":"1.2.826.0.1.3680043.8.498.66389429997752823126060289767580088241","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10432568348356281020515782178064869805","series":"1.2.826.0.1.3680043.8.498.12635235030218343952200612891471327955","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10440858363080910322920496514558347561","series":"1.2.826.0.1.3680043.8.498.49548360622394955484230715065559446306","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10445893053733106322562086357169418409","series":"1.2.826.0.1.3680043.8.498.64362202088851071176361650586336727570","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10455589411958266725060942213473374667","series":"1.2.826.0.1.3680043.8.498.82718264404554461684654552703241367378","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10459901081962705567398560859229437818","series":"1.2.826.0.1.3680043.8.498.57698081383786123075761666916775177472","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10475397432011699698577978597417345091","series":"1.2.826.0.1.3680043.8.498.89877019199518818360776237810431890278","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10491012711700179118010977263805828326","series":"1.2.826.0.1.3680043.8.498.86569521473200743674638383158989750021","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10495855084258410825408867079175646590","series":"1.2.826.0.1.3680043.8.498.11354363484248194921935995988873903569","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10501068619333941198079851978532746117","series":"1.2.826.0.1.3680043.8.498.12493182928542097939674663986508624073","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10511712617129216774539441890488742560","series":"1.2.826.0.1.3680043.8.498.12361757366518514640950703946106070894","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10519480654942067141173172019311858213","series":"1.2.826.0.1.3680043.8.498.90274661971014130279728236437861685259","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10522585237725235866013737575867341946","series":"1.2.826.0.1.3680043.8.498.12280532479271882879927457123485279806","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10532708800468111850607752497432656694","series":"1.2.826.0.1.3680043.8.498.79194636454186667828827034186274647568","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10534260101860670685224894870019533364","series":"1.2.826.0.1.3680043.8.498.45939124979073210014761999075082439194","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10537503354035115750949427337500909327","series":"1.2.826.0.1.3680043.8.498.11162278440235061156649335972703209840","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10540980111087702906123811593906850156","series":"1.2.826.0.1.3680043.8.498.87394329414885499542217779516402336859","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10541215772904358938061268226270989801","series":"1.2.826.0.1.3680043.8.498.22679858897482905112967772671834515184","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10546313646515875953943156661172963953","series":"1.2.826.0.1.3680043.8.498.31617320902614840944588365071145572530","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10562412345413239960204081248510238736","series":"1.2.826.0.1.3680043.8.498.10637925259235402880926451594821918687","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10578013533179563448221888658506023084","series":"1.2.826.0.1.3680043.8.498.11740123088468535877813662758014013229","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10581718331223731732815674582085184169","series":"1.2.826.0.1.3680043.8.498.81217025910453057976632765369170319920","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10590148272001480024086117269799629816","series":"1.2.826.0.1.3680043.8.498.12893540131612552385520143262809458137","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10593694750925047629619718537436931628","series":"1.2.826.0.1.3680043.8.498.53631112165553270415150928056532501925","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10600654987896381983520458496207741798","series":"1.2.826.0.1.3680043.8.498.46335445306861916646378221650123177514","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10601711635138681269270505152909539290","series":"1.2.826.0.1.3680043.8.498.56465902341771590564563360216157387433","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10603952065748622499432940419853732241","series":"1.2.826.0.1.3680043.8.498.51570111973883658903465110193466174048","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10619486897279334206912764495067756449","series":"1.2.826.0.1.3680043.8.498.10210143438193051659495078906186389466","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10624417936475721846835761375334601839","series":"1.2.826.0.1.3680043.8.498.64152736236636358732211791712060887002","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10627019153481144392726933328057818971","series":"1.2.826.0.1.3680043.8.498.28124515570704621518023486971054581630","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10631238417904652761254062374051500506","series":"1.2.826.0.1.3680043.8.498.14649173688130704678046626214854302802","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10642286343421893916348249679230632066","series":"1.2.826.0.1.3680043.8.498.56406882535116717485888469266420101844","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10652685418462611085839874321267337079","series":"1.2.826.0.1.3680043.8.498.68878097637247569886753663424554119119","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10653432159060774337155662807644609967","series":"1.2.826.0.1.3680043.8.498.54842403377479658884843637751388620856","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10657506540109137829814058249895525831","series":"1.2.826.0.1.3680043.8.498.58360552985708904581527567828634759720","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10661650555947788912511478165000737040","series":"1.2.826.0.1.3680043.8.498.11464573324892252519776750278801583796","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10674673641872871321890995010616205087","series":"1.2.826.0.1.3680043.8.498.10589773355337960239999538700855853626","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10676253869668012791900504544106495589","series":"1.2.826.0.1.3680043.8.498.73605674428796963864569326270525013012","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10688670034893539180306173203903350286","series":"1.2.826.0.1.3680043.8.498.14377454035672531016707705866710049593","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10702284568670902901846767166679293654","series":"1.2.826.0.1.3680043.8.498.60905968473340934081273587531767518789","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10713285016170190922777298047146228811","series":"1.2.826.0.1.3680043.8.498.40318198588842957904914662885189718567","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10726624821670417064567481545750543282","series":"1.2.826.0.1.3680043.8.498.48375381448360416658266043083132278745","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10731612202208916303411280624156291171","series":"1.2.826.0.1.3680043.8.498.32602838704493599489891594646184286493","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10738640862162795323010965765443420384","series":"1.2.826.0.1.3680043.8.498.11603583478171666391988029860405578582","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10740857795874468664161171813136103519","series":"1.2.826.0.1.3680043.8.498.41552951872646988864818439719132778517","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10751346650129497348457481341700957445","series":"1.2.826.0.1.3680043.8.498.68807755933173332550575330327965707308","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10759288702627176691392048851622294054","series":"1.2.826.0.1.3680043.8.498.29207067593231135761452941387938250460","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10761658878615069270304502752374858610","series":"1.2.826.0.1.3680043.8.498.42794650916372748532274310771028934789","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10765617676329232046176300833850901740","series":"1.2.826.0.1.3680043.8.498.10513954148039077177115334652467620949","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10770877369291591553624215762368014357","series":"1.2.826.0.1.3680043.8.498.89182250861658901173713714247465053547","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10782963448708192069959366583555573334","series":"1.2.826.0.1.3680043.8.498.93932450946202434961134120403937290988","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10790689419760453870458170966064168002","series":"1.2.826.0.1.3680043.8.498.13196229038046547812325914980778118433","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10803283287658556463524649793955055077","series":"1.2.826.0.1.3680043.8.498.26614509798408297901598406965975999411","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10818980124981334596808438908811623528","series":"1.2.826.0.1.3680043.8.498.83078924514793351456898312674961371184","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10829579914809383212986717269028170263","series":"1.2.826.0.1.3680043.8.498.24206883750835725474267122972011106945","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10839126901366636822676830560774505865","series":"1.2.826.0.1.3680043.8.498.16028398713204851979555884248055167953","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10843463954622076609489312662891432839","series":"1.2.826.0.1.3680043.8.498.12352363451108476518578557153043561806","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10850742579000782417673172155504794301","series":"1.2.826.0.1.3680043.8.498.12859779983531406715959381504195216623","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10852517092524464338210061305792376636","series":"1.2.826.0.1.3680043.8.498.83438674730395968108311222520048994831","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10855290714402117092990651362697310270","series":"1.2.826.0.1.3680043.8.498.41520151419916152354291355144223852311","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10857771747813858957718058322384884551","series":"1.2.826.0.1.3680043.8.498.27538996055361001853140012053959794921","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10876177342063734507664188801203192617","series":"1.2.826.0.1.3680043.8.498.95339242223095541677742084077459103376","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10884987036948550523442514147204321748","series":"1.2.826.0.1.3680043.8.498.44862956891135202690079439236238171850","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10895521387894821238259709507311341758","series":"1.2.826.0.1.3680043.8.498.10604327527704399604659858980879308272","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10903003118372337016190215211789983264","series":"1.2.826.0.1.3680043.8.498.51656157398949785953609134734854724999","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10918580785296934228715514366408269433","series":"1.2.826.0.1.3680043.8.498.75956320112168352803495396962001496825","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10920269679387171659375091645187772303","series":"1.2.826.0.1.3680043.8.498.12131368119588876413423658108051384278","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10927716750119165925963563081261605359","series":"1.2.826.0.1.3680043.8.498.89827309875773348755043568828895340223","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10931111730564885182398163970692599398","series":"1.2.826.0.1.3680043.8.498.26932998385518304169714246705708146332","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10937181824204907065071760564745305722","series":"1.2.826.0.1.3680043.8.498.10942063984791437227126025396937194812","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10937585701850306334458420860276069606","series":"1.2.826.0.1.3680043.8.498.10095455231504836138651094411629998816","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10939964887585312209861601414669494409","series":"1.2.826.0.1.3680043.8.498.17092201598597623706892346405907008969","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10960770430559425421609885784366918575","series":"1.2.826.0.1.3680043.8.498.10920292118738972767932956839796666228","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10965573028231658307306828073340279415","series":"1.2.826.0.1.3680043.8.498.50905657380016423651314099700434756010","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10979105661314213697886164132444977106","series":"1.2.826.0.1.3680043.8.498.72628343424479460231481681550855067629","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10992875360571368325079907908221899699","series":"1.2.826.0.1.3680043.8.498.31629019479581566439867036817099886717","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10994848378662155711176898659205016445","series":"1.2.826.0.1.3680043.8.498.91393122109295131690826856203956291400","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10995980880227536691827030731644023480","series":"1.2.826.0.1.3680043.8.498.12226445555670114269292196022250758556","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10998405409474759941744305665540376339","series":"1.2.826.0.1.3680043.8.498.19550189169403601256315168917698236155","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11010004366885474301784150582039137111","series":"1.2.826.0.1.3680043.8.498.11487572300133788385308231981000908764","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11014091506864318694851337553197477906","series":"1.2.826.0.1.3680043.8.498.33461354072153205323492520728233466358","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11041694833778404465582610810664391088","series":"1.2.826.0.1.3680043.8.498.11739682885624380260977612130921260965","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11043926695471016727738735328317975282","series":"1.2.826.0.1.3680043.8.498.47188465852587472932120640771222759256","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11050604958275820465127935558416408008","series":"1.2.826.0.1.3680043.8.498.12590319592916935451647381743955124528","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11057883757434391643878168168845592475","series":"1.2.826.0.1.3680043.8.498.61876027219887729773572031773349239180","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11064977170395595339882840534354229547","series":"1.2.826.0.1.3680043.8.498.13001145534603442580051474039790076823","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11077406555624926104628200252710914930","series":"1.2.826.0.1.3680043.8.498.11256846246967080105651850424090484197","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11084010833221855396433326769153950681","series":"1.2.826.0.1.3680043.8.498.79133388869368115960594153743468086036","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11091958567356981151034727948765010881","series":"1.2.826.0.1.3680043.8.498.12466335535631629397426499102507026078","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11103894265181149878864749992746236768","series":"1.2.826.0.1.3680043.8.498.12397913202248914249735968965992515016","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11113442398569287228475456000446876768","series":"1.2.826.0.1.3680043.8.498.72837477588833076304380664243061833103","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11122485005317809892694992204053124213","series":"1.2.826.0.1.3680043.8.498.80795826659493581342048955940347900186","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11125718814639515292482773418372267821","series":"1.2.826.0.1.3680043.8.498.62895631316143783707883133619871572680","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11129751904857491327872353316480050290","series":"1.2.826.0.1.3680043.8.498.17184381764505093448899692678937741582","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11135115047200279264914390528987153419","series":"1.2.826.0.1.3680043.8.498.10724674925723039372459249968487720765","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11145632111083286399683467139301564425","series":"1.2.826.0.1.3680043.8.498.21627738508661115912350573245413168879","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11150193626240150189232364321568347937","series":"1.2.826.0.1.3680043.8.498.10002940759939066728493680169552264940","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11150751959089942633754454715290410758","series":"1.2.826.0.1.3680043.8.498.74873484105862584337633713948951610947","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11156884483511690263238401598901614096","series":"1.2.826.0.1.3680043.8.498.28050656540004598218144922456274580449","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11173951743276817877673759975853741079","series":"1.2.826.0.1.3680043.8.498.39052017269888352215719870500492091286","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11183915112115196028799391678356464392","series":"1.2.826.0.1.3680043.8.498.70457245305382710844626032511291571927","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11191015483427903713800997920940844550","series":"1.2.826.0.1.3680043.8.498.11679150105104895223191059145084301087","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11201589885283493748455257854104771618","series":"1.2.826.0.1.3680043.8.498.72434435618394605050500688091703849325","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11208107516750574657838495761140109258","series":"1.2.826.0.1.3680043.8.498.11535036162687450931477186514531920455","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11210361780779474361817208167009080083","series":"1.2.826.0.1.3680043.8.498.10602393129644143537460780391051950474","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11214158717183172156320144105486247892","series":"1.2.826.0.1.3680043.8.498.72244717270224760742473823464741496380","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11218782547268547937641080283066300678","series":"1.2.826.0.1.3680043.8.498.74633703413567939026409102133697609755","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11223691171233118900160265287691267884","series":"1.2.826.0.1.3680043.8.498.63289082175389710480186400130444128708","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11229566047228169850473554486278603754","series":"1.2.826.0.1.3680043.8.498.73191925856001710745765340736006595766","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11232674915186872899189440191805868376","series":"1.2.826.0.1.3680043.8.498.60351267758086136370083440693326875368","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11254890841796835340329521299102281678","series":"1.2.826.0.1.3680043.8.498.54693214791742305799729873249645791777","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11257869433929994221899994675448655047","series":"1.2.826.0.1.3680043.8.498.84838559811084146139953356627934540579","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11264140845869814478794346490273061499","series":"1.2.826.0.1.3680043.8.498.12526253688221742090393737934770369439","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11270845320057097450047912107461684885","series":"1.2.826.0.1.3680043.8.498.49230156672091147857185307812943623674","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11276732020240247755618413418003401291","series":"1.2.826.0.1.3680043.8.498.30212141265827025730928882921333295087","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11282162920338602001456704018617788412","series":"1.2.826.0.1.3680043.8.498.10105591872058793263500730458675281734","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11287173314232027941677698384842764593","series":"1.2.826.0.1.3680043.8.498.70145905462607039582649572508546278983","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11294885816514924203337081832122619605","series":"1.2.826.0.1.3680043.8.498.83055526077143861632768088611616297225","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11296867854260286881418279496022384461","series":"1.2.826.0.1.3680043.8.498.29329937261374717801584945585043213228","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11302750434639976191333636496428935632","series":"1.2.826.0.1.3680043.8.498.10447659878973149169146676644258233554","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11312798102268182343541500463824266929","series":"1.2.826.0.1.3680043.8.498.83914092883219099799906239909489985421","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11316091466685605758061778045311969672","series":"1.2.826.0.1.3680043.8.498.42339255088864017754953976990254217332","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11334486185870324597335039163116965763","series":"1.2.826.0.1.3680043.8.498.10542909974403142221156925524724670711","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11338962847958062668991874160027887372","series":"1.2.826.0.1.3680043.8.498.24071887307280849324861258057505619728","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11347466940974045182979684641300336177","series":"1.2.826.0.1.3680043.8.498.22747613938276941045436303216443925377","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11353976221682532422630231842008146722","series":"1.2.826.0.1.3680043.8.498.53691548752849718343100880011565958153","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11371400503360468962117579439852064607","series":"1.2.826.0.1.3680043.8.498.51344680205362236271529507335820310780","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11383056981606331444758741778014516431","series":"1.2.826.0.1.3680043.8.498.10006895116086398819258067200723106151","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11391095418955817124325938917797687542","series":"1.2.826.0.1.3680043.8.498.10321430993083802116370841364637789681","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11395349736725035422736613041202251830","series":"1.2.826.0.1.3680043.8.498.12430834926865897110723738398925233805","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11406304618685790128709325242250648417","series":"1.2.826.0.1.3680043.8.498.65940512798093508218353096326561421222","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11414215447313658600502835138277811260","series":"1.2.826.0.1.3680043.8.498.37299546703720579002544711086463626444","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11425019165071014217290822852865654127","series":"1.2.826.0.1.3680043.8.498.11466737744680233493064074769420938642","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11432815311029789072253610823554308183","series":"1.2.826.0.1.3680043.8.498.39260630187842720868186116009214864297","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11440217845998767853986099058965644463","series":"1.2.826.0.1.3680043.8.498.31757378628519633442151154578131906399","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11446264194433135282793848490754001623","series":"1.2.826.0.1.3680043.8.498.13477821137105966308865850388927075306","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11451448731901996559834481853733404099","series":"1.2.826.0.1.3680043.8.498.34724967703489117616196448676960784391","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11455388290952699607973723923118848769","series":"1.2.826.0.1.3680043.8.498.55616261330690566232810198938234119112","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11465586454035959390661499120751232686","series":"1.2.826.0.1.3680043.8.498.97652344491521224545569350837294667429","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11468812519969607938527116557104410121","series":"1.2.826.0.1.3680043.8.498.10432362621902023665864217533961155874","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11472441320199233244785757534656853558","series":"1.2.826.0.1.3680043.8.498.12875961949937140860970436891232467648","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11482325994550095339373157256174349858","series":"1.2.826.0.1.3680043.8.498.37127844993226545594859252540144804098","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11495133447993934376048707763538781434","series":"1.2.826.0.1.3680043.8.498.18544054840500116518354918436460943038","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11498239309891927151456629915927889165","series":"1.2.826.0.1.3680043.8.498.46037288570040165902536183541193660228","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11498845504486569595587386245872511981","series":"1.2.826.0.1.3680043.8.498.13320093771440004091620999492486878143","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11506642732332295874076832453198901115","series":"1.2.826.0.1.3680043.8.498.34328256310728711709802843288762731461","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11508340006652282767645009271305921426","series":"1.2.826.0.1.3680043.8.498.83325208863509227757343799387783474478","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11516498203727058146656616413699979433","series":"1.2.826.0.1.3680043.8.498.40676256367536615983903698604743421040","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11521971289581057766555206729274639736","series":"1.2.826.0.1.3680043.8.498.80632889771498183131586144728221464441","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11532517984322509215429681648257299355","series":"1.2.826.0.1.3680043.8.498.80448379008254264685596871857648343975","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11542298537902846156898976374522470463","series":"1.2.826.0.1.3680043.8.498.10717477909616914451702415154648326728","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11546127923715190914128277599849591183","series":"1.2.826.0.1.3680043.8.498.10003271121171039993431767719983720312","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935","series":"1.2.826.0.1.3680043.8.498.54680010975301762365724523486752133418","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11551770225828320378339882849245348148","series":"1.2.826.0.1.3680043.8.498.11026492033167762926022857314699213576","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11569355546217846610131708883985793793","series":"1.2.826.0.1.3680043.8.498.48751422533360543879191309592621599991","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11571057273590444914079281238002921868","series":"1.2.826.0.1.3680043.8.498.95704270523479269993684934821071974806","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11574667600340420151274680172753922967","series":"1.2.826.0.1.3680043.8.498.14898846956313122843979296062194477187","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11583098598455610459898849451117846552","series":"1.2.826.0.1.3680043.8.498.12836562282954350249815325022804955052","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11587989960888698769912899464470116193","series":"1.2.826.0.1.3680043.8.498.85994247566759075758631825768482069121","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11592056376890319271545710970155310824","series":"1.2.826.0.1.3680043.8.498.75264314665270055165448243683093011221","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11596665780617412984766822528862741584","series":"1.2.826.0.1.3680043.8.498.59737098021306262062080408528674327977","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11598606533748611505510113033734692457","series":"1.2.826.0.1.3680043.8.498.12539209216853648435447839932307240011","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11604298807403466096392316117803661479","series":"1.2.826.0.1.3680043.8.498.30431957309315496099699200441661930063","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11610176134207131408691324676304640501","series":"1.2.826.0.1.3680043.8.498.70641613365721375716530328239526275844","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11619883329923252065839671636992770849","series":"1.2.826.0.1.3680043.8.498.18940148402452640516761304245785128456","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11629659616217352895144469980856259097","series":"1.2.826.0.1.3680043.8.498.61301729143628841277262615707112508439","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11642228346306175407288158335830376197","series":"1.2.826.0.1.3680043.8.498.86534532774698828538347214464805645858","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11646465814751330102194657154003685957","series":"1.2.826.0.1.3680043.8.498.11460924239080835444169029101704692687","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11652570369213315569275956969935617843","series":"1.2.826.0.1.3680043.8.498.46826733311182369695951889337742990956","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11655692245049710970881619360721342623","series":"1.2.826.0.1.3680043.8.498.86523229844915557167347552043392501400","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11669120316759278775792190599036288128","series":"1.2.826.0.1.3680043.8.498.72532103512964934280912614223273992429","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11679044101697033975478370934058085847","series":"1.2.826.0.1.3680043.8.498.75982154634826001013122931259477686033","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11682243736539231948790371952356165171","series":"1.2.826.0.1.3680043.8.498.86786423842981088369173119201432799188","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11691245281415120685024733512803575288","series":"1.2.826.0.1.3680043.8.498.64155602568234058230084629874707213572","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11703195077261884009442719578918319962","series":"1.2.826.0.1.3680043.8.498.11951410571993631727328256263731489675","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11716605112302071535035323666308656381","series":"1.2.826.0.1.3680043.8.498.57058234349160438072366167424518849269","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11719502398697390809777045891822044637","series":"1.2.826.0.1.3680043.8.498.12199571219898345032499941911016057026","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11722185671504770687850569082348083612","series":"1.2.826.0.1.3680043.8.498.74183351281975903116616312135489875849","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11723945299462627737346907048285143063","series":"1.2.826.0.1.3680043.8.498.81940254545482381780509148762610535985","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11738872955471637985947490962985266753","series":"1.2.826.0.1.3680043.8.498.42483423179626352333123131027017583045","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11753067309373104365558887398441364695","series":"1.2.826.0.1.3680043.8.498.21456504595809477059991552936166915764","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11770690655120964721059321709525979950","series":"1.2.826.0.1.3680043.8.498.12462812072477282160071382561598285270","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11775352515236862473778507708524782015","series":"1.2.826.0.1.3680043.8.498.12064601221370534775260799570794911143","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11783597160745641720078978406968189488","series":"1.2.826.0.1.3680043.8.498.30212169777744737761408860886885611210","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11786296135006283515679011075884245663","series":"1.2.826.0.1.3680043.8.498.10649111812707443649174033656845687186","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11791567830371683660578052737702299124","series":"1.2.826.0.1.3680043.8.498.68643217968240655781515731558585826660","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11796547950032543541071896156852778697","series":"1.2.826.0.1.3680043.8.498.43218047672708375610002931518881307309","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11799272823743145498601173910069737855","series":"1.2.826.0.1.3680043.8.498.28915416749927980184608336323050983756","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11805131119810457174456565128522037991","series":"1.2.826.0.1.3680043.8.498.11749604345236998441896591671330835060","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11812607848520072826145426247813670653","series":"1.2.826.0.1.3680043.8.498.90839057325123336961676613046845819537","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11823120042104793954689778719229868108","series":"1.2.826.0.1.3680043.8.498.30193094320355160032572764489152992149","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11848828525319749834876185415845381479","series":"1.2.826.0.1.3680043.8.498.72321838271145897891422945622193723047","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11860731961337988611847638299761007498","series":"1.2.826.0.1.3680043.8.498.76609325392565675716609703152017291919","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11867198902144602395671904591496233129","series":"1.2.826.0.1.3680043.8.498.25383346010987155758613889082531008817","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11871142388984150456249558619289234011","series":"1.2.826.0.1.3680043.8.498.11087112225767928247326918517101775781","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11880385031047315524154761508473409762","series":"1.2.826.0.1.3680043.8.498.12974976071771723439083951988476811466","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11892908237449514945544380023820626319","series":"1.2.826.0.1.3680043.8.498.20689283201380418378456141393846511923","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11900355716044995266104285198121445744","series":"1.2.826.0.1.3680043.8.498.10483320688984193643148990402764550752","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11904156585987383639134967886361403369","series":"1.2.826.0.1.3680043.8.498.42174711069316276116447691870606135278","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11906266535122698149967624908585010734","series":"1.2.826.0.1.3680043.8.498.12707869102152332708128286340532048587","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11912276256810863127132294167870461707","series":"1.2.826.0.1.3680043.8.498.36527637895436502936080932053431010040","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11915937982684988073644209606907169581","series":"1.2.826.0.1.3680043.8.498.11990303844095911723613574201435309071","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11920531053482540304184657218030242597","series":"1.2.826.0.1.3680043.8.498.40972426236379970149838803059529864743","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11924999889245769866426422681253524920","series":"1.2.826.0.1.3680043.8.498.74002872999569953084975734172590388602","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11934483804567594363179973225449686890","series":"1.2.826.0.1.3680043.8.498.99943316054853328343693184612516474561","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11935893582662912138522962760478513770","series":"1.2.826.0.1.3680043.8.498.13314661652844499619208264014218803660","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11944762815616453054528091721553484857","series":"1.2.826.0.1.3680043.8.498.12743791973858428468419388618731575337","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11957476321850340968244508976719889277","series":"1.2.826.0.1.3680043.8.498.40739724050720909659438071267170308202","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11964826300307458382604479841253457304","series":"1.2.826.0.1.3680043.8.498.49787467196263954195491350192145953599","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11979034558288454336812613626594510186","series":"1.2.826.0.1.3680043.8.498.13307121793006748421276365930095466341","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11980805687949950504930975019161587970","series":"1.2.826.0.1.3680043.8.498.59709328614706078319035061357204928503","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11988551795896617815207749665102049499","series":"1.2.826.0.1.3680043.8.498.88640073391454482711021647583068353884","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11999008448480965933785988718780864948","series":"1.2.826.0.1.3680043.8.498.27345086264014171753183296821228480171","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12006027138204983547873134458107233878","series":"1.2.826.0.1.3680043.8.498.13063540258534010310833149828148100962","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12009816058271421855131593296751372723","series":"1.2.826.0.1.3680043.8.498.12803797679917596161780541170187114263","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12015406188790105426257083591807220760","series":"1.2.826.0.1.3680043.8.498.35158562113700164790346857127069554610","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12031315219947933845048665397149229969","series":"1.2.826.0.1.3680043.8.498.12200790263213257473780367148211033582","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12041392088066705782960819598258792338","series":"1.2.826.0.1.3680043.8.498.13576296382891878329131320984691662145","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12045300645580015230470991493790324857","series":"1.2.826.0.1.3680043.8.498.89477719356641464259204716245237092644","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12054099347723802635102026371218576198","series":"1.2.826.0.1.3680043.8.498.96253562072655557568736397607422130304","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12064822036518837577612019712902340953","series":"1.2.826.0.1.3680043.8.498.11591440518165842045256561646017601624","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12068674252637046261029360607215822017","series":"1.2.826.0.1.3680043.8.498.61928105796790378289636235386111839682","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12092744899746753313905432642371563786","series":"1.2.826.0.1.3680043.8.498.10094481510632980581263452989140586022","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12100254887364913022980951672127944515","series":"1.2.826.0.1.3680043.8.498.11166215740865134881264727284339566626","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12106775587428595108771304868101362618","series":"1.2.826.0.1.3680043.8.498.10229461273051471855769341718676699886","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12117377043894182951345966247899775612","series":"1.2.826.0.1.3680043.8.498.51987433068243891642318504752850457915","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12131866860554871145836544571379633595","series":"1.2.826.0.1.3680043.8.498.72843623161682532155055544583973175044","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12135456587022560662140325717612726990","series":"1.2.826.0.1.3680043.8.498.17306288816329005490547175759554574305","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12148236440353424874546037143134438088","series":"1.2.826.0.1.3680043.8.498.68047002709194479083668253625780447578","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12153755823296803843819361055386265595","series":"1.2.826.0.1.3680043.8.498.58313871254257520649846427940724728250","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12160065317167766993207036411434801297","series":"1.2.826.0.1.3680043.8.498.54867471675306519739918143956196526345","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12166081590548185636776183599317715045","series":"1.2.826.0.1.3680043.8.498.50932285257124209622887179608479515207","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12170753305862973517190756418865414984","series":"1.2.826.0.1.3680043.8.498.44882031459773206417390045952094910240","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12171790864123519428649395041090102217","series":"1.2.826.0.1.3680043.8.498.65335795196301010887764705007311439504","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12178921459424917661210327999672203211","series":"1.2.826.0.1.3680043.8.498.60066159591433036782201381872265555952","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12182309401920763498667366352681074302","series":"1.2.826.0.1.3680043.8.498.63270357751551632749928459904181619295","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12189863023888657759738596583562992408","series":"1.2.826.0.1.3680043.8.498.12667524418905322526199073495762335597","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12195007606427082866449558290311905939","series":"1.2.826.0.1.3680043.8.498.58348381923413404285380382910816251361","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12201598444731259697380878112783653796","series":"1.2.826.0.1.3680043.8.498.97275248012232603280407861861864544530","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12207279366369605175255203969558548753","series":"1.2.826.0.1.3680043.8.498.62361438414944499321911491745681398850","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12219658385745696203053079500246399481","series":"1.2.826.0.1.3680043.8.498.12076916237486783142828488311320147953","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12225975251304101975065271521710735735","series":"1.2.826.0.1.3680043.8.498.27413579634878639384873293492189166357","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12238753439953831519247789198174756615","series":"1.2.826.0.1.3680043.8.498.58607832141470727812909466054320117251","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12241413500128553133597113709525575494","series":"1.2.826.0.1.3680043.8.498.10135085758541254302051254552798844163","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12243764506921404196245986681613075136","series":"1.2.826.0.1.3680043.8.498.94012746347110951192829574692121650585","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12246086238999179953298261549112671014","series":"1.2.826.0.1.3680043.8.498.65473676243394803983455585055500052141","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12257119852150088212071954292747252243","series":"1.2.826.0.1.3680043.8.498.40357522486441610718951357500453507247","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12259896940707176389739553110492954399","series":"1.2.826.0.1.3680043.8.498.33163598074959725523606028296686854450","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12264607993101717201407955391437731134","series":"1.2.826.0.1.3680043.8.498.70580320138882335581359049970679681858","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12266133889557145601371064833389800065","series":"1.2.826.0.1.3680043.8.498.72447854118107975453192165872731818271","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12271616365050313084031931009756086606","series":"1.2.826.0.1.3680043.8.498.82198588269469596301626147151884422789","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12273490180961458167930039242826008249","series":"1.2.826.0.1.3680043.8.498.93840835099060919952207184570565410473","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12275265458583331749472987758046974406","series":"1.2.826.0.1.3680043.8.498.30938350831896438734570620832790828420","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12283662652762881783845275662580546217","series":"1.2.826.0.1.3680043.8.498.15497568644816449893299629007711660138","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12286894057773964720268768529280193085","series":"1.2.826.0.1.3680043.8.498.31507597040821724882343837781120713269","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12295284437770380354725533247473017729","series":"1.2.826.0.1.3680043.8.498.12438199441699505482213713357048208815","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12296624595750602443033689366743784387","series":"1.2.826.0.1.3680043.8.498.88474815127210924512249333726905287290","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12303161232396569915032951761728884199","series":"1.2.826.0.1.3680043.8.498.96982277543421879775103365218170896628","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12305733049746414359748178404911549573","series":"1.2.826.0.1.3680043.8.498.63325345005266608168681716085332282519","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12314729913655626716039376606973474150","series":"1.2.826.0.1.3680043.8.498.35693576675468519447604417889589773848","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12328313245396930020974053819810033945","series":"1.2.826.0.1.3680043.8.498.12552816999364196481086536135924864248","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12332727488899501354972751168284397151","series":"1.2.826.0.1.3680043.8.498.40960424643598953356797520228556418180","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12345064031074704673835101507426902246","series":"1.2.826.0.1.3680043.8.498.11593487296511399971518972412371871945","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12348352904452887226227118450588511578","series":"1.2.826.0.1.3680043.8.498.80380541195740357933440904517351617271","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12349656752404066719917550972333542508","series":"1.2.826.0.1.3680043.8.498.62552876476810839250127337265964007652","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12355011301785011370279005103251815187","series":"1.2.826.0.1.3680043.8.498.83752118669508671525661701549889592240","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12361144153852256276011948786791830435","series":"1.2.826.0.1.3680043.8.498.72586029709660159260032319415222018303","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12373923338033469765189220784089350380","series":"1.2.826.0.1.3680043.8.498.78174542057083477904817357290592940062","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12377852033407536077897684568488365785","series":"1.2.826.0.1.3680043.8.498.98816980885924158883748748256320668364","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12381325071746339154385415220972069782","series":"1.2.826.0.1.3680043.8.498.25615944480855946447418088853558132822","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12386255059403233724886571725482313876","series":"1.2.826.0.1.3680043.8.498.60270109925472586697812455986709798981","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12387241932914995417493041212958089910","series":"1.2.826.0.1.3680043.8.498.98939755784986100778677781047172526465","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12401147012438969595202076588471168894","series":"1.2.826.0.1.3680043.8.498.58417970180473955731895891498015828255","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12408186961945791320086301369036829266","series":"1.2.826.0.1.3680043.8.498.12481643756267578806813797181545106372","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12408933790879905193765468926022393696","series":"1.2.826.0.1.3680043.8.498.11852485293457601081821731783158498131","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12411396124304166984011280826756258628","series":"1.2.826.0.1.3680043.8.498.28090738624279563158564136754019126673","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12431272351851438889289190308550697909","series":"1.2.826.0.1.3680043.8.498.47853477090343302978616310125389794125","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12437154858167700438432674983971817217","series":"1.2.826.0.1.3680043.8.498.12295516339656347668358363129583139044","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12438775479300781065075835270622538230","series":"1.2.826.0.1.3680043.8.498.89281316089358354662751096236237904944","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12462498156569316578044317782899399656","series":"1.2.826.0.1.3680043.8.498.36759244224719029639557499883340106304","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12471165461877621368840037945454256612","series":"1.2.826.0.1.3680043.8.498.10937545226452402209953788178013903669","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12481913908649439189738319460342358360","series":"1.2.826.0.1.3680043.8.498.68679035769857752150328601150140828035","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12484700311993345120015691857089576898","series":"1.2.826.0.1.3680043.8.498.37652801049726327088077975804750515169","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12489752598467506094826904655029553699","series":"1.2.826.0.1.3680043.8.498.71148584135522859547641866831806814684","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12491395125580387847235487841392723348","series":"1.2.826.0.1.3680043.8.498.62633647128517686732526942255662697405","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12491796764641306046176352066215240074","series":"1.2.826.0.1.3680043.8.498.53915388308848728651396649604946112901","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12499208731138180053966892599139021038","series":"1.2.826.0.1.3680043.8.498.75384548520934882381834381324245185709","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12505035424093604269515328931488770819","series":"1.2.826.0.1.3680043.8.498.63349206091681534493848463157511154322","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12509944331769453135724043962410129357","series":"1.2.826.0.1.3680043.8.498.96267937060512592536468209149687770975","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12515322634403509077334846234426636136","series":"1.2.826.0.1.3680043.8.498.81980343483726909524748314196563218300","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12520203592477518476890010185037272575","series":"1.2.826.0.1.3680043.8.498.29962298392163346690892006137522736081","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12525752631743097782762806859112218620","series":"1.2.826.0.1.3680043.8.498.27387769442800453087650720646138631832","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12526876471494728147635430816990398219","series":"1.2.826.0.1.3680043.8.498.65451759599757170958393929870810282074","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12533105585687352326867082376911240942","series":"1.2.826.0.1.3680043.8.498.76451732602010115046316815859970940697","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12548176380550230498070609559486512004","series":"1.2.826.0.1.3680043.8.498.33939675720645877980469070043814815702","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12553010776519233077752150342664160826","series":"1.2.826.0.1.3680043.8.498.46993622175764294061913442689518216662","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12559687473583953266143897736224215680","series":"1.2.826.0.1.3680043.8.498.11209791107272132375571600839462553277","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12568337973304376222940287737691898395","series":"1.2.826.0.1.3680043.8.498.11453474609693491411481319484170329879","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12573518342767030025851169814440863785","series":"1.2.826.0.1.3680043.8.498.24491045766744527802311172212104148481","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12576055921678075813718486704978381993","series":"1.2.826.0.1.3680043.8.498.12468151665433405700142648074533919173","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12580232188757225504172709417466515184","series":"1.2.826.0.1.3680043.8.498.73669439454573927434413980181936688322","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12598627296066406747921546933391681918","series":"1.2.826.0.1.3680043.8.498.12542962550577487218207639148838778052","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12606657226568558340797193167488111973","series":"1.2.826.0.1.3680043.8.498.11119341733412446515084456525158577841","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12607959449008110729566106776800628990","series":"1.2.826.0.1.3680043.8.498.11603610699738380277691196583030115296","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12616376042564380071463081913949255574","series":"1.2.826.0.1.3680043.8.498.10119877605610345723650124821808676412","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12625250919597111075072308824921101918","series":"1.2.826.0.1.3680043.8.498.93499784853315228118087797530879927191","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12630423349626505319792444053874011718","series":"1.2.826.0.1.3680043.8.498.11287686010310004028790853805188095015","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12632129640405661298623288977632682289","series":"1.2.826.0.1.3680043.8.498.74757921427363601986059420657357268632","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12638205949145637289277744267473965719","series":"1.2.826.0.1.3680043.8.498.75381115404126115427550513788295532981","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12644400073937544865441360809610886635","series":"1.2.826.0.1.3680043.8.498.65863813217668797676126888356481043450","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12655324218817524683138594937984186560","series":"1.2.826.0.1.3680043.8.498.12333243653567225070464487318959009065","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12660901572462562819575002681831361427","series":"1.2.826.0.1.3680043.8.498.11760324404234552512717810680723909508","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12673561338227595483297605409292531492","series":"1.2.826.0.1.3680043.8.498.11553405159777863689670566736306559429","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12676359576477452816896379524598090410","series":"1.2.826.0.1.3680043.8.498.10254942249279065554772144913175790575","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12689874339887047433232628649886917274","series":"1.2.826.0.1.3680043.8.498.96890652071244179096837879553120405908","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12692579161952587215276483587686562922","series":"1.2.826.0.1.3680043.8.498.16039203716534428721362391451704827210","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12699751030439008569144814819343161608","series":"1.2.826.0.1.3680043.8.498.91028179241954504405145162989698265414","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12701097867143790763644826659974337046","series":"1.2.826.0.1.3680043.8.498.40273546120339682103520700810993029752","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12715067947657150279232238280712066972","series":"1.2.826.0.1.3680043.8.498.71509411020065032297795580007373289117","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12715402081094788880283826571710131940","series":"1.2.826.0.1.3680043.8.498.77698072758721900503087517142148652294","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12719397512102459853075227679498459688","series":"1.2.826.0.1.3680043.8.498.12777369621227214871625415514639485882","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12724250000565012779853884787450109956","series":"1.2.826.0.1.3680043.8.498.58049465768330056339094271226886958318","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12728397615513347615111391662798012012","series":"1.2.826.0.1.3680043.8.498.66712454421994557841658185638320497258","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12748840878706905309050091576449488666","series":"1.2.826.0.1.3680043.8.498.12883203121343673998318129032817153828","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12761906443846433581503942530016459978","series":"1.2.826.0.1.3680043.8.498.14291780066865918861778131501245637967","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12765434812409174436720333864278447770","series":"1.2.826.0.1.3680043.8.498.12198929533903559909257720277117404741","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12769887673556938425710414718385118283","series":"1.2.826.0.1.3680043.8.498.21268592281403468212523448565638280816","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12775105605170844918620305197829216665","series":"1.2.826.0.1.3680043.8.498.48538401348561834951578942042733219829","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12778920060586364664953350981285170949","series":"1.2.826.0.1.3680043.8.498.10673733312144054809521154335603954483","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12785792890626338008419452097492192088","series":"1.2.826.0.1.3680043.8.498.10268452316210685465409481764694458412","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12792431109288033212787919118858331455","series":"1.2.826.0.1.3680043.8.498.93804920158244919751249004889701430975","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12800130397185143653180509040746675358","series":"1.2.826.0.1.3680043.8.498.97186045578564657185767774473333066408","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12801643268939133682891906705186571010","series":"1.2.826.0.1.3680043.8.498.11467138572238808015835936247474612292","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12810074901642984174782853829149619043","series":"1.2.826.0.1.3680043.8.498.91805621513263013803427329028671197766","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12817753008281742621758605755308036035","series":"1.2.826.0.1.3680043.8.498.99539756327932848524695023359508344468","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12819437506700585448407291050085063174","series":"1.2.826.0.1.3680043.8.498.49216817851303353787463557006764927659","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12826828527176716109949704557150950675","series":"1.2.826.0.1.3680043.8.498.35966200053008338101076692225426902984","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12836702174475748025129787471026627810","series":"1.2.826.0.1.3680043.8.498.44357036509591886172365463703299102617","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12839498544183083639548298117995964432","series":"1.2.826.0.1.3680043.8.498.61675475947746645267096236307560563625","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12866937766198610064474201436305532400","series":"1.2.826.0.1.3680043.8.498.97898051069977037635917213089536999526","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12899995703999073968090810947564075608","series":"1.2.826.0.1.3680043.8.498.12248377125874894815957688921897210017","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12907457183137517669505067507754394039","series":"1.2.826.0.1.3680043.8.498.13154147843314049498943996088209786640","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12914677763148740041755977075799271057","series":"1.2.826.0.1.3680043.8.498.77977085552752693336338818123352099360","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12919303621342250989179219420795277559","series":"1.2.826.0.1.3680043.8.498.11051728213528410962644621606043711689","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12936296973456045538701035923586653502","series":"1.2.826.0.1.3680043.8.498.11799162913698887642924984283363237911","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12946163460972553911280889205685745877","series":"1.2.826.0.1.3680043.8.498.67231058360731704826012220003158433442","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12950639892345812884896649629409199101","series":"1.2.826.0.1.3680043.8.498.44796095970793352574801613631296382568","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12957252711278416684461267069498115130","series":"1.2.826.0.1.3680043.8.498.53858009156292860013287393958538418075","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12961116776122838195555132309258963046","series":"1.2.826.0.1.3680043.8.498.17287343795993424126714286407343352758","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12968535455086298222684461157825070395","series":"1.2.826.0.1.3680043.8.498.89259340599455154858465090590047033149","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12978510157202852202776899910529174803","series":"1.2.826.0.1.3680043.8.498.43918311649410688736808710698060757556","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12981413356524881858859912294078498486","series":"1.2.826.0.1.3680043.8.498.84095758560676663257604683352969735004","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12985683732818911488433522743768046254","series":"1.2.826.0.1.3680043.8.498.10197406329378895802824089362268502213","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12987428318535758055829100395092244687","series":"1.2.826.0.1.3680043.8.498.25436230354593544363765529105739150949","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13003520962744192762179786723724282013","series":"1.2.826.0.1.3680043.8.498.74989053149568740613462098105538572795","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13015502057686077045104167339480215528","series":"1.2.826.0.1.3680043.8.498.12848380969831342811941906401637520973","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13017199074506191657941905498813292773","series":"1.2.826.0.1.3680043.8.498.57484643038908797729957610070683075713","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13024250150960117282590397343184827466","series":"1.2.826.0.1.3680043.8.498.50608858775560107633339329024568924801","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13026024768908635083636086393662414870","series":"1.2.826.0.1.3680043.8.498.61992770843090304582685359043926387818","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13027024577729659065187419038453181852","series":"1.2.826.0.1.3680043.8.498.92224062944715430670668365578972932958","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13030662056306690969977140713179461363","series":"1.2.826.0.1.3680043.8.498.13343335005280674432151597070745675665","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13042847643086819857036458569632034999","series":"1.2.826.0.1.3680043.8.498.36584924119474853355968201136773577101","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13062438493893996542058869514390908492","series":"1.2.826.0.1.3680043.8.498.25484236096408888143212202840479097735","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13068696303639759696458130704554709416","series":"1.2.826.0.1.3680043.8.498.32558366128431520401582800342339696862","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13073387272228394257811861905056479438","series":"1.2.826.0.1.3680043.8.498.40936048782480894276386907111545767326","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13078009378379325641560321462188164769","series":"1.2.826.0.1.3680043.8.498.84048555440710188609230339719630180341","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13078834627392793401892111941627273421","series":"1.2.826.0.1.3680043.8.498.57283704638178551711162894709349326824","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13087770868099843948960419968576882974","series":"1.2.826.0.1.3680043.8.498.10604248550704666758315578628322210744","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13092094949394905924922534008771583745","series":"1.2.826.0.1.3680043.8.498.10167200812081167194695893058933952686","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13093751810869039645959119469231437894","series":"1.2.826.0.1.3680043.8.498.10204828226735016951911606441844270159","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13102058401501684064017656674827527261","series":"1.2.826.0.1.3680043.8.498.41715308062773997112298082337191582269","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13113623966675790946816351311283701141","series":"1.2.826.0.1.3680043.8.498.94973890485663085556322243126694073584","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13114693814781737495494700902699619138","series":"1.2.826.0.1.3680043.8.498.11078391824707156256667045221298575703","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13118325043138522704531128725514354285","series":"1.2.826.0.1.3680043.8.498.24527335630913571117790563428892537632","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13126070481622692281046898445918832841","series":"1.2.826.0.1.3680043.8.498.11355036913228175347927738480670587315","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13128352910661672966143173193241171100","series":"1.2.826.0.1.3680043.8.498.48022855756395172469243984776724972699","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13130534736322715018024786213131944607","series":"1.2.826.0.1.3680043.8.498.44166923967318019605816195127591082752","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13133755791254469039849289186598412660","series":"1.2.826.0.1.3680043.8.498.38717663391214025312415217552130128759","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13136367866260644484250529322937554927","series":"1.2.826.0.1.3680043.8.498.78333392102446551542026077125873888286","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13137510961466386836387239510287919268","series":"1.2.826.0.1.3680043.8.498.58093230756615667680322422914145584471","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13143446880514535170343243531129864116","series":"1.2.826.0.1.3680043.8.498.79900795875504187456539764286733526744","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13147249459968468500804358644901642270","series":"1.2.826.0.1.3680043.8.498.83149498469258321819240383673190917220","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13150794288945922409613136250303663502","series":"1.2.826.0.1.3680043.8.498.65978958109591928776660369663749725233","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13152690996267297597178587721720788387","series":"1.2.826.0.1.3680043.8.498.80288601539091456296896825008836334486","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13158227087287104386790063637592969032","series":"1.2.826.0.1.3680043.8.498.77870994370136264276715433497765887145","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13162284470388627411596358366248494046","series":"1.2.826.0.1.3680043.8.498.80175879543003769761248714963054008916","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13163727085668231782866823012535268245","series":"1.2.826.0.1.3680043.8.498.59760620017954521062433966722919625217","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13172295193446466338347273836250944090","series":"1.2.826.0.1.3680043.8.498.11816979017889394122622179037841091009","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13173781757294564816839795307893228067","series":"1.2.826.0.1.3680043.8.498.13266113323584223845785375677885973278","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13179169600887013946657470830682778717","series":"1.2.826.0.1.3680043.8.498.79286036863293446467099186249745722244","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13197409499722328227876152940338160369","series":"1.2.826.0.1.3680043.8.498.55046450590441167179627693419353258034","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13203246712933982189680087013566455075","series":"1.2.826.0.1.3680043.8.498.64399940603841492985384841175876335697","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13211838518836535713919214396985811434","series":"1.2.826.0.1.3680043.8.498.10097905774691777164144172695541191375","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13214267416673058581197714011913363750","series":"1.2.826.0.1.3680043.8.498.27182895313784726362989759743919405010","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13224933005220329091792052916905776921","series":"1.2.826.0.1.3680043.8.498.23680779345797355296925948896209776287","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13229563888378977350299319159748823477","series":"1.2.826.0.1.3680043.8.498.43353043189005297693739187999034033656","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13240749152416621711688053465035013133","series":"1.2.826.0.1.3680043.8.498.74502786875827768095712443321214647747","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13245409855393847268063261203253424169","series":"1.2.826.0.1.3680043.8.498.55386825172965871367730634899314792122","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13261595972961358158990410740461308054","series":"1.2.826.0.1.3680043.8.498.11258082983363144680367706368050427309","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13262583253812134425581922368942846684","series":"1.2.826.0.1.3680043.8.498.58591879482030606305221164015439934261","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13267708980958529833316446819156908381","series":"1.2.826.0.1.3680043.8.498.62979738499345273638547359273229957077","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13273890409523605456488117581706115075","series":"1.2.826.0.1.3680043.8.498.52643156795052538751112587273963251774","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13283411144589724334502427926341646116","series":"1.2.826.0.1.3680043.8.498.10358509904880230295174940106159090229","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13288244567806030859347115222642155186","series":"1.2.826.0.1.3680043.8.498.10652736364293089319556962858920810518","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13291943370279119939499960846843879156","series":"1.2.826.0.1.3680043.8.498.59001493619550394751349444281465434364","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13295633458001436592005068637719195786","series":"1.2.826.0.1.3680043.8.498.12080497948424033713300354197200039785","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13312139694018789912028129989483016875","series":"1.2.826.0.1.3680043.8.498.11400782415751700171066250191715503242","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13319886927127591971322856062062654146","series":"1.2.826.0.1.3680043.8.498.89746318685803843829578890262973792821","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13328029743559327467301971917273397645","series":"1.2.826.0.1.3680043.8.498.17844953298239780229395061591482758725","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13332521365069604440230185026865466121","series":"1.2.826.0.1.3680043.8.498.12055367955612629516072456427213720380","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13341507176140340924941654608476073030","series":"1.2.826.0.1.3680043.8.498.44230712575747346565636948253868649990","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13353409340862821090624127486679097099","series":"1.2.826.0.1.3680043.8.498.96432067833669175866385142045415223186","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13358702230966891064267033765095124604","series":"1.2.826.0.1.3680043.8.498.10925493581370554406046856240821845251","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13363856251607349805111249441972460575","series":"1.2.826.0.1.3680043.8.498.88300806492307731886089323471870045974","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13367740087756379568826046059252678700","series":"1.2.826.0.1.3680043.8.498.43539042322278850379152656352376288247","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13371001997511542011416510364359588752","series":"1.2.826.0.1.3680043.8.498.11903225676331231416324385919270392385","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13380735447520796902327272696579348141","series":"1.2.826.0.1.3680043.8.498.88547968405327450531435358663069741720","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13390551515761269433524925969561757962","series":"1.2.826.0.1.3680043.8.498.16461395995890654801732939361491373912","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13399254446600380420635633478666850540","series":"1.2.826.0.1.3680043.8.498.99694215230506486531273970834179063255","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13405120554981098042415743645998258385","series":"1.2.826.0.1.3680043.8.498.28459331255422762523729996092665538340","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13440909104569690641314836192184434461","series":"1.2.826.0.1.3680043.8.498.67444882683650722867470959861998262399","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13460078631594032326006554321483799928","series":"1.2.826.0.1.3680043.8.498.60214300287991996805221529403707733458","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13461073016327661159616957998589287085","series":"1.2.826.0.1.3680043.8.498.11797638894657482652184434877162113247","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13575953449958751965232580597585303226","series":"1.2.826.0.1.3680043.8.498.41459724417591746229429905870525495688","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13616982533395291590655841043722813973","series":"1.2.826.0.1.3680043.8.498.13164331664225578396963600067456858402","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13695989890112408005495264068064110179","series":"1.2.826.0.1.3680043.8.498.74054904766766646021456028951089876610","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13829111847660458141056077151781118782","series":"1.2.826.0.1.3680043.8.498.85077853547283438633527636998613965900","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13915367834258386094001181500560177470","series":"1.2.826.0.1.3680043.8.498.12692171625978580887408996263449876872","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13972325309264527611989585341636140156","series":"1.2.826.0.1.3680043.8.498.12973128317252488053068430368217377887","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14032903253348079919799627243690484791","series":"1.2.826.0.1.3680043.8.498.79298927630274883864926029902340677149","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14066537811671722419109228137841995847","series":"1.2.826.0.1.3680043.8.498.31810851891310164077942481246933231611","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14188540532926067083914263496492255704","series":"1.2.826.0.1.3680043.8.498.84093625140582342897132806935014270526","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14229598279617168024212382336641104878","series":"1.2.826.0.1.3680043.8.498.65398367761464042224711101871711676497","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14325424371811230497993759959862738838","series":"1.2.826.0.1.3680043.8.498.33600417488528211722382181756137971750","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14377917653321813299237274871837695586","series":"1.2.826.0.1.3680043.8.498.76547263596795599733052703525282664450","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14453847272947043746856632410254555895","series":"1.2.826.0.1.3680043.8.498.18815081422387639323225684246489692020","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14479913217228604289584726096114526268","series":"1.2.826.0.1.3680043.8.498.98943769060135035557564522572082560274","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14495028403937132710814780922642512791","series":"1.2.826.0.1.3680043.8.498.76360421277415163214911201560328593365","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14679140388328648617386311272361396555","series":"1.2.826.0.1.3680043.8.498.79793931028834332994653881470539193867","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14744714658078107007743063095375646143","series":"1.2.826.0.1.3680043.8.498.40655348372143194358057981255910863011","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14806659119591146961784818305031900863","series":"1.2.826.0.1.3680043.8.498.86667844187689107729696119115600750550","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14850534390650184292244380816212794996","series":"1.2.826.0.1.3680043.8.498.88817187996972272488499883573416767335","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14959732862523218997149041503831001633","series":"1.2.826.0.1.3680043.8.498.10238510351172287350607701042165261741","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15024602758408626305833210011882949904","series":"1.2.826.0.1.3680043.8.498.41640913449995578444734468136396516720","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15090991392324602559982474708414427621","series":"1.2.826.0.1.3680043.8.498.43139481526057827759463403693172438049","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15133818248365103660456294775831248835","series":"1.2.826.0.1.3680043.8.498.92888655827648731540938657011863480608","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15219824433491621460467738389345697247","series":"1.2.826.0.1.3680043.8.498.34440766542095340943601371010038306304","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15337067581747365729183643142512558411","series":"1.2.826.0.1.3680043.8.498.63253291414098670777859615755392504651","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15376535778246978844757121776560020722","series":"1.2.826.0.1.3680043.8.498.12835800391210044470191409671735754238","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15444316810805746453392291780593934328","series":"1.2.826.0.1.3680043.8.498.33364887288322975124910865050360075542","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15486463042236628073590077209793828847","series":"1.2.826.0.1.3680043.8.498.97273992356717292477917173293903588110","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15593638897292057356864060466120253309","series":"1.2.826.0.1.3680043.8.498.52431053584784925784510783478723202039","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15599161312748548709240395406977042093","series":"1.2.826.0.1.3680043.8.498.76342813892044754015425891445215653541","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15744573360087157139672078937679122840","series":"1.2.826.0.1.3680043.8.498.13034320834999580491576146537744494676","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15863325224755490561125835019647303556","series":"1.2.826.0.1.3680043.8.498.16544345985341256132013484669618139566","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16051798246004436152597344963060310360","series":"1.2.826.0.1.3680043.8.498.79501700565763159532180859145561748723","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16101435445990180482665354366724730305","series":"1.2.826.0.1.3680043.8.498.81558154581772814011325313329220618042","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16189833935983604051983090678102023832","series":"1.2.826.0.1.3680043.8.498.10399528271327674865542203421287959885","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16225091905018667993201414191711776943","series":"1.2.826.0.1.3680043.8.498.11766454514088375278654972147039106579","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16318078762749420171833135145622117452","series":"1.2.826.0.1.3680043.8.498.99994312049398560882626966131758380565","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16370303687278282143772106415664171358","series":"1.2.826.0.1.3680043.8.498.75589397677488361101487917782202376323","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16483470188406187000599644150600279596","series":"1.2.826.0.1.3680043.8.498.90671170865166704834829117273269882521","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16534078612126477228267765508485297367","series":"1.2.826.0.1.3680043.8.498.22573284671221511626087980913282556984","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16550793987525699514242241133391355835","series":"1.2.826.0.1.3680043.8.498.96199537833610289211554206557511886297","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16657553974040122719239506008948241297","series":"1.2.826.0.1.3680043.8.498.40496677455884885455212025966294117499","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16691383858003633107641297162554868325","series":"1.2.826.0.1.3680043.8.498.37944427189089590410022856910641991728","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16709696133689803501344038926512501779","series":"1.2.826.0.1.3680043.8.498.45183914324618822777911577223854321279","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16774450022563283092653506225603012645","series":"1.2.826.0.1.3680043.8.498.54705918668756234568745906287331257410","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16845451648760595922758166878121325861","series":"1.2.826.0.1.3680043.8.498.65967672811001322628298472116171892705","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16864649493021092944977766326505053153","series":"1.2.826.0.1.3680043.8.498.11150151270793694543000471670755998026","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17056997047488992455301411194938263817","series":"1.2.826.0.1.3680043.8.498.77204010186872806229566786808516714070","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.17074746747536999918446136673966046653","series":"1.2.826.0.1.3680043.8.498.93091956460930177002702300920496812723","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17093025979426350838724553020541643719","series":"1.2.826.0.1.3680043.8.498.15715352332658090211140159532071370152","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17146868851921828837871671912815843012","series":"1.2.826.0.1.3680043.8.498.52760581789212074327876967389507438322","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17241550277273207658303011467222594165","series":"1.2.826.0.1.3680043.8.498.76159443837356724396339192103965802743","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17274490471295558668613733667390457509","series":"1.2.826.0.1.3680043.8.498.10196860452519015050280231146565470277","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17415936504684075438800895036740822145","series":"1.2.826.0.1.3680043.8.498.11024728298424910464618487626247858765","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17517950899674150719043097786728121608","series":"1.2.826.0.1.3680043.8.498.70707497081106240283634429428618633597","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17596657287050550465463272509976106854","series":"1.2.826.0.1.3680043.8.498.51392661961382110245623782712119849664","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17663005209256897892831319144686413853","series":"1.2.826.0.1.3680043.8.498.25990912898291973424653981311112859217","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17741907690913585163443565102293192244","series":"1.2.826.0.1.3680043.8.498.45777799229604650046722969156748188196","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17769942698581399165275319502954100325","series":"1.2.826.0.1.3680043.8.498.72457850081559011575332841348131640584","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17844546765907321649997094867791102711","series":"1.2.826.0.1.3680043.8.498.11261251202884490916489531421908521865","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17865613300681542407687000381594067366","series":"1.2.826.0.1.3680043.8.498.11696399044106161138066918328501448254","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17967978373588651520812399064902037467","series":"1.2.826.0.1.3680043.8.498.64558542580508981890017293145643271134","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18080511167840608448833484000937072258","series":"1.2.826.0.1.3680043.8.498.18885597732916044100685289158733339751","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18113123512696698441392990570166201548","series":"1.2.826.0.1.3680043.8.498.26731660227058256885776530185400581594","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18175450083338316799857698898702333009","series":"1.2.826.0.1.3680043.8.498.46931130201556048739030145002865898563","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18230847745299902998425838090653825745","series":"1.2.826.0.1.3680043.8.498.67621088918090038218729770378800160652","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18344563362014614046776037921812502569","series":"1.2.826.0.1.3680043.8.498.10663668416012310689003959592716440871","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18389676218363070462526838642303976332","series":"1.2.826.0.1.3680043.8.498.98757675872762385986268708628382210226","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18403627643718880914843209837672242908","series":"1.2.826.0.1.3680043.8.498.12873181211839997553833496606491963666","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18529656610752887218162968403101035390","series":"1.2.826.0.1.3680043.8.498.24076693986734904599012576532263328607","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18614704377104425868545800677445972261","series":"1.2.826.0.1.3680043.8.498.81106614348542406824720893749903658326","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18671006890719350655167441375718348829","series":"1.2.826.0.1.3680043.8.498.11798026444077118809695110503527355760","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18695727379446620258431748063749176639","series":"1.2.826.0.1.3680043.8.498.11237758114031564397884477050746184645","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18758199166360428760124471581842191566","series":"1.2.826.0.1.3680043.8.498.75103383895801352736726531448271814063","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18825393854436144614202928260751418496","series":"1.2.826.0.1.3680043.8.498.28919072040737824048033235048641483958","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18879049235810379709186936621085221483","series":"1.2.826.0.1.3680043.8.498.29889516794565834540618821317591594207","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18968967345606818331567480514437075646","series":"1.2.826.0.1.3680043.8.498.82622707974423831915785815880478114537","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19034825848229751701363461935217290295","series":"1.2.826.0.1.3680043.8.498.31901516135481743474499985441072501291","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19178214677492694135853596920291058397","series":"1.2.826.0.1.3680043.8.498.77065310706315840104732430537652427926","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19194090255958016414082788444327981898","series":"1.2.826.0.1.3680043.8.498.59363311502492983400254102452466188108","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19291375342005405073321673638971145777","series":"1.2.826.0.1.3680043.8.498.55526332828978979139031572199231375595","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19374331384306309057311171252852774841","series":"1.2.826.0.1.3680043.8.498.11961421621753154148527099011483411328","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19486520578867272329833751210055639515","series":"1.2.826.0.1.3680043.8.498.28747250039695939263270753650033211485","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19512754570664577075249836633718382506","series":"1.2.826.0.1.3680043.8.498.27734274588985234912338149493140950674","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19592989024994864431870593504844309885","series":"1.2.826.0.1.3680043.8.498.59583017819966558754310822792586658594","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19624674714110931897302481276137067638","series":"1.2.826.0.1.3680043.8.498.10298390427610871153535291351381832892","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19788612491138642031531719942303619648","series":"1.2.826.0.1.3680043.8.498.12255928642691446777561816817950053015","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19821498229702528191367700655079805824","series":"1.2.826.0.1.3680043.8.498.63932882355222264344808351897428174748","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19845711038895023426012115550237130530","series":"1.2.826.0.1.3680043.8.498.50090858481676976354583772153747238058","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19887850741193535685529812263489892001","series":"1.2.826.0.1.3680043.8.498.12783106801578304547479848140459380670","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19965044457637713843792494926652048462","series":"1.2.826.0.1.3680043.8.498.17478433496779454214674792244609654987","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20023559426196341254845485444587607391","series":"1.2.826.0.1.3680043.8.498.97185137148528986072446965258094667172","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20134168618929769962391688049013971830","series":"1.2.826.0.1.3680043.8.498.10037702445482286030346900811136595809","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20192024727432864451309151144591371229","series":"1.2.826.0.1.3680043.8.498.40295211164374335888286997510928688484","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20204907530808218314127264658715054041","series":"1.2.826.0.1.3680043.8.498.17424169971525085456455045935212256579","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20269296702024202621734017056347194499","series":"1.2.826.0.1.3680043.8.498.98616140031557681176264551812241250098","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20375739629244657719385962474248679850","series":"1.2.826.0.1.3680043.8.498.59980778555196536693531299034620839077","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20472525526345683475357475395677168372","series":"1.2.826.0.1.3680043.8.498.49465864739061264559999315028273323702","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20509336519532058090925280050885010886","series":"1.2.826.0.1.3680043.8.498.41520201472889314874828763225571475338","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20522330861615570894210641293344468905","series":"1.2.826.0.1.3680043.8.498.35819444299653411627215582483967011941","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20600318722531772866761132262772105640","series":"1.2.826.0.1.3680043.8.498.12371571603295181150767319850480651367","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20733151481921261415295204209910573639","series":"1.2.826.0.1.3680043.8.498.84888461092337926769044629386858993343","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20801986011906346655458394198951999413","series":"1.2.826.0.1.3680043.8.498.29212683099610864330770071711943239851","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20859518005209632163032213488405671537","series":"1.2.826.0.1.3680043.8.498.55174381193925100030542578383379574299","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20947841686457891570351579406919873254","series":"1.2.826.0.1.3680043.8.498.12847593812411325801137858123014009171","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21012635480218712816069264326295410954","series":"1.2.826.0.1.3680043.8.498.10165331407665465061983997853170158469","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21060274255072711241653174133146945101","series":"1.2.826.0.1.3680043.8.498.77773499413028298158971716956176861116","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21142867813782396036287673290033987859","series":"1.2.826.0.1.3680043.8.498.76387058802804550793216084565839161517","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21294732642571417618432023007693943849","series":"1.2.826.0.1.3680043.8.498.18298481211045244308266992633384326725","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21389336408905524316915532623385847283","series":"1.2.826.0.1.3680043.8.498.10304014540597184876286099552933836979","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21413285047211074156709106352750752275","series":"1.2.826.0.1.3680043.8.498.23726324591928859106092149394325106376","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21478660341431677113136859217606773871","series":"1.2.826.0.1.3680043.8.498.27306369991938040307328367086662980366","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21558570608416990974711670673431746511","series":"1.2.826.0.1.3680043.8.498.10815450037110808084777982360739835955","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21606196343708804451390109412118036330","series":"1.2.826.0.1.3680043.8.498.14468084823565367887972935879691037860","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21636337451527049130190928852193267826","series":"1.2.826.0.1.3680043.8.498.13136174690973501243358804569306831705","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21798342483930597563659425189400765869","series":"1.2.826.0.1.3680043.8.498.91560121236822692721956331722181000013","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21874562538764236819894456417694074673","series":"1.2.826.0.1.3680043.8.498.39008113317243736466740511636352990879","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21937000442536121262190458446148124910","series":"1.2.826.0.1.3680043.8.498.58411422805788324251279975057313724480","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21991252201684723900045301917670383228","series":"1.2.826.0.1.3680043.8.498.33314721472125828628933925942101242960","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22027914323057444746912555313058666950","series":"1.2.826.0.1.3680043.8.498.11699056882754646165209859766987021477","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22057596860528658818833903685827585805","series":"1.2.826.0.1.3680043.8.498.54829801782093407154477843775344165211","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22109739962224309418874538994436903404","series":"1.2.826.0.1.3680043.8.498.52627968695314286677230767737762639225","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22135282054716841452811687331325271982","series":"1.2.826.0.1.3680043.8.498.17059506779752659316239567563529676809","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22249763791584161324425922920269052844","series":"1.2.826.0.1.3680043.8.498.10314834889213432594346486361416312305","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22338400095583099753286935111647807038","series":"1.2.826.0.1.3680043.8.498.66791453672488411342525473736901724738","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22398041676780235097751294563297953841","series":"1.2.826.0.1.3680043.8.498.28555013540980325243536064689265729922","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22426390571898968732573184721547490379","series":"1.2.826.0.1.3680043.8.498.78463475423634224680759524987000937095","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22480218016816209183384996714060953859","series":"1.2.826.0.1.3680043.8.498.10720364591261893744746711984374105293","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22583985355672251619550173090948161714","series":"1.2.826.0.1.3680043.8.498.91758844599580325032197314958890734576","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22596004572382994883130705870731193046","series":"1.2.826.0.1.3680043.8.498.13232411309655372210540775323491942095","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22632919018724241015271844108954804020","series":"1.2.826.0.1.3680043.8.498.95074300241248665038905223977203651351","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22684505799921376181913295621462268931","series":"1.2.826.0.1.3680043.8.498.65779926973290374224666070254938037320","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22886251629508830700774506697670139848","series":"1.2.826.0.1.3680043.8.498.28942463641655126716247749328580415315","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22985467719058223198642920273233975053","series":"1.2.826.0.1.3680043.8.498.69567259975931509709221756626102710558","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23046618786812839762801821364158313134","series":"1.2.826.0.1.3680043.8.498.31275167685630605961220088297874173794","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23065868481290440008286663193810913414","series":"1.2.826.0.1.3680043.8.498.29475854190824377857200321205658378744","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23094733099955217022059584418468076048","series":"1.2.826.0.1.3680043.8.498.10237324586853250052540280401056389062","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23166300530486600338569917595744444657","series":"1.2.826.0.1.3680043.8.498.10851559814373512534138964577209130637","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23394601328148358087542766577727876635","series":"1.2.826.0.1.3680043.8.498.59535846654970852606997949668189710200","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23427198594727947883953289189443006288","series":"1.2.826.0.1.3680043.8.498.43358753180910661364851769280791630954","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23523176367321534630723586470190430254","series":"1.2.826.0.1.3680043.8.498.35692555862808983046082924259878904915","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.23703817737293327679300330167234268929","series":"1.2.826.0.1.3680043.8.498.75896288115948989866006154303551576254","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23719982492392272262684855675016475265","series":"1.2.826.0.1.3680043.8.498.66451332882238713279048443464918006364","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23822313132245448173452915228147497501","series":"1.2.826.0.1.3680043.8.498.32094803877681859554780887271212683064","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23911658810234305013984976739924353704","series":"1.2.826.0.1.3680043.8.498.40821791906363410032907688847821757086","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23956169649865635269920565218680494366","series":"1.2.826.0.1.3680043.8.498.13368753964728441036888769664567978232","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24068239310155798723127213793623172530","series":"1.2.826.0.1.3680043.8.498.10581073027762054080116036461787352548","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24126422042626612740219325272793737656","series":"1.2.826.0.1.3680043.8.498.19780902531025353502277210949904645160","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24220732394554303343522053195582322788","series":"1.2.826.0.1.3680043.8.498.93069275381457126072670985104814589914","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24283484945988825507450239904606754405","series":"1.2.826.0.1.3680043.8.498.27423679132125243988246560269212833511","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24313647347273496316728514161733307557","series":"1.2.826.0.1.3680043.8.498.20436618059908878034404081718049093303","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24478195415117310702720951368429189290","series":"1.2.826.0.1.3680043.8.498.27562667054096209003494251428636582347","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24529300482982370922479679056174585978","series":"1.2.826.0.1.3680043.8.498.11830100630781907035516807140993884939","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24588524807799377432185260702152371503","series":"1.2.826.0.1.3680043.8.498.79349802455862430800530761630684350687","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24661137747502154216858169258339731263","series":"1.2.826.0.1.3680043.8.498.30387733980097788001939393367316662687","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24714029982431301486050388345836960910","series":"1.2.826.0.1.3680043.8.498.11756994226611525609852329022653575083","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24808246532191535554375304081795021306","series":"1.2.826.0.1.3680043.8.498.36489849908781911893005045369053611649","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24872746733270791851241663716306993824","series":"1.2.826.0.1.3680043.8.498.40934001670096911842567321759101056035","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24940437326555494150792221007907527983","series":"1.2.826.0.1.3680043.8.498.11143087642344383135739279387194930503","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25088333117273619121936521384053709674","series":"1.2.826.0.1.3680043.8.498.62684475436712872334773881654595778700","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25107837907285885353845034555062540189","series":"1.2.826.0.1.3680043.8.498.10808513275822795478464844581510887208","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25167060436407065195851060078175404789","series":"1.2.826.0.1.3680043.8.498.32007343664592890469392836436850266817","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25206096958520407302630747466352252249","series":"1.2.826.0.1.3680043.8.498.83897843477746009925524576868322726957","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25259588756496584739814764896903786709","series":"1.2.826.0.1.3680043.8.498.12036565016700828106085057856019143425","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25292715244575997322139502072242615319","series":"1.2.826.0.1.3680043.8.498.28604567593800120083068093490375531431","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25431207706481141354761246438973384670","series":"1.2.826.0.1.3680043.8.498.22383898251093197220584303756235552047","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25480883220477368504575353972328124891","series":"1.2.826.0.1.3680043.8.498.12903775263713024598533452557821515657","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25512240159357553134500245590935010104","series":"1.2.826.0.1.3680043.8.498.41379559044321946052604124461615634255","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25550871519416839807700061317524004550","series":"1.2.826.0.1.3680043.8.498.65852441501044991195546161156135384528","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25616615589791001678067099907823838797","series":"1.2.826.0.1.3680043.8.498.56260707343451606348681430979175583118","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25650384476012633612528314070894186451","series":"1.2.826.0.1.3680043.8.498.61307441903837327957692363861152381689","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25663827732039068221184137540713755716","series":"1.2.826.0.1.3680043.8.498.34929874465834465850562844289469807538","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25695966186129395049687747156570466645","series":"1.2.826.0.1.3680043.8.498.96414939296600703610899289700780546771","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25765711614507755527825654507920391096","series":"1.2.826.0.1.3680043.8.498.10188220657721638432671532193102467344","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25766673238950703352855821505938941878","series":"1.2.826.0.1.3680043.8.498.77657938744580383422564284388534455346","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25818541516137783435601858274495597736","series":"1.2.826.0.1.3680043.8.498.44832791710888368550731642992628514274","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25899681172115698926926115441929702502","series":"1.2.826.0.1.3680043.8.498.12050297223602214096498832657180708116","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25928779639851562853500186715690156268","series":"1.2.826.0.1.3680043.8.498.10194484310009277513385604468657453298","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25977532612504997807641730439567378227","series":"1.2.826.0.1.3680043.8.498.82159736662506622201456314598753020859","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26006372873438224908989402529530833528","series":"1.2.826.0.1.3680043.8.498.11543952244274217438983181668087849111","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26072622476167889001255150030772906081","series":"1.2.826.0.1.3680043.8.498.33590531747334008816914371177650412269","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26201626427712146573573952548179938304","series":"1.2.826.0.1.3680043.8.498.42445518571592526495470401234201163365","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26325951519908477683595799515094966069","series":"1.2.826.0.1.3680043.8.498.31484694371157276467729935244082006854","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26351577824204863242390892891061594387","series":"1.2.826.0.1.3680043.8.498.96674860996504773679186944112298381801","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26431974442245797095978351976773354669","series":"1.2.826.0.1.3680043.8.498.40683889381819586665217492814108147867","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.26485916396240515006042297316506505324","series":"1.2.826.0.1.3680043.8.498.11541503034413153412715510361253358130","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26536992666456596813876340849399448897","series":"1.2.826.0.1.3680043.8.498.47962644026751944140246518629620324796","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26651323353208729423825552073457972796","series":"1.2.826.0.1.3680043.8.498.23729397168911234341886447570521662157","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26790702379506190936834447203448882465","series":"1.2.826.0.1.3680043.8.498.11361238702894937883953390052085014194","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26843088620128166809417361066548447201","series":"1.2.826.0.1.3680043.8.498.65039071017351494306407696689490766528","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26875920364303755911342338961252130512","series":"1.2.826.0.1.3680043.8.498.13094293164413464880160296144221204028","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26978027400525885904983020900996228004","series":"1.2.826.0.1.3680043.8.498.73599797216447404753196876984170132347","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27020884753639915245324093232218617967","series":"1.2.826.0.1.3680043.8.498.74471871311507335558389202765989845182","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27034189091164693165076510006436405194","series":"1.2.826.0.1.3680043.8.498.11737191167317906488573611304418034595","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27157074709305277871045901905120405881","series":"1.2.826.0.1.3680043.8.498.38611065943699038625337996553037009678","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27186001621007168581611489439038285508","series":"1.2.826.0.1.3680043.8.498.17756349427940745521076736932806560419","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27192678730060203972082511503480566620","series":"1.2.826.0.1.3680043.8.498.50983458669832053488744331263235119104","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27224327142578332905227939015479767285","series":"1.2.826.0.1.3680043.8.498.17571517783315627802511219237369999830","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27306039618857661522863167157208122018","series":"1.2.826.0.1.3680043.8.498.71494760316702866130184064221358903816","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27324284859290040148888098857880927353","series":"1.2.826.0.1.3680043.8.498.11283201007845719296091222026605563223","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.27398034409980350921804398626171135707","series":"1.2.826.0.1.3680043.8.498.12062737488493140233992354434778469311","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27437263843446879932281897446100134924","series":"1.2.826.0.1.3680043.8.498.40175815168544280787064836169050383117","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27474440682441854303485403899867525864","series":"1.2.826.0.1.3680043.8.498.77822779895550148763126151726121972725","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27627190971637735537007595480396264063","series":"1.2.826.0.1.3680043.8.498.11423795381631475167935075394289680738","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27655410895168998898506695863000928400","series":"1.2.826.0.1.3680043.8.498.13333740438328989090367355133231948143","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27671779587445983390840223475173360482","series":"1.2.826.0.1.3680043.8.498.10828432721478958598777944567691579713","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27723413941379681806317930349613970721","series":"1.2.826.0.1.3680043.8.498.22204007224362001929442255779127976641","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.27737101979486411869208236811885597351","series":"1.2.826.0.1.3680043.8.498.11363609493067147357751157713318216579","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27796604203156439069742212039368371002","series":"1.2.826.0.1.3680043.8.498.11238922574500584842524934007115621646","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27921111050680927418542588425335218896","series":"1.2.826.0.1.3680043.8.498.58439922721034440664203607347735212221","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27934397251575987048967084987994202973","series":"1.2.826.0.1.3680043.8.498.12958989171433940199209647123640373814","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27990965288663885936880638302158503894","series":"1.2.826.0.1.3680043.8.498.12297659174681008458340327388185373586","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28028890493668487238845659163252231337","series":"1.2.826.0.1.3680043.8.498.12658798564436867084657657760017384423","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28045606887691550130031344985830975325","series":"1.2.826.0.1.3680043.8.498.11064109764685084786946639764892086098","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28153286315438977482271934270979508349","series":"1.2.826.0.1.3680043.8.498.50595116675045848091532091659842975649","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28237068093947300427040925444622840795","series":"1.2.826.0.1.3680043.8.498.24423291683270685091688572724436924915","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28289838818003822677994912735532050038","series":"1.2.826.0.1.3680043.8.498.80813658986416590391912153956458218683","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28309626953500461628112343997868050028","series":"1.2.826.0.1.3680043.8.498.42357714135772810925548409157367468649","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28360174615866226538417948042981668998","series":"1.2.826.0.1.3680043.8.498.13280142363231169900891825854254172653","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28481442608487320644829189127206806471","series":"1.2.826.0.1.3680043.8.498.11057573577897416562440644061735071903","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28490861391717560759667589681288427737","series":"1.2.826.0.1.3680043.8.498.51875938064120641177842533408405915524","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28595729988867300278425411189173775382","series":"1.2.826.0.1.3680043.8.498.95964013576352506407269531192760186639","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28628472249424790112957421215774036156","series":"1.2.826.0.1.3680043.8.498.12427732169235811881932174579250457921","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28646304373057947277669139986982208473","series":"1.2.826.0.1.3680043.8.498.27756440384004804595690344845548936410","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.28670999234453413675502551916979760344","series":"1.2.826.0.1.3680043.8.498.27962905997609619432131656882948992368","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28746954364434312909368235768936980607","series":"1.2.826.0.1.3680043.8.498.58202947869066069382410295858502974930","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28779021651975021236045277212517629154","series":"1.2.826.0.1.3680043.8.498.10118611894766940897517672723885257772","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28888941572409703700315590339486528256","series":"1.2.826.0.1.3680043.8.498.83575299055197526791809893637491466257","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28944156754203202831026290463520445489","series":"1.2.826.0.1.3680043.8.498.13222580064867972960615237533635561703","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28954361146470748868500417103935640742","series":"1.2.826.0.1.3680043.8.498.10766526563059850396116017928446073325","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29090158364700006304493717187346344597","series":"1.2.826.0.1.3680043.8.498.35352237674187229391916386479084381267","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29280448517432444306374036429824425668","series":"1.2.826.0.1.3680043.8.498.91949805242690199102037469908532939344","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29343735048224321966824641432873034291","series":"1.2.826.0.1.3680043.8.498.52393553336092980508648499654518994678","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29364640434226980751405780979697632001","series":"1.2.826.0.1.3680043.8.498.69950374033910117930696342597225384094","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29384016102624554484300773136383156153","series":"1.2.826.0.1.3680043.8.498.27389734809295043665370553493999489852","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29443692810539372078922146461173545451","series":"1.2.826.0.1.3680043.8.498.20911740890089978744532658202920664789","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29579560656789178333629609998252952856","series":"1.2.826.0.1.3680043.8.498.22624849080106184322842102366247268237","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29739171278960083313048266628194182569","series":"1.2.826.0.1.3680043.8.498.86611393192343250335066685378209336231","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29758331841591711520514486913006094054","series":"1.2.826.0.1.3680043.8.498.30684611705269449263210719764419969292","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29826129803278661744835136993610646069","series":"1.2.826.0.1.3680043.8.498.11810684519791916258384054599870980963","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29971686675947075024168289004002445393","series":"1.2.826.0.1.3680043.8.498.38212404951766813778584085492012864990","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30058585660043363937345050478072453854","series":"1.2.826.0.1.3680043.8.498.23480217721204609780806379699607382745","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30131339052241668380936148362472245259","series":"1.2.826.0.1.3680043.8.498.47332820569540064578410065438056736178","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30167179890902431464942556467698238073","series":"1.2.826.0.1.3680043.8.498.30644997055358251606201510114292899953","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30222189977108980260768952691472020722","series":"1.2.826.0.1.3680043.8.498.46621852224007008225087687375507131871","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30246079718471552972130572444383079911","series":"1.2.826.0.1.3680043.8.498.75078392068446053207082784619084087737","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30252501257451878373370333329266264057","series":"1.2.826.0.1.3680043.8.498.30197605814766116038169374143547583328","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30330001893437233719772129404417384472","series":"1.2.826.0.1.3680043.8.498.29746158558647159871680208479407013786","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30420245540489522189154634720778032533","series":"1.2.826.0.1.3680043.8.498.59128505297916879697577964100037012493","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30490857770169699336468319880130884989","series":"1.2.826.0.1.3680043.8.498.69858095290060242347074022553287471509","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30520444048368652235940346583890702451","series":"1.2.826.0.1.3680043.8.498.64872339537733325487636039454452267712","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30557810085867943803354292590079995622","series":"1.2.826.0.1.3680043.8.498.24077785817362409098999952415943378232","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30672390215843894382866701432377109444","series":"1.2.826.0.1.3680043.8.498.40760349688941955027111760297871899909","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30690232747840004203140392776057147177","series":"1.2.826.0.1.3680043.8.498.70169279312215156510623742863570110586","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30763696628968291934469221453461209579","series":"1.2.826.0.1.3680043.8.498.43373769273794831096618927111548982224","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30832771419882354941430609179589560374","series":"1.2.826.0.1.3680043.8.498.63876905052857568802629667076401576433","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30838586189496669675465778955893367497","series":"1.2.826.0.1.3680043.8.498.60563416048554524688879034214696497656","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30942874102560162331831960649006411381","series":"1.2.826.0.1.3680043.8.498.10568028957966866187992963807955413226","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30978547345367837000803077743620171961","series":"1.2.826.0.1.3680043.8.498.28224216075672555249442527621565233108","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31080452089421812812019413302222437434","series":"1.2.826.0.1.3680043.8.498.97425326833235960429102898569740603296","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31099878224304681441631176766255029271","series":"1.2.826.0.1.3680043.8.498.11881835185926960237555976926976118104","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31171050331393253655760166990008263793","series":"1.2.826.0.1.3680043.8.498.10002978947438170388686989793017172190","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31175617350083221245857813741060042777","series":"1.2.826.0.1.3680043.8.498.14757712296921149160337433946177087889","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31220970065722563280311764269386457090","series":"1.2.826.0.1.3680043.8.498.11527291024991421518430070398708707006","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.31280487369922340170695453174997912050","series":"1.2.826.0.1.3680043.8.498.95880070161932590048171686176102051703","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31352187408311685267245948214616940728","series":"1.2.826.0.1.3680043.8.498.11293858464238158950576799850299010979","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31421989235441217674752676866539112529","series":"1.2.826.0.1.3680043.8.498.12152665780343990151518190489459206617","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31555333355990829772911515362502000209","series":"1.2.826.0.1.3680043.8.498.11604237841435081697274002640287867539","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31640159014711750808214354395404624815","series":"1.2.826.0.1.3680043.8.498.63408146166997663820335070418994929380","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31790457319528233269194462675995569260","series":"1.2.826.0.1.3680043.8.498.55897013575854000566807234359955196548","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31827987476038668074615045841102711882","series":"1.2.826.0.1.3680043.8.498.55942710480828292950133062088459227303","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31851627014639252866084081561126592250","series":"1.2.826.0.1.3680043.8.498.13285004461442883381456163818511324524","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31899121807187535405063131351086765296","series":"1.2.826.0.1.3680043.8.498.11914063626101064023382544261851029578","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32043012602421532910420313983803503778","series":"1.2.826.0.1.3680043.8.498.65582622051030114319388100877996526361","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32138869097959653224534642965199855627","series":"1.2.826.0.1.3680043.8.498.92383844023094748669988788430433347345","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32226253011052584155674401943276961907","series":"1.2.826.0.1.3680043.8.498.20376797085516267164156864075035478095","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32265622493980668732317541941196894795","series":"1.2.826.0.1.3680043.8.498.33301963288916587176468030754122994608","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32328699307307661181881949763360098439","series":"1.2.826.0.1.3680043.8.498.12368883878178371084372850122917878293","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32413534416513678586709935453506443461","series":"1.2.826.0.1.3680043.8.498.82270988988686684503880603623797370186","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32479930837336533083798155873656743007","series":"1.2.826.0.1.3680043.8.498.27956496427049782272588382155221657369","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32504785212530110880087745488871181111","series":"1.2.826.0.1.3680043.8.498.50514528896686574694012001445343174612","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32571577475600416042179474541176580933","series":"1.2.826.0.1.3680043.8.498.98123242473835938000899009472605420541","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32611545463217490575457831874695918229","series":"1.2.826.0.1.3680043.8.498.91238074041000345912025331315716010476","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32680606387445562892045224227371304764","series":"1.2.826.0.1.3680043.8.498.83725569270209380550746733386381525626","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32742383730417896527915284404039812924","series":"1.2.826.0.1.3680043.8.498.11712223204019693221641902520744452406","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32835080036019877751472792740274297358","series":"1.2.826.0.1.3680043.8.498.54978249115538840627135584646804040273","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32916544979513354644075485240094539251","series":"1.2.826.0.1.3680043.8.498.22834746388828975723570915962574040420","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33045687849853291066101186962073873042","series":"1.2.826.0.1.3680043.8.498.12296347748185014589767012687431791818","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33114390259375873203134285352951765534","series":"1.2.826.0.1.3680043.8.498.12207198393595039801838165878510832067","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33247377165700300890880607485836825807","series":"1.2.826.0.1.3680043.8.498.85011217264587180597825393349573575359","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33324606756355240783076971858231422510","series":"1.2.826.0.1.3680043.8.498.37179110279453919873822823278880900150","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33339945632086976351786752793064764443","series":"1.2.826.0.1.3680043.8.498.11819169239048940562521527736325192554","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33419086668421177906530808458004386826","series":"1.2.826.0.1.3680043.8.498.35837537606794520458599874516110400150","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33479806419966566464727310013949313521","series":"1.2.826.0.1.3680043.8.498.29238683240019420436430522435918872801","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33614319611050071354692274770073371314","series":"1.2.826.0.1.3680043.8.498.12781753622990062596540104980788460124","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33707148112045883958228656491167647755","series":"1.2.826.0.1.3680043.8.498.42124705220579937516767718465894536355","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33734536545775442043827840341408174347","series":"1.2.826.0.1.3680043.8.498.43976749955528341389968560610821012528","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33880013318702285028756482543453984598","series":"1.2.826.0.1.3680043.8.498.15911466032527485071308994755174971570","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.33899056854391617633929258302459140830","series":"1.2.826.0.1.3680043.8.498.68839805364998796277344976842497193957","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33990091057243328830745904178686745167","series":"1.2.826.0.1.3680043.8.498.12041535414862508102341570192531649254","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34040631818813472592501398076075999835","series":"1.2.826.0.1.3680043.8.498.81395570270168516395950876710379380549","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34076627216941578548083498328399240999","series":"1.2.826.0.1.3680043.8.498.86398155169574326696974092366869024821","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34126843286927697018624309801437341727","series":"1.2.826.0.1.3680043.8.498.72958215562514358914183425256886467544","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34262242337778924864334788374037992215","series":"1.2.826.0.1.3680043.8.498.66540723904905158700493932699844728066","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34360298211002490108642003575962873984","series":"1.2.826.0.1.3680043.8.498.10411077001789531688334109497563134092","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34433343440016591348979084714300062954","series":"1.2.826.0.1.3680043.8.498.89430804326513432915062758006141101246","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34461302710296907289543656738174044465","series":"1.2.826.0.1.3680043.8.498.30384101658003999716759869077284095656","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34546106114963066554367298449299214677","series":"1.2.826.0.1.3680043.8.498.58273959992966133545331769132411177077","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34574046861696910828478657346725687542","series":"1.2.826.0.1.3680043.8.498.73520121452697220711760029756363139482","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34685905030370793639196564723935583035","series":"1.2.826.0.1.3680043.8.498.64923993132617839685637225229996868277","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34717974828302316666446325243570202245","series":"1.2.826.0.1.3680043.8.498.78583837277358586022904567223919076450","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34753112960188810416088070814158959510","series":"1.2.826.0.1.3680043.8.498.13201775091685468100150819667596578260","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34826100703045042103752460066224521343","series":"1.2.826.0.1.3680043.8.498.55980542480176108558169548330505536498","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34861721301001052975626513770908147451","series":"1.2.826.0.1.3680043.8.498.69056969732188450915839064569448647582","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35015675197378142698488944856288804597","series":"1.2.826.0.1.3680043.8.498.44775076637146015471048040406178171002","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35027750475162395113025853006474122989","series":"1.2.826.0.1.3680043.8.498.96246300860440805154635653424901201656","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35090276406828326770336478753776261776","series":"1.2.826.0.1.3680043.8.498.10101248689435682481882435559124375663","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35127678462764233938878060108411012978","series":"1.2.826.0.1.3680043.8.498.59134626400853887492353190321283309780","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35257158249961136373696255071228322165","series":"1.2.826.0.1.3680043.8.498.11538824460572766801705440706688073340","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35330802038595377723999341992570259909","series":"1.2.826.0.1.3680043.8.498.12885010318416278213970902663680187891","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35526385429946467546636621603749416429","series":"1.2.826.0.1.3680043.8.498.46272407953527446422420930984860569492","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35572306796137927247415883445612561056","series":"1.2.826.0.1.3680043.8.498.99289716907012131361267331802598311755","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35639250779062198905356481136455419313","series":"1.2.826.0.1.3680043.8.498.12967300358806553708550829286838955526","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35717294165699019392845830077937768611","series":"1.2.826.0.1.3680043.8.498.63626232248803430327424304973911202072","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35777075794856195065665011443485613681","series":"1.2.826.0.1.3680043.8.498.40932141904835755644519658008064109159","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35820650938063925991582813178954225720","series":"1.2.826.0.1.3680043.8.498.57273179488527852592295208019957212676","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35836928928058091851993989259747528067","series":"1.2.826.0.1.3680043.8.498.12047853241592058631446535994187947614","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35871201339095431578743898508613853278","series":"1.2.826.0.1.3680043.8.498.20488981391366192812206731920718895398","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35923146679164443588381195543533271939","series":"1.2.826.0.1.3680043.8.498.52098266318294833015149541557239193479","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35923753382244056317223397206343376734","series":"1.2.826.0.1.3680043.8.498.12816980805622072670533382587033385708","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35961087303991442751747064990497975939","series":"1.2.826.0.1.3680043.8.498.12756896007049726411448082735518971222","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35971292331621735865925455612813197566","series":"1.2.826.0.1.3680043.8.498.11534541556571742549605354705463611441","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36035264348264378094569428634302238394","series":"1.2.826.0.1.3680043.8.498.67264241031203544525737013977579197940","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36131369236532190338276701919104598475","series":"1.2.826.0.1.3680043.8.498.11085737024100445112088385413984444628","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.36180248571044206939894546783755557323","series":"1.2.826.0.1.3680043.8.498.32758834666090372391576315103942827580","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36221680997794815863088530960610069278","series":"1.2.826.0.1.3680043.8.498.12257028330298148888976279402376323759","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.36293583160636884934622362743062825202","series":"1.2.826.0.1.3680043.8.498.13782220074100495530533073777630186520","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36315811661139684771871696847516662339","series":"1.2.826.0.1.3680043.8.498.46969620048753616814121615771588422379","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36488342368050295711098478501722217415","series":"1.2.826.0.1.3680043.8.498.62189820757868298011919928253167859056","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36588608184578739092409073014150374185","series":"1.2.826.0.1.3680043.8.498.12137537830848388126948207804617025615","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36648377536792309029975432419163533533","series":"1.2.826.0.1.3680043.8.498.71484365200994899789536470401254140764","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36688959138536319644194458211515765203","series":"1.2.826.0.1.3680043.8.498.52288261927805100961829875311057749764","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.36757476214794170118868510630678488185","series":"1.2.826.0.1.3680043.8.498.17702057353200771926314077471785373602","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36846391753332367835804699576399553980","series":"1.2.826.0.1.3680043.8.498.39092290483647838514059439547667497866","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36914870739904705183135763739850515617","series":"1.2.826.0.1.3680043.8.498.70582988794466423622126666918232585664","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36995206854684735704099881847485073991","series":"1.2.826.0.1.3680043.8.498.13121129983651897996608312606536475458","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37007302166102452703965686598524467188","series":"1.2.826.0.1.3680043.8.498.74982284925772426894631141971042226769","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37046961643473249831189497076699393868","series":"1.2.826.0.1.3680043.8.498.92843860266172051363358095386955070975","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37110727485959679337431199930441321268","series":"1.2.826.0.1.3680043.8.498.12184820084484728219735047084455656192","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37257728513136112802363958568529758895","series":"1.2.826.0.1.3680043.8.498.44073796923076705453442314756874998225","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37332768072749723117839591872457536315","series":"1.2.826.0.1.3680043.8.498.13027009759689039364751272858104344083","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37390494423735156928337187351900345080","series":"1.2.826.0.1.3680043.8.498.10405128681359344052988943207347969628","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37413349933694832641902637847629969234","series":"1.2.826.0.1.3680043.8.498.58589162725899686912147262289926655719","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37432328401993499374675847888297416449","series":"1.2.826.0.1.3680043.8.498.23211609184924284084409780750548334989","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37463970179664003092832463732043124908","series":"1.2.826.0.1.3680043.8.498.95183130408974467926726419286583035340","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37510478969810784660124747242051166554","series":"1.2.826.0.1.3680043.8.498.11844965342894214666561359130483239951","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37558426555271635608942179071280432530","series":"1.2.826.0.1.3680043.8.498.49855787994961575660436334295504269241","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37591186611460576939009950935820113903","series":"1.2.826.0.1.3680043.8.498.12556459453428943509630530509760859260","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37633035024065331400434438202028794597","series":"1.2.826.0.1.3680043.8.498.90019655624257943881391622495306492156","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37695832484726177391252504639155717084","series":"1.2.826.0.1.3680043.8.498.95266284460756026528941025031087660767","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37782918081276484321055690005029111967","series":"1.2.826.0.1.3680043.8.498.13385067250833224970793468918801217093","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37846940236952007502147453013489359606","series":"1.2.826.0.1.3680043.8.498.22664839195145994022227186015471783060","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37879265071349098151803248077034280617","series":"1.2.826.0.1.3680043.8.498.10838050201626370619213121403874972934","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37921127120678939078868474340027123335","series":"1.2.826.0.1.3680043.8.498.93296394352435133152225703724099502373","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38005916106577944091708997377832932938","series":"1.2.826.0.1.3680043.8.498.43651376105732300823549505041197856009","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38039631093067555454700266397073080430","series":"1.2.826.0.1.3680043.8.498.68350538603496389211592495570786846793","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38069035744880456154810472748123499221","series":"1.2.826.0.1.3680043.8.498.11244454995410171214652450632367189707","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38104228771638294693272534500734878990","series":"1.2.826.0.1.3680043.8.498.59297295278420449822113180803363075848","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38233845434092134418342290639986261345","series":"1.2.826.0.1.3680043.8.498.82784020840643295842585766902678521789","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38272345519163551901865251730721530169","series":"1.2.826.0.1.3680043.8.498.10467656696119828251048737256295640263","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38303307720827621886420004896097661262","series":"1.2.826.0.1.3680043.8.498.37708820621178246108295750570746422126","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38333934623775971943734069301776225018","series":"1.2.826.0.1.3680043.8.498.34436580109961958403672320788230270666","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38365187857616428023213060377488046375","series":"1.2.826.0.1.3680043.8.498.68719428474786653074885644531578327364","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38420384692181444061893234153745102059","series":"1.2.826.0.1.3680043.8.498.10816917951596978574877591329696012453","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38485168845116671268554752030254772359","series":"1.2.826.0.1.3680043.8.498.65247768487922537957265841250892704409","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.38528511947818361078957998546327084024","series":"1.2.826.0.1.3680043.8.498.51904981937493121832107983110645288297","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38584486469155131627808759862008577200","series":"1.2.826.0.1.3680043.8.498.59489915187143587339707604707118453584","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38593351637971248239815933638950449587","series":"1.2.826.0.1.3680043.8.498.89723817826035646582869855751767962509","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38628165743951175602803204505216754926","series":"1.2.826.0.1.3680043.8.498.11127908826264139499009460342059673976","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38702677049080452580748506476598606517","series":"1.2.826.0.1.3680043.8.498.21722667192752109202614098757594870073","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38750761816114659458563557493234655604","series":"1.2.826.0.1.3680043.8.498.12092852689210078304072738013346524852","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38796519786809225694393617946244012065","series":"1.2.826.0.1.3680043.8.498.40021736172496735894074136920664300350","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38870467957974446108337293912930572060","series":"1.2.826.0.1.3680043.8.498.10708103126325432456506186326007344898","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38898732985465285166803609252960353405","series":"1.2.826.0.1.3680043.8.498.12781084861184560853836091117866414206","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38925755872951351471027638563742132144","series":"1.2.826.0.1.3680043.8.498.13677394038314065715139766544708532078","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38940868544480800333574786964232944756","series":"1.2.826.0.1.3680043.8.498.51735740091131210196867095409707905944","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38981318408220047366318316620365040799","series":"1.2.826.0.1.3680043.8.498.16550537561954660595261771082511205744","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39025433784711757736168236829157583043","series":"1.2.826.0.1.3680043.8.498.88571439354525113398849445746474127488","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39070169128115370701189151138307035053","series":"1.2.826.0.1.3680043.8.498.11877158951590079086922882073475170359","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39190090804401525573012570218955054016","series":"1.2.826.0.1.3680043.8.498.30706723444932020510357582973007360831","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39264696726258404531295961801831089064","series":"1.2.826.0.1.3680043.8.498.10364757228377984623204270364751990233","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39324076659429494696721784912308233388","series":"1.2.826.0.1.3680043.8.498.83898953717380675101321155428632494775","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39399668753650041254826892657562722764","series":"1.2.826.0.1.3680043.8.498.91453022531139758456377917679540363407","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39500553372517290823815606829518305500","series":"1.2.826.0.1.3680043.8.498.33500735563674808256834776499303611578","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39578126940710463296572383286587298912","series":"1.2.826.0.1.3680043.8.498.10549674831208489531373998571583097991","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39636073039676814228254715705296535819","series":"1.2.826.0.1.3680043.8.498.70297356663824814143925062985853254149","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39787654910948635229369478006027304578","series":"1.2.826.0.1.3680043.8.498.12214277700721663196289043682420621780","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39875636232483792306109110284167439592","series":"1.2.826.0.1.3680043.8.498.19412219041051810844189103081092987262","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39932424305167611414527482641898907759","series":"1.2.826.0.1.3680043.8.498.10019629802122213675374932250349863152","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39959632509842689058626880477779808473","series":"1.2.826.0.1.3680043.8.498.83174440531077691659422392384829923671","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40036526071307024601256787605725348112","series":"1.2.826.0.1.3680043.8.498.37292664067179917354354278582559110793","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40131899651518057045444739705301716085","series":"1.2.826.0.1.3680043.8.498.12173730166175255178151343637715353231","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40223239401390462911823193489179244754","series":"1.2.826.0.1.3680043.8.498.13002361408024705794849207139320195944","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40254123984175499564145728084464192446","series":"1.2.826.0.1.3680043.8.498.12620105724267803874725621269102668036","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40308838286515017547707130085914948230","series":"1.2.826.0.1.3680043.8.498.15585286695839433958854535814362779642","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40372643537313871452054321931253684619","series":"1.2.826.0.1.3680043.8.498.40299818391535625817458402268566121036","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40419759559020255481034237210540871312","series":"1.2.826.0.1.3680043.8.498.12730414183679521834777077545964612227","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40441879276927383122480339472406105572","series":"1.2.826.0.1.3680043.8.498.13295285310966801886968952819297234451","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40533893516114346078820881178450524201","series":"1.2.826.0.1.3680043.8.498.10662793534753485076077010396621042547","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40608233778918249560056537030108548037","series":"1.2.826.0.1.3680043.8.498.92370735296502131671964589764889517225","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40618283771317160262473004328692613267","series":"1.2.826.0.1.3680043.8.498.40360825128538413201076621812683425877","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40654763252951267464957826289984701872","series":"1.2.826.0.1.3680043.8.498.12713949210961280060664432652336945336","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40723835064354784133289444204866484476","series":"1.2.826.0.1.3680043.8.498.10391570269736500420914457709941810276","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40823842819404259265489417470859248075","series":"1.2.826.0.1.3680043.8.498.73296859407051378347799753820705446477","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40837296112545776586904280088902000849","series":"1.2.826.0.1.3680043.8.498.81033936903305426129789810324146165545","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40930045208346676628855922755791787949","series":"1.2.826.0.1.3680043.8.498.52545372585662158255897697315352895208","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40994616246264625946610553907227975893","series":"1.2.826.0.1.3680043.8.498.13634227510607348592741278951128527848","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41057428195711190931490429649014776496","series":"1.2.826.0.1.3680043.8.498.23198640540067439377498471022429513021","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41129090437522031508960438281451672212","series":"1.2.826.0.1.3680043.8.498.31892606189494173334460291208012279737","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41240503248178304281405515398211762506","series":"1.2.826.0.1.3680043.8.498.33056178251771042824492656679682665333","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41326575785634778198143222323166756210","series":"1.2.826.0.1.3680043.8.498.96772312656203031576988663491131396505","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41366889271483886167284378157366992074","series":"1.2.826.0.1.3680043.8.498.67078227894895472847691474252832658175","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41439889720274233955076963938741061358","series":"1.2.826.0.1.3680043.8.498.11341088182141161875054344787850820380","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41564229176277158115386256329157870722","series":"1.2.826.0.1.3680043.8.498.39534796374132298720656171224740828633","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41600498783384864111179175518614837125","series":"1.2.826.0.1.3680043.8.498.11820289040357912936136362656225665229","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41623512571866040054960002151226127775","series":"1.2.826.0.1.3680043.8.498.39010719166009225197294849796111100130","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41657361598083583047831423768034379414","series":"1.2.826.0.1.3680043.8.498.30628867572934275922246887202581160037","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41672145969790132883562432403529293545","series":"1.2.826.0.1.3680043.8.498.11847666308817702287808962478797661284","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41717437729988689273718049162153247404","series":"1.2.826.0.1.3680043.8.498.10410035595883952768316748577314760043","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41769275806689876192405091865340286535","series":"1.2.826.0.1.3680043.8.498.40017353847696948935594810895815592402","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.41810826648175757249770990269365542940","series":"1.2.826.0.1.3680043.8.498.22654803366622828475417893812374217268","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41854610297797849683195403204758363476","series":"1.2.826.0.1.3680043.8.498.20936634473430741209081247528866059681","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.41990779875873214154237496750949660480","series":"1.2.826.0.1.3680043.8.498.93727472876618174417501676849884599203","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42018683052544325209168649309206539721","series":"1.2.826.0.1.3680043.8.498.77494019260227986800202608855939543006","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42051385452979123325023608442860201782","series":"1.2.826.0.1.3680043.8.498.45210186801802880554313494770387816541","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42130173624033946465704266507948122088","series":"1.2.826.0.1.3680043.8.498.62096776366952580901191885015101847665","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42221373360893091454504422964928099406","series":"1.2.826.0.1.3680043.8.498.92078559496935970339811991676153949379","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42389063782063360163798575455522539633","series":"1.2.826.0.1.3680043.8.498.46530670831132793510141396172619072342","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42414269167296543219685897036829241885","series":"1.2.826.0.1.3680043.8.498.87348932379754008882651209097497191100","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.42486421970612099441197418307958561877","series":"1.2.826.0.1.3680043.8.498.11854471942746262415413200692443532129","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42580107838226807862930203257414265276","series":"1.2.826.0.1.3680043.8.498.99727156574557145439102718973689807001","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42598518026540237824479232797503218019","series":"1.2.826.0.1.3680043.8.498.46924180689139232147183032686281356946","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42634709955067416383713194799683605006","series":"1.2.826.0.1.3680043.8.498.26372804217768507853681254762469616051","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42776652100045349404152232788825670056","series":"1.2.826.0.1.3680043.8.498.25751369344228312350332103351865616116","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42816339547796375466199938922141987775","series":"1.2.826.0.1.3680043.8.498.62676301741403781528310278260355713290","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42887028634948156598417204377884297339","series":"1.2.826.0.1.3680043.8.498.36795784643307178990315145588772482158","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.42921593713351589642747633505679170597","series":"1.2.826.0.1.3680043.8.498.91898995662726217961501325384850073741","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42967970121147528831983176261453235744","series":"1.2.826.0.1.3680043.8.498.11736489880372335375865228447775569844","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.43014288535685961874615794778401152033","series":"1.2.826.0.1.3680043.8.498.12167743434731744331412363290522160582","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43117007553940146763245418710268028741","series":"1.2.826.0.1.3680043.8.498.14096229508332475990590957407195431088","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.43144544376687968624598661376472625623","series":"1.2.826.0.1.3680043.8.498.11418148177618967627189747964186786438","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43300763182450509440662392516941842389","series":"1.2.826.0.1.3680043.8.498.65352089584968371259169331663777383317","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.43339007024446439709455570527734115557","series":"1.2.826.0.1.3680043.8.498.84730090174935075674863083383560109798","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43412154740072739455825163326814980838","series":"1.2.826.0.1.3680043.8.498.11268625730549819341310509952031699316","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.43442933855807685981061557274166869461","series":"1.2.826.0.1.3680043.8.498.98697647044359856636448519068155070398","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43483537125776347707929354413013709736","series":"1.2.826.0.1.3680043.8.498.32579194790631258029274483419019223217","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43692339773096477132070133629776394739","series":"1.2.826.0.1.3680043.8.498.99945476856693727800853176393711829787","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43779228281849457650343989788070421822","series":"1.2.826.0.1.3680043.8.498.12314521641474229177711529709912818696","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43870319732657730742794291443940225011","series":"1.2.826.0.1.3680043.8.498.12958427127534225999817695089189445002","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44024350801191362011463613321559949052","series":"1.2.826.0.1.3680043.8.498.10966748574481340699786223161260396611","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44259687212868598275713310246316207497","series":"1.2.826.0.1.3680043.8.498.13229875320174624002534108713333603946","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44323131330689365651417881352766725366","series":"1.2.826.0.1.3680043.8.498.90060004130106506499907681642113345086","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44350638600163778456163342454087669278","series":"1.2.826.0.1.3680043.8.498.12391877115092131385322812052323174098","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.44417768181231991989438194369583907493","series":"1.2.826.0.1.3680043.8.498.10541024563899324940738644528592409876","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44441813293962885868294317830142850713","series":"1.2.826.0.1.3680043.8.498.12728676083010906333441665515790664240","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44525218699266472141419416162979441275","series":"1.2.826.0.1.3680043.8.498.10252120765602374592738038949030698493","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44670636848627131733157841382853275817","series":"1.2.826.0.1.3680043.8.498.40836588996613271532413019880906216105","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44733200650377282012534389271385687852","series":"1.2.826.0.1.3680043.8.498.46077767463956810547953915067720757483","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44863396093454506630473164990623458425","series":"1.2.826.0.1.3680043.8.498.15356663572163681466224454558998823288","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44879085997210091159117906674172231025","series":"1.2.826.0.1.3680043.8.498.11264508781676662114172793996655057373","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44902036121220687805715010480088379768","series":"1.2.826.0.1.3680043.8.498.97248207564137740792492435255376740884","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45033326048521572717771708818514772861","series":"1.2.826.0.1.3680043.8.498.44641571065403881513075910626767643412","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45072896942121803896760563154385918032","series":"1.2.826.0.1.3680043.8.498.45377129434639142257390125991722644142","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45170517567841608067443743961197123093","series":"1.2.826.0.1.3680043.8.498.13304641341287046191228614953521652852","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45251306032215692953561852960585282987","series":"1.2.826.0.1.3680043.8.498.79833690781863499808608007933236134837","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45304570280714567513866386380356988876","series":"1.2.826.0.1.3680043.8.498.10245475887210009992147130573566107713","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45326662718293919909188368237039419949","series":"1.2.826.0.1.3680043.8.498.92735463227722076211748289636285711877","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45368532788186860828867663606651818835","series":"1.2.826.0.1.3680043.8.498.73756488755139518775735961056422380336","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45392604471538776666611947492530314159","series":"1.2.826.0.1.3680043.8.498.17974913869058050388603503245761637953","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45462389009834095091353507592651342877","series":"1.2.826.0.1.3680043.8.498.90592034088783775521537153409918473256","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45491168638937605395466251659273753775","series":"1.2.826.0.1.3680043.8.498.84523037443942605383800219042173319110","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45625191292968830335971647388746041117","series":"1.2.826.0.1.3680043.8.498.21259113117036499581080551655283673602","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45678969592219293243366784683191701674","series":"1.2.826.0.1.3680043.8.498.10411020546042022702942748887477961461","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45768777378108699271714894225826640121","series":"1.2.826.0.1.3680043.8.498.15979542635244081240996274375123844554","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45865541491386404818767748310365919685","series":"1.2.826.0.1.3680043.8.498.26218259767099949543639111738875866162","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45894475516096841241467193775629362381","series":"1.2.826.0.1.3680043.8.498.10277875411590434922847558564926134567","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46016776602328665980436110567854612595","series":"1.2.826.0.1.3680043.8.498.35167121674366803804301834914653737955","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46052469424267861803032899866428547156","series":"1.2.826.0.1.3680043.8.498.12444276824197104939455580094780321827","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46172451572836863687073454997158931662","series":"1.2.826.0.1.3680043.8.498.13123673506421302207042671731444746362","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46259494895672269475940484605821837768","series":"1.2.826.0.1.3680043.8.498.25041960589594417848776115661386437990","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46282234455466954264170065462242039568","series":"1.2.826.0.1.3680043.8.498.21110107188889643916429064774425645221","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46309540159823359540686782651264853894","series":"1.2.826.0.1.3680043.8.498.44357709593103782141368959091961509438","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46407639440476738746130651250276012493","series":"1.2.826.0.1.3680043.8.498.13192092214677303767382587592190309407","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46491051379327985199309365945879841064","series":"1.2.826.0.1.3680043.8.498.72241430192346346908971801196663198315","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46658713832425741743740258795376270265","series":"1.2.826.0.1.3680043.8.498.21611040016765592924854173612952056847","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46715670142711601593792518196789347552","series":"1.2.826.0.1.3680043.8.498.12643998478551654415862580530002504852","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46770589129181839292542004762882468541","series":"1.2.826.0.1.3680043.8.498.30827499653737521035361403786649585948","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46825768767498441994570613185981335949","series":"1.2.826.0.1.3680043.8.498.26168529167492981747184752024253723144","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46884541791777826238103426426267571657","series":"1.2.826.0.1.3680043.8.498.28437182087896443584321597116360338205","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46925959240037679795735705370016021232","series":"1.2.826.0.1.3680043.8.498.79179331524934790267205381671057085740","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47036446751578280498105392189811189787","series":"1.2.826.0.1.3680043.8.498.44365526814823394758154069367811819181","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47080537724499166054609508776778189902","series":"1.2.826.0.1.3680043.8.498.72210499381468872710990451274288893650","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47130872195336642107576758658144949813","series":"1.2.826.0.1.3680043.8.498.10400615635907007884749441563204275311","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47267174205472073784818634988669693642","series":"1.2.826.0.1.3680043.8.498.49348194842461516707545942156237708954","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47336007784831556446570294227732688335","series":"1.2.826.0.1.3680043.8.498.72604019043448128680164980731810760030","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47449893776425045020340604662487159684","series":"1.2.826.0.1.3680043.8.498.13053924557139387848581510398711563873","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47495385306095610608327984787779011133","series":"1.2.826.0.1.3680043.8.498.11288483455255915101803682749767783707","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47655269404919497034805209368250028981","series":"1.2.826.0.1.3680043.8.498.52041885796011363057692723956949790770","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47678121014268671436574141288012829166","series":"1.2.826.0.1.3680043.8.498.14588024958763571629229793617617374225","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47784065888386216103165400551121771174","series":"1.2.826.0.1.3680043.8.498.10825116397945866946834715022276083250","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47864423097056282241181943425451597067","series":"1.2.826.0.1.3680043.8.498.10950175761415304391653261974763568997","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47925688677871325890380893763458019552","series":"1.2.826.0.1.3680043.8.498.55713156297033203569285766033589401318","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48033467661219494489168041469309669510","series":"1.2.826.0.1.3680043.8.498.79968495249107284980718393223352025844","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48160569779413970442472500116176876660","series":"1.2.826.0.1.3680043.8.498.41068090559727243699000989681159700317","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48348615349418615469875801439348424274","series":"1.2.826.0.1.3680043.8.498.14477678774712126060440341154300476721","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.48417903971145834309474934717837250776","series":"1.2.826.0.1.3680043.8.498.48151977056426267488068061570459967995","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.48486018503198264262324670490116120612","series":"1.2.826.0.1.3680043.8.498.11476882743332567410674449071179819294","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.48531746264504523370547117933271161633","series":"1.2.826.0.1.3680043.8.498.84225547434549226852977006790212002805","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48658459018915173972212724570862424208","series":"1.2.826.0.1.3680043.8.498.73185015553836734640407117245642883427","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48771187298294322173131630438467163244","series":"1.2.826.0.1.3680043.8.498.10221484930984455692702028001764109127","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48784244479029704181755878324450207126","series":"1.2.826.0.1.3680043.8.498.58706026685033161589109237855658480444","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48855562507887692822284056171390459619","series":"1.2.826.0.1.3680043.8.498.10682396695610160681690017657355199670","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48918090379630186340047006216221526186","series":"1.2.826.0.1.3680043.8.498.12743362012088897400335133142466991789","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48977699132767849812826572970218325717","series":"1.2.826.0.1.3680043.8.498.11408167162797716528429310336085460791","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49047255106144196602523585429683869547","series":"1.2.826.0.1.3680043.8.498.66764005589618487200000098286293344367","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49094276335794811195117175603556929507","series":"1.2.826.0.1.3680043.8.498.11309157351352858482684205211050556500","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49186859342644783675130322288585477727","series":"1.2.826.0.1.3680043.8.498.78836795040368942589377984110841313271","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49287143478474218875661109488367501329","series":"1.2.826.0.1.3680043.8.498.93578210708411304247548165886088763988","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49379034023506626029648394840207360925","series":"1.2.826.0.1.3680043.8.498.13114880851576137559449532162041050050","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49443962795025989994487968185070137217","series":"1.2.826.0.1.3680043.8.498.65637195761761662482584329841621327514","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49492260217998945587168161555623476238","series":"1.2.826.0.1.3680043.8.498.93564524270110483248180515499690701470","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49575149159425506211571562732929634662","series":"1.2.826.0.1.3680043.8.498.18368610099869582088483884753460633427","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49603554452147117475589797389254733654","series":"1.2.826.0.1.3680043.8.498.12339236012351991087811177600357917477","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49703744074809050221928706774732375079","series":"1.2.826.0.1.3680043.8.498.51014923072923068177261980320793359598","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49847104080351361589960954776757484668","series":"1.2.826.0.1.3680043.8.498.43097791335269777567894073240418345752","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49855677166670499939366963036804180912","series":"1.2.826.0.1.3680043.8.498.13062751732114222646435238074739425423","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49884243466758407760126403944842240776","series":"1.2.826.0.1.3680043.8.498.86854204286003744093676373522118389754","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49926524898703962863714803838421958913","series":"1.2.826.0.1.3680043.8.498.10683197776482518663080857392391065047","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50051845967696843074391018845332480653","series":"1.2.826.0.1.3680043.8.498.12338165325801747235839178167964342827","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50073674465735509400964384004695041387","series":"1.2.826.0.1.3680043.8.498.20188599258147667852163348448534164419","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50109255875403812845391217314726500341","series":"1.2.826.0.1.3680043.8.498.51844017636017256311351210930243997979","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50183004658202912273462671442829139110","series":"1.2.826.0.1.3680043.8.498.32910088881734584764950562121994760114","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50220991646207533313828550949660577504","series":"1.2.826.0.1.3680043.8.498.10229886895133741367183750825626139919","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50261788831539215224639813005933285052","series":"1.2.826.0.1.3680043.8.498.38547761243193886045238530423010925743","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50342726517205151681842826322659275894","series":"1.2.826.0.1.3680043.8.498.83853222198558120364229284709788022079","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50424421287869498746932271082980000521","series":"1.2.826.0.1.3680043.8.498.52575825292714775533800515481156149696","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50487158921933419035809338652990508020","series":"1.2.826.0.1.3680043.8.498.10845328004144452774791248000263824533","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50571692547974449396635351108344029536","series":"1.2.826.0.1.3680043.8.498.59796077042191752082808410335949920217","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50666048236278074641860077012751071758","series":"1.2.826.0.1.3680043.8.498.11491991922654590224439184638227350008","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.50680622091377404463269575858175673158","series":"1.2.826.0.1.3680043.8.498.58952567096346464314089259966341575054","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50694733929204626284366818623993960684","series":"1.2.826.0.1.3680043.8.498.90464477090746094335033993757320847175","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50777252749942894325007417893973309310","series":"1.2.826.0.1.3680043.8.498.50156994169756978673431471354242598471","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50810596378229719761556926464290468652","series":"1.2.826.0.1.3680043.8.498.45820812638065800553177635636747743004","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50823279853002320074080586693604976784","series":"1.2.826.0.1.3680043.8.498.90122116565683830614800449521346707886","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50848799900564667487859253004538870622","series":"1.2.826.0.1.3680043.8.498.34422991430272680436952255236220490791","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50935247372419775856440294835587827419","series":"1.2.826.0.1.3680043.8.498.12326296137050405006698348128067639803","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51006813205605862503064447831381009425","series":"1.2.826.0.1.3680043.8.498.65074122612897999185704002988041774259","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51059575642903646008860869691477305022","series":"1.2.826.0.1.3680043.8.498.53583388023889263198717658928472591905","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51179760508587500025973916664101120069","series":"1.2.826.0.1.3680043.8.498.56848554807340510190673313569595196279","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51297777726540529860433097009654453742","series":"1.2.826.0.1.3680043.8.498.19912095905291790170678869087967298955","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51379389400506715529323532959275254794","series":"1.2.826.0.1.3680043.8.498.15192275962186990985570188487341107997","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51429653138043597611341459799295143598","series":"1.2.826.0.1.3680043.8.498.11572325954953082513736792653930830327","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51536234256866063100891626028999490545","series":"1.2.826.0.1.3680043.8.498.50897876716178267434200535830241309880","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51684117251092766010524179062007524257","series":"1.2.826.0.1.3680043.8.498.23443296660085567811395759789791199425","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51768733026033198941231586462443503827","series":"1.2.826.0.1.3680043.8.498.51923508030530023462927219697439332466","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51809092313496476873962641559638828608","series":"1.2.826.0.1.3680043.8.498.50220915953201999183869802290218633224","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51828180600648126726057100018010765613","series":"1.2.826.0.1.3680043.8.498.33558814461563328520582900116304196481","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.51837740845904075589194479205218060915","series":"1.2.826.0.1.3680043.8.498.77908976751506517835389114721137468067","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51910852739185117358682097116614556910","series":"1.2.826.0.1.3680043.8.498.70158237800351941636090287144742048810","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51970198810758997428998886534742243215","series":"1.2.826.0.1.3680043.8.498.67829055696932741644916395299742952718","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52010165788889734743850946282043771445","series":"1.2.826.0.1.3680043.8.498.10636876425919162283127749249962678477","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52046170319252016846878355722404423329","series":"1.2.826.0.1.3680043.8.498.99651601794886223011987800620676234138","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52098249444968803324423421624756966294","series":"1.2.826.0.1.3680043.8.498.10432374481448479492962601299241349851","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.52132258638267172087240021957310219977","series":"1.2.826.0.1.3680043.8.498.11048589721897608599570680863936237675","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52204390785696944340286127542242341202","series":"1.2.826.0.1.3680043.8.498.28366872095042181994944584329339581222","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52347547859131274836026480407333415929","series":"1.2.826.0.1.3680043.8.498.98183872475813842429458277516706395372","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52409113683924904563676007800283978062","series":"1.2.826.0.1.3680043.8.498.12040973448820930694278836134910451792","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52456272643822591755257802747431239152","series":"1.2.826.0.1.3680043.8.498.66507966678059791526049517258658984330","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52499379042723926379822784411787551701","series":"1.2.826.0.1.3680043.8.498.13141994106004736201294676204458259012","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52599302991259061057950834864441634066","series":"1.2.826.0.1.3680043.8.498.26291514713746332288860287503218986773","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52639459702574533721011212286428487650","series":"1.2.826.0.1.3680043.8.498.10223940419623434837128274200185317155","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52844966504845792334740560021996858939","series":"1.2.826.0.1.3680043.8.498.33505481496959745564562856997526083746","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52849154011665864497632297320961015159","series":"1.2.826.0.1.3680043.8.498.59493385656961152856070271577594946730","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52985748674486376309978868177895755694","series":"1.2.826.0.1.3680043.8.498.10039168824072025519905826717252750953","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53078461307849490495536549806091931824","series":"1.2.826.0.1.3680043.8.498.44968860697313761246446065802848740349","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.53114802640679819441521055029024037716","series":"1.2.826.0.1.3680043.8.498.12805951190879046535927626041685877224","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53137953094305471774552883960361922832","series":"1.2.826.0.1.3680043.8.498.15183548209841874280340839967857069539","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53173579822475854424126184867040523127","series":"1.2.826.0.1.3680043.8.498.10764277235615039143796620062399265550","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53262958748075737242820629127001253146","series":"1.2.826.0.1.3680043.8.498.15728119603143135974947399801279298882","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53331626867618182966740022261110760133","series":"1.2.826.0.1.3680043.8.498.67377713619104366528757124137570016916","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53355961816756477601843499985089590263","series":"1.2.826.0.1.3680043.8.498.68759449047542633846131004122763335581","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53370670740736715669115043403494610200","series":"1.2.826.0.1.3680043.8.498.58702908600035342286287428107054223324","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53490080963614272234072703760363052476","series":"1.2.826.0.1.3680043.8.498.11294289843723911767319012232478314962","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.53579416035647672067965892020505016454","series":"1.2.826.0.1.3680043.8.498.88329491824702062150385993467207269750","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53649895712172408476570770924813501118","series":"1.2.826.0.1.3680043.8.498.55001525560724286909982828831051336347","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53792159922324410857755054939229546555","series":"1.2.826.0.1.3680043.8.498.40880849057995270617384982963413898134","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53884715696186989798696544661396487883","series":"1.2.826.0.1.3680043.8.498.74396420144585983527029914302186287022","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53901077733518036755308692304268815965","series":"1.2.826.0.1.3680043.8.498.27706960862546627780791178076944673197","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54000782656400326230944526201670909133","series":"1.2.826.0.1.3680043.8.498.59255203128130338302324055167279316072","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54099199514076260519579096546600505763","series":"1.2.826.0.1.3680043.8.498.92993629839715323175829345461461066585","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54101830029742852876183926998159899424","series":"1.2.826.0.1.3680043.8.498.24658334147171636611246860888540992546","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54142170846176833568961426171739782214","series":"1.2.826.0.1.3680043.8.498.48903738156919402777847943634414790459","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54270039578867689669466339980682684126","series":"1.2.826.0.1.3680043.8.498.10559251738071010446253431565076394064","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54306467780023777062920737767581126383","series":"1.2.826.0.1.3680043.8.498.52185743804777788803128191025072067584","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54336962625483046366337626878609850219","series":"1.2.826.0.1.3680043.8.498.55213070625394167782041001688675002393","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54427039222108917725902677899081931510","series":"1.2.826.0.1.3680043.8.498.87350462636297082174609893045754822734","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54451515695681765924346479727204451807","series":"1.2.826.0.1.3680043.8.498.11893474983607630246492077004834951070","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54539703415639765348239790227512479583","series":"1.2.826.0.1.3680043.8.498.11172098823864555208096944227681977215","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54619696938940618755334224550745735815","series":"1.2.826.0.1.3680043.8.498.70267881713400611912041743121142227374","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54669065204808162914386927559244866557","series":"1.2.826.0.1.3680043.8.498.12413060835580829121483624681927373412","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54693572065777563403567480644050627025","series":"1.2.826.0.1.3680043.8.498.10357000764739340775810582686235984355","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54755249425443469939742671497042113665","series":"1.2.826.0.1.3680043.8.498.12377068076244001084971882161256618192","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.54792838438079861454532183998999249213","series":"1.2.826.0.1.3680043.8.498.14703118547242381638987914333502227720","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54977581323931277817074741182670080450","series":"1.2.826.0.1.3680043.8.498.57422781906807036056285204418650818704","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55015061060346597564205577461227720715","series":"1.2.826.0.1.3680043.8.498.56863401778944282993895786660313756187","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55065105486648830377200570585789726800","series":"1.2.826.0.1.3680043.8.498.12440069810888022392085831969636049754","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55105303802677863700162285680213496441","series":"1.2.826.0.1.3680043.8.498.51390757088418884133105790265386751082","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55300053252365037422745072128951402593","series":"1.2.826.0.1.3680043.8.498.25080436047463833117315635771230476213","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55344075316305815718253234447285669886","series":"1.2.826.0.1.3680043.8.498.64379557971331838056494285444450833640","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55397602116058381866334455456729470963","series":"1.2.826.0.1.3680043.8.498.45621542904368721074783868838801748214","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55461938012492393262172182405932212246","series":"1.2.826.0.1.3680043.8.498.48114332935259415387216870376142715428","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55653770127824574123236841437998533498","series":"1.2.826.0.1.3680043.8.498.11471473164105665157945956843904026707","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55700532609077428304194916132759352518","series":"1.2.826.0.1.3680043.8.498.25485781932288149101306961240877985112","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55729193057949125029301812320609764642","series":"1.2.826.0.1.3680043.8.498.11433055243773264810574052103223007278","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55808772152667135571939417929724526296","series":"1.2.826.0.1.3680043.8.498.36391342583835277672952535296113396361","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55861544093014499478739864837675161978","series":"1.2.826.0.1.3680043.8.498.11738153680443515182493007500657918018","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.55927741980493661714408715886641034285","series":"1.2.826.0.1.3680043.8.498.91017183900480405698045978311425774622","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55971241672517044174019654998425819738","series":"1.2.826.0.1.3680043.8.498.31668422074580420885672865336218767102","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56016606949603829995818303105991224499","series":"1.2.826.0.1.3680043.8.498.10215425136918705663399826558215215382","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56061505201159269536816810086342635391","series":"1.2.826.0.1.3680043.8.498.99876245666912456361098678800165100212","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.56146582420561901778012523946835033628","series":"1.2.826.0.1.3680043.8.498.52098492412008198092854974660293039366","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56170570792418244380186761574367557535","series":"1.2.826.0.1.3680043.8.498.58993443376892706842582527942491320544","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56266280744044680626884459696038109616","series":"1.2.826.0.1.3680043.8.498.21878393953126645108441108409891864278","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56385473079196655120993854770623075553","series":"1.2.826.0.1.3680043.8.498.91404382429768318981521981372231231638","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56474364950531483257947155855279443851","series":"1.2.826.0.1.3680043.8.498.10201315045472865274359404567701425059","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56512564464301544960446419143135447363","series":"1.2.826.0.1.3680043.8.498.78093385660449749624663646675509261313","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56564801707939049324472369774251260607","series":"1.2.826.0.1.3680043.8.498.35983006535938526615959696750431623037","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56686055146646210144964207684044815006","series":"1.2.826.0.1.3680043.8.498.15493338434417466610724684766462766961","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56762592839620371588520593573846414860","series":"1.2.826.0.1.3680043.8.498.57946025445401966329387026286689911102","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56796622077427019119540733986070534126","series":"1.2.826.0.1.3680043.8.498.12419841013842964732013514636220901687","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56814987912934301686833132930648230999","series":"1.2.826.0.1.3680043.8.498.59519819550842477630478850913043922287","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56882015310689536014061315110969323429","series":"1.2.826.0.1.3680043.8.498.12544842025953705908005227822117128640","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56894990686683651282564584582335346350","series":"1.2.826.0.1.3680043.8.498.13569546275343480813288533751724023543","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56945039212126810522718038626526579591","series":"1.2.826.0.1.3680043.8.498.93089116786854542829677860811484753230","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56991590802231436015843619633497607680","series":"1.2.826.0.1.3680043.8.498.53998740877237930019841533032498093151","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57064618560116981041722203028857007300","series":"1.2.826.0.1.3680043.8.498.69439258318746977798517141743638329241","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.57084445429567494093960261980941917499","series":"1.2.826.0.1.3680043.8.498.91796943579838140236006886094701175553","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57160448024145680660335079050450776852","series":"1.2.826.0.1.3680043.8.498.10965728973823842729329728143245853060","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57342523828166309133982222435655662667","series":"1.2.826.0.1.3680043.8.498.61134243780199632871178310555254603708","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57491375928585600499323136123872536597","series":"1.2.826.0.1.3680043.8.498.60510107342747564440823675637424317083","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.57615268093937994500995086275007694564","series":"1.2.826.0.1.3680043.8.498.72279653371453845468021757154438124465","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.57775389384191131322519457853517177210","series":"1.2.826.0.1.3680043.8.498.46514963552985833261243015714238746069","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.57802887045514681373294932749517914959","series":"1.2.826.0.1.3680043.8.498.45685577193511045309135749778303927314","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.57836524675365700674244122764523161699","series":"1.2.826.0.1.3680043.8.498.67682700353087702953950044841225227742","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.57859543314667538612269642333387175763","series":"1.2.826.0.1.3680043.8.498.90475827877885742072914577856917667335","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.57917634151851433037090858498504057826","series":"1.2.826.0.1.3680043.8.498.17560593655813087967593361309485431904","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58051085126748057986491277367254533832","series":"1.2.826.0.1.3680043.8.498.36790593179022832608174878701522576121","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58167256197067081436345136244032069163","series":"1.2.826.0.1.3680043.8.498.74901081164233228295547547442551178195","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58280111882249056715877316026191844548","series":"1.2.826.0.1.3680043.8.498.11600907113450964774847431470526752314","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.58298767414933858487578597918936525600","series":"1.2.826.0.1.3680043.8.498.19244800242885118209206115294893179689","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58381033890285289577066078050126251787","series":"1.2.826.0.1.3680043.8.498.87374510556136730141840757570615170298","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58475208045272914292874190001248518687","series":"1.2.826.0.1.3680043.8.498.83821174701884939144725967098468784750","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58482779112389764165599701941310919660","series":"1.2.826.0.1.3680043.8.498.93042735800071746030358103894729256120","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58550899000434548218916895907938175917","series":"1.2.826.0.1.3680043.8.498.49458757725027388642276465585520181420","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58626635384305289939700801774393536415","series":"1.2.826.0.1.3680043.8.498.29498693084179669311997182346084737507","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58685941646495891824559391997783302656","series":"1.2.826.0.1.3680043.8.498.83530487114102849847652736511661748416","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58728013757221328590898166513995249334","series":"1.2.826.0.1.3680043.8.498.12679838378628450252154543511382154075","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58749835013162544157291112101941849646","series":"1.2.826.0.1.3680043.8.498.75153258602617461780218432172172762716","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58776844157027040145262759105410792687","series":"1.2.826.0.1.3680043.8.498.11986299954585583353949348140794599817","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58835863392949696556026907211309159182","series":"1.2.826.0.1.3680043.8.498.48887764538068524196133512289684069217","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58925843611945635442423587804169242644","series":"1.2.826.0.1.3680043.8.498.53609051180244203550475250368649441132","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59021913092219789232890327780064145948","series":"1.2.826.0.1.3680043.8.498.83135088442175013093513126589694574736","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59048646289883992375610835845636263714","series":"1.2.826.0.1.3680043.8.498.62386225980902627552056073631903130152","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59126324546257448818397467078224944768","series":"1.2.826.0.1.3680043.8.498.48741299259735540553233671830151967441","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59254379036462832374262221063106167294","series":"1.2.826.0.1.3680043.8.498.68362889750971025904954502335109155221","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59394856717216352192037635065925607933","series":"1.2.826.0.1.3680043.8.498.98994608811262189595584846021733652879","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59449702487184611096113620550399616423","series":"1.2.826.0.1.3680043.8.498.15807804564321256228469716829256020268","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59476657510212258499047356518058209126","series":"1.2.826.0.1.3680043.8.498.13949403750334376276606794122232557324","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59527896591755229337596505685198957637","series":"1.2.826.0.1.3680043.8.498.22741134108766280720679656881850726469","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59646951619367295963072805933618947027","series":"1.2.826.0.1.3680043.8.498.45556974437202683612445350514343766522","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59758232061190668535098343218115599228","series":"1.2.826.0.1.3680043.8.498.54865031004505678637447358297652038556","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59828592894392286103154555572376381359","series":"1.2.826.0.1.3680043.8.498.11532282880993362407526811363665768041","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59923507410103964137819195926587517759","series":"1.2.826.0.1.3680043.8.498.30682506496176055851344010498307259473","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.59938475193950583662125713924156951751","series":"1.2.826.0.1.3680043.8.498.94923635985020499021928592580755142501","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59948559729295284519115137429127740378","series":"1.2.826.0.1.3680043.8.498.60955172934918646859114587034872499804","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60064080350033586611530748685370040101","series":"1.2.826.0.1.3680043.8.498.11320960230172804483959204464278402704","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60094182309575937331152278560900767484","series":"1.2.826.0.1.3680043.8.498.12743972636021936108105052539937912188","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60189166269847256202786309096136242746","series":"1.2.826.0.1.3680043.8.498.10088908044697796302515976154359387474","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60259828181878278590614805828898992241","series":"1.2.826.0.1.3680043.8.498.11995820320200346444336251362952180650","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60347768295306794807599672836772410567","series":"1.2.826.0.1.3680043.8.498.10407335691515679974821358987744854656","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60435128564448668334758790827765960420","series":"1.2.826.0.1.3680043.8.498.12806287882502955129401007601614680763","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60446085094710623303973373214794179184","series":"1.2.826.0.1.3680043.8.498.12150133794865325756617147261652643750","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60477027832057450004004942199870813886","series":"1.2.826.0.1.3680043.8.498.53339248268206266939457649964213152722","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60548477793215320533051194155539307724","series":"1.2.826.0.1.3680043.8.498.81386454064012403785977767005174724640","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60611146141493700985009985723275645474","series":"1.2.826.0.1.3680043.8.498.87406579760676680645864924040006335087","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60661676782543347731000874708337132276","series":"1.2.826.0.1.3680043.8.498.25564245679223214883450948445386725971","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60736996513214411991331917490659995815","series":"1.2.826.0.1.3680043.8.498.10033005616764317111880819291437805791","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60931716414402855657852057585917605786","series":"1.2.826.0.1.3680043.8.498.11536229713914197972902757736844648165","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61024356179284390979230045743542661223","series":"1.2.826.0.1.3680043.8.498.12326810621836672214470093611029939220","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.61072566792285434404885416160728089641","series":"1.2.826.0.1.3680043.8.498.27502239812376656802939626249915143906","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61148667330113929729568621580194520169","series":"1.2.826.0.1.3680043.8.498.75436914385797122243082408454805817065","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61227644940421894951950605627016308160","series":"1.2.826.0.1.3680043.8.498.28903461005319159111125181268927310611","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61313243050348988917987796330691448770","series":"1.2.826.0.1.3680043.8.498.12449651313152665198030327940351019024","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61435096069921474761812731150911059491","series":"1.2.826.0.1.3680043.8.498.14109748365861787992916921564935955091","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61490475141024845836646569993549106628","series":"1.2.826.0.1.3680043.8.498.10160915079525602651118792685086286045","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61529451559250327125552112075119161933","series":"1.2.826.0.1.3680043.8.498.35599487522311491232531402925264334241","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.61570904948128887877542043572765432417","series":"1.2.826.0.1.3680043.8.498.12898653598437372012868887151090998072","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61715630583919056026446344483871058799","series":"1.2.826.0.1.3680043.8.498.51601096887628152744284220684290052944","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61864939963243675140060416958055914981","series":"1.2.826.0.1.3680043.8.498.94886348121265711654625663605229496009","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61934170443339188744573234190617483017","series":"1.2.826.0.1.3680043.8.498.53976057252559359796406579886874674582","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62025036385410820041698409408092924893","series":"1.2.826.0.1.3680043.8.498.63398182592634221648126697109620860456","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62109244272539959226824769201542205950","series":"1.2.826.0.1.3680043.8.498.92875479708470816571469601345475096401","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62205799963299902160268565319924399775","series":"1.2.826.0.1.3680043.8.498.74278390613568866758660832530599708893","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62218684089117864632544493263539598403","series":"1.2.826.0.1.3680043.8.498.74292976294266178689628331143147477237","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62252176347004905787746821113523889523","series":"1.2.826.0.1.3680043.8.498.39273931396409388979816464097918309405","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62400371770962739002393364782172386752","series":"1.2.826.0.1.3680043.8.498.12392385874808941511732045782506116231","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62420632149863295140175740643640707423","series":"1.2.826.0.1.3680043.8.498.42811497567929838081904759956570259830","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.62465595376489211274225312453216559395","series":"1.2.826.0.1.3680043.8.498.59400018247623467195394787723432499465","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62693151030819598514729637644804213983","series":"1.2.826.0.1.3680043.8.498.60863514151967605683380287559360744241","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62760659205261492447481830774781950875","series":"1.2.826.0.1.3680043.8.498.31513733632641414096041478551176883282","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62830031603985608774418534497520663269","series":"1.2.826.0.1.3680043.8.498.12876204402707657548841022454632016083","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62924835279557724526138406447993672103","series":"1.2.826.0.1.3680043.8.498.20256831532816583733976151491067763734","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62983291359021407345388012674614537807","series":"1.2.826.0.1.3680043.8.498.70977439274357516291375538554661909489","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63068687840516335034631353955761036465","series":"1.2.826.0.1.3680043.8.498.19097073527868332500133040762056039442","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63096952391833769395204944538276970114","series":"1.2.826.0.1.3680043.8.498.13190075907558876086789285048330357651","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63126604639589769235241216875934773992","series":"1.2.826.0.1.3680043.8.498.11720254543214861396482080747287375317","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63197503121546251937003511751049515360","series":"1.2.826.0.1.3680043.8.498.10372115439082147247895598797683254113","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63216944317982327305447295456609309952","series":"1.2.826.0.1.3680043.8.498.11656811838961437334533853863632271444","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63242029756172180508167933519465902827","series":"1.2.826.0.1.3680043.8.498.11145869802906535411142466766326921317","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63280826430574305683941548157934410641","series":"1.2.826.0.1.3680043.8.498.10154063991074306889958587418080328403","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63377901538320536934403969965624202519","series":"1.2.826.0.1.3680043.8.498.89204547051971057504029294408693724857","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63426442500798189991842988966227275166","series":"1.2.826.0.1.3680043.8.498.14351129593649633208706544647724867373","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63510378689078892579575716369639643386","series":"1.2.826.0.1.3680043.8.498.39496989532030157851697744227887690907","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63521726454879203860288713455843702564","series":"1.2.826.0.1.3680043.8.498.12768786626052319236009835357468903680","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63550258873504696667112631869753870779","series":"1.2.826.0.1.3680043.8.498.50419626698937283479973182462711631359","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63564058504808107209753434989270686941","series":"1.2.826.0.1.3680043.8.498.46328935102518365746975310688709059174","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63673733836218886397484265807806548473","series":"1.2.826.0.1.3680043.8.498.22066632036163267774309140075064501393","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.63797588614544518196205411126336672334","series":"1.2.826.0.1.3680043.8.498.11511695079226647826954175315661213670","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63884838982826315257233262184190435906","series":"1.2.826.0.1.3680043.8.498.91195020870892183368931436110737181094","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63986652686300452077410299429386420400","series":"1.2.826.0.1.3680043.8.498.10910872679082391056451747535599138220","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64029768638431636860053484202080135316","series":"1.2.826.0.1.3680043.8.498.66522863643571842033949316225159000904","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64109231850522212435708176045583779359","series":"1.2.826.0.1.3680043.8.498.29052784760152232718216899649366735895","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64172939203926034064618447795131041647","series":"1.2.826.0.1.3680043.8.498.66745447937781325627331504532884245657","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64283618749539713756220539395944025878","series":"1.2.826.0.1.3680043.8.498.12577041265154901467738370516342180782","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64325734401816269429960464734504322970","series":"1.2.826.0.1.3680043.8.498.10664393847034029110537303841908286361","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64408609256163127278435484217683272910","series":"1.2.826.0.1.3680043.8.498.36393041690910660528146908425612939726","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64421299492271548595446260681698412384","series":"1.2.826.0.1.3680043.8.498.49679788088943092436127219883930878635","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64526757783166918696475501252110447010","series":"1.2.826.0.1.3680043.8.498.10895713111352891463424204426383876722","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64590392444259942263746282444188079724","series":"1.2.826.0.1.3680043.8.498.92229289034200301718543240685935640241","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64652085172505930317660036583152917472","series":"1.2.826.0.1.3680043.8.498.91635920642625419603487542429088490088","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64673069370410118387363952155758553567","series":"1.2.826.0.1.3680043.8.498.43799879423262439980477156288475253569","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64865300853258390816475799272221318005","series":"1.2.826.0.1.3680043.8.498.10714041891756060264625726242555745977","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64946212153370539647301308676689632525","series":"1.2.826.0.1.3680043.8.498.11542344488227491444766222217494782129","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65013994414976103275014874284064145461","series":"1.2.826.0.1.3680043.8.498.44534847026398229730024367165784964377","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65123055741109319778634264103224173252","series":"1.2.826.0.1.3680043.8.498.13083579891704477173018689803956724486","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65160241567146620406193593170843451582","series":"1.2.826.0.1.3680043.8.498.12079416759919919507773340026925288914","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65224808465738083245406235000047993647","series":"1.2.826.0.1.3680043.8.498.23916815711014849929788333129308018037","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65312506382127486486519184894274326380","series":"1.2.826.0.1.3680043.8.498.12809141588602966110770372397964104529","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65403984120528711726317048984986599537","series":"1.2.826.0.1.3680043.8.498.10879729737134851481973435737976489268","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65515828911930515002046151253057175178","series":"1.2.826.0.1.3680043.8.498.93116508758911487321670462070766179722","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65543472488214519069910341591668399186","series":"1.2.826.0.1.3680043.8.498.74977096685268390785075756095106996910","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65597375829884314152221851871550349856","series":"1.2.826.0.1.3680043.8.498.22358615316746570343697539079675270815","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65703265163052901613211590083675186626","series":"1.2.826.0.1.3680043.8.498.51474358496541633753352760943093784241","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65715827585629489888518528736848969678","series":"1.2.826.0.1.3680043.8.498.42079770108947877834759865074786556920","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65783500506568702195262661666345236666","series":"1.2.826.0.1.3680043.8.498.12252427970362615885787607069034933104","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65792625478083832901309715174678358900","series":"1.2.826.0.1.3680043.8.498.50260941139825565039522629413307797010","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65855247249608663104822603172047250057","series":"1.2.826.0.1.3680043.8.498.34976339868392591500162183642682262383","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65909865035783831607773206653044421345","series":"1.2.826.0.1.3680043.8.498.60931403465305188619055218511800133810","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65977711270942843864266358352913175672","series":"1.2.826.0.1.3680043.8.498.11940285557221297356502556521983559735","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66022464665791424345622894674375661089","series":"1.2.826.0.1.3680043.8.498.11597928343357499411504869798572802224","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66077420344931334696112616504179985591","series":"1.2.826.0.1.3680043.8.498.19608178277985096551001929848920035864","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66199248385068685539872840826351152652","series":"1.2.826.0.1.3680043.8.498.22211589231676088467237990269527626898","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66264392187277583747691643146920431469","series":"1.2.826.0.1.3680043.8.498.66702344704497909486823880922932112275","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66305771884507759587182077473863996177","series":"1.2.826.0.1.3680043.8.498.46120240614742420025046064985330034914","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66321903250295261063309947910189882952","series":"1.2.826.0.1.3680043.8.498.57517844980917118706737311816390181086","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66364462115470823900176770413527217503","series":"1.2.826.0.1.3680043.8.498.42818175336848417988004093434558544098","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66404725797392899950170138361835649917","series":"1.2.826.0.1.3680043.8.498.39221159311378653658500978478839410362","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66596090141798159058634289358454183351","series":"1.2.826.0.1.3680043.8.498.53088463012358377421589215325235771446","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66690100241274246406145618641481787709","series":"1.2.826.0.1.3680043.8.498.12240857718265647542828820348858006420","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66805139646641294853509958688742554525","series":"1.2.826.0.1.3680043.8.498.13181029423242634522031899417192636667","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.66865810098959732285731429066569782524","series":"1.2.826.0.1.3680043.8.498.44617454016541159177878134881546178470","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66993076561387452114335652021593805727","series":"1.2.826.0.1.3680043.8.498.51564246276799999805291611929194447152","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67153684605171746899496816770567866269","series":"1.2.826.0.1.3680043.8.498.47842987401818702129631112790380221388","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67188121544063723837669983597453941774","series":"1.2.826.0.1.3680043.8.498.91288640121632138425669406741773825040","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67208608901061914528631080268819069203","series":"1.2.826.0.1.3680043.8.498.68493377404341687365286137370035029379","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67284681623851829916403311739987186159","series":"1.2.826.0.1.3680043.8.498.61430449200274344534253615584110553473","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67386780309737226410965627751074124678","series":"1.2.826.0.1.3680043.8.498.39299993324033442890190617209469179112","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67456402757136973484427155324664966595","series":"1.2.826.0.1.3680043.8.498.70651360938121234801929969266137769065","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67490059263503953166467514121525025332","series":"1.2.826.0.1.3680043.8.498.12356153610773960444399949403102629003","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67550262192397377344258276170992188234","series":"1.2.826.0.1.3680043.8.498.22384759470759817168056761210883675476","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.67613388743126519966672593774754849330","series":"1.2.826.0.1.3680043.8.498.81937290804235659605660488362505752733","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67678576373934022409084801948795250361","series":"1.2.826.0.1.3680043.8.498.38703689805709163434065392515322660641","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67716063997837501476742746033172868748","series":"1.2.826.0.1.3680043.8.498.12905699172892348528102721807932985699","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67740000270419228186274186410889719154","series":"1.2.826.0.1.3680043.8.498.11411795102312839914883636755555253826","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67788047649994083591931245362169694306","series":"1.2.826.0.1.3680043.8.498.69839990227932433660133957390706484314","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67860057199678546706175336855949799228","series":"1.2.826.0.1.3680043.8.498.95858235087208472983811986445765755363","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67902658411525109462089381503197501728","series":"1.2.826.0.1.3680043.8.498.74457369752051953696195014253151046749","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67941678374744711267840655419606994796","series":"1.2.826.0.1.3680043.8.498.50704473953263377240045069989802036538","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67981539513814867568631217960473023750","series":"1.2.826.0.1.3680043.8.498.74259354028490464049499064881665216550","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68079818232244983808025467536454890669","series":"1.2.826.0.1.3680043.8.498.11309803557147807579898350640387104929","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68106879690636836695681159513106553666","series":"1.2.826.0.1.3680043.8.498.34332324815499947514915273636657358733","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68142054543837964766571710567214307830","series":"1.2.826.0.1.3680043.8.498.72867604533508072272918985754775314691","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.68166103964518457268906879600985959436","series":"1.2.826.0.1.3680043.8.498.22929371502457108005488900868697611074","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68219942161081583831276769978472014317","series":"1.2.826.0.1.3680043.8.498.10467727209975163907690268922575080782","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68256141451764547710152696919981893952","series":"1.2.826.0.1.3680043.8.498.94070978746291333273249531711550793367","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68304853223764870957854101604223809218","series":"1.2.826.0.1.3680043.8.498.82886176932853780646981833859904943964","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68328948893840135511720724977892905019","series":"1.2.826.0.1.3680043.8.498.52930636977874490650404151015516881148","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68435927323644457286635872824563629385","series":"1.2.826.0.1.3680043.8.498.39171988588689109803487189041632312871","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68448811873610037553554213141537599150","series":"1.2.826.0.1.3680043.8.498.13411041938249749141790162040363951885","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68570907895010847368926943092385685462","series":"1.2.826.0.1.3680043.8.498.38216356081642743670762738533149892192","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.68578659816221770558506780445608768600","series":"1.2.826.0.1.3680043.8.498.11366290547590293049394905090305050015","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68637419175382604890136839259344633105","series":"1.2.826.0.1.3680043.8.498.33091597369275269387110982780938444853","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68740466085414970518655922465908223436","series":"1.2.826.0.1.3680043.8.498.64934493728712281296364021855099086143","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68777783869253673804970857981124324335","series":"1.2.826.0.1.3680043.8.498.21654198182103929375916031853079004466","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68806105227826942108149090508198910591","series":"1.2.826.0.1.3680043.8.498.57229178542028969917243658742484525578","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68931510756844060683643298583132884960","series":"1.2.826.0.1.3680043.8.498.25990201675561181060471329571779275179","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68971603226593154050514800137550171537","series":"1.2.826.0.1.3680043.8.498.27696358745480866284938849982138632037","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69033651423155214548081618535669641812","series":"1.2.826.0.1.3680043.8.498.64286714071449175801679188359427475014","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69141410772327587821159027820329185006","series":"1.2.826.0.1.3680043.8.498.32864493543185905176033380422131332564","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69236835611860340913671698884308946455","series":"1.2.826.0.1.3680043.8.498.12579455638562819340460194793126300668","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69267821440055811776827497076481587198","series":"1.2.826.0.1.3680043.8.498.12790264401692992085871757140455328710","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69397705155385306818511499752200941968","series":"1.2.826.0.1.3680043.8.498.56468774059928214906209521160959754051","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69417319234704516520173921900954337376","series":"1.2.826.0.1.3680043.8.498.61830153207462382675820314073373429889","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69495014653333523580255444915082056825","series":"1.2.826.0.1.3680043.8.498.98947206047549908583571863498014703128","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69596668032633976121623939445795750384","series":"1.2.826.0.1.3680043.8.498.12379176500307849154115011593260311076","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69635429172929809539574727887787303282","series":"1.2.826.0.1.3680043.8.498.30690359493179815366862046577303596072","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69673283369917182409597968479300552579","series":"1.2.826.0.1.3680043.8.498.25525249641878376913328103502443033149","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69730556680449158538173708009262014812","series":"1.2.826.0.1.3680043.8.498.12877681046585985878641929588965669389","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69828573582708185393942443598783108800","series":"1.2.826.0.1.3680043.8.498.10426908051287807982200701421718276110","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69930569170269472474738549886709596023","series":"1.2.826.0.1.3680043.8.498.13009575392343898663648687738496202529","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70124572708965486791612654433965515063","series":"1.2.826.0.1.3680043.8.498.65836375281148112535554945310071545145","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70204818275951821519024007466988533363","series":"1.2.826.0.1.3680043.8.498.90359436030552164188037381722081910162","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70291739279544255146458190540233023337","series":"1.2.826.0.1.3680043.8.498.99362076637708614371418704287390871497","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70358024114589255428192032298530338260","series":"1.2.826.0.1.3680043.8.498.92538094028652645388550078453602693431","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70409266114961050236894235887802126808","series":"1.2.826.0.1.3680043.8.498.56793131162914323862365815497089211432","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70464717849928808075676221329034929056","series":"1.2.826.0.1.3680043.8.498.48400419394436495732369543521147549487","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70478068758081824347762712452252763894","series":"1.2.826.0.1.3680043.8.498.10164420145174100771789335128134336859","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70532653867267627708152426358320452518","series":"1.2.826.0.1.3680043.8.498.85562068864681479334879424174166756931","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70674155597869824035214737999044110827","series":"1.2.826.0.1.3680043.8.498.59683791864458662298971075220349060262","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70796360440320543674384511859375295769","series":"1.2.826.0.1.3680043.8.498.84964706707614046751499049842735893115","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.70865607713855272648070757909359669526","series":"1.2.826.0.1.3680043.8.498.61134968267567828084017611087499501194","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70877157427595288430686609728186100169","series":"1.2.826.0.1.3680043.8.498.96940038227754432127780810344272704289","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70913971933753517251340687745718346828","series":"1.2.826.0.1.3680043.8.498.50753230570516750997367433144751494215","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70931245733460964524244528230239226104","series":"1.2.826.0.1.3680043.8.498.12316832726279835923569951550312234173","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70955936086115471632446529979621948261","series":"1.2.826.0.1.3680043.8.498.95874093835152410657092639688906915821","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70984418257970716669463829979819981861","series":"1.2.826.0.1.3680043.8.498.10574102549068785120150299817609655904","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71063550263564056105844586167361750567","series":"1.2.826.0.1.3680043.8.498.11634483354716475167427786892932277708","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71157888664602228768823145752952837565","series":"1.2.826.0.1.3680043.8.498.18840237195477521315952133725659991304","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71311031275189592818091380416268522846","series":"1.2.826.0.1.3680043.8.498.47509260665434664565058319271034930427","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71388532549899713881913113571627425347","series":"1.2.826.0.1.3680043.8.498.45722918284219910573603807234632100873","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71507059468810869004142324822642900237","series":"1.2.826.0.1.3680043.8.498.80390739089621736300540570821604481932","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71660070729774772658965427776586988585","series":"1.2.826.0.1.3680043.8.498.46751948412894364807744629542688232765","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71686717249431292334159336816742076478","series":"1.2.826.0.1.3680043.8.498.12849359094645625361882703939722682571","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71707317731065645278250956442336515504","series":"1.2.826.0.1.3680043.8.498.41810761139560246320041693881064442511","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71736046309441396676679830874326797437","series":"1.2.826.0.1.3680043.8.498.57364844271929320531179673853700323881","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71753603955749189961638324806586359888","series":"1.2.826.0.1.3680043.8.498.66463181649204769921648434822393417536","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71792468450491064417953341293663396190","series":"1.2.826.0.1.3680043.8.498.13109692608135055362186142277268480460","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71830371861099586573789550797712053914","series":"1.2.826.0.1.3680043.8.498.56258513288925128604003943112895213177","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71898908031968158312013185809213669768","series":"1.2.826.0.1.3680043.8.498.93387252658776162760630249712510307909","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71985417286226511458068110832784023627","series":"1.2.826.0.1.3680043.8.498.56435300300248350330506079550296392924","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72017096768214637927798648334301077145","series":"1.2.826.0.1.3680043.8.498.11594588530908013609615869369406562828","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72023797276837219152365428999317131110","series":"1.2.826.0.1.3680043.8.498.50555257219929622347006692326953110622","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72172265826902387784016098389863005666","series":"1.2.826.0.1.3680043.8.498.78031233539552824193292976140182607851","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72185504245484151161044972024469667666","series":"1.2.826.0.1.3680043.8.498.94464430362301428792262815353871283285","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72248728319398595146365363062799556637","series":"1.2.826.0.1.3680043.8.498.79015991358090694441124180370103546434","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72255181548595299889516400134053439401","series":"1.2.826.0.1.3680043.8.498.56314381170020160412533890160599592397","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72313325634455670216423864504103231505","series":"1.2.826.0.1.3680043.8.498.35382315677878679643758133866889984729","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72368645606329000923061796699431437924","series":"1.2.826.0.1.3680043.8.498.62980807387063805994149296100832927369","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72453426317225885428514595399496580896","series":"1.2.826.0.1.3680043.8.498.21476930428187294580749519238306514015","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72495590491484814220060275823761767516","series":"1.2.826.0.1.3680043.8.498.14784314868477253785798100825118197709","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72547312490317214326791189007411860203","series":"1.2.826.0.1.3680043.8.498.31400126119479025571974862490159949849","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72653666963736486779382314125782779004","series":"1.2.826.0.1.3680043.8.498.11943054064750850060017679791841038235","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72697172162233919540021967137047646179","series":"1.2.826.0.1.3680043.8.498.10353981675248545112707085203125700554","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72771360268592017354044657882090174639","series":"1.2.826.0.1.3680043.8.498.81535593488754812895248952859908800503","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72800659596099468282135738543971261904","series":"1.2.826.0.1.3680043.8.498.88208105502029063326875243616289560989","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.72853333220043794904856138561095171921","series":"1.2.826.0.1.3680043.8.498.41786504123190759621834893248893641957","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72896586188735703676995894830815493004","series":"1.2.826.0.1.3680043.8.498.61704609122075724323257742507509958127","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72937134140407212536042212109839895294","series":"1.2.826.0.1.3680043.8.498.22049131065523944034823574692548395318","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73021428171736137311489171988512484948","series":"1.2.826.0.1.3680043.8.498.58278851198264110236304604137274614176","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73037642285055457162691486258363101398","series":"1.2.826.0.1.3680043.8.498.10327361441658978145469298906168547711","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73083786204651635096412213233334934941","series":"1.2.826.0.1.3680043.8.498.10669712146984044384921802104176371936","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73170541731840260065335942048010387599","series":"1.2.826.0.1.3680043.8.498.86395925017969923522465823278245211506","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73205916973275504240156555900460720571","series":"1.2.826.0.1.3680043.8.498.52102934030912159621352934881037849845","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73231153161312542696053060380619432126","series":"1.2.826.0.1.3680043.8.498.12876544648425017443970583829795077104","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73254365064404586162477089636000427751","series":"1.2.826.0.1.3680043.8.498.35393821097652825836107080534418089687","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73275019425494043461771574462523171618","series":"1.2.826.0.1.3680043.8.498.71165601792724458458563543161840220884","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73356831368999268488355650751758108912","series":"1.2.826.0.1.3680043.8.498.10235184796611231755729005914270711113","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73448420644251136304786298254801920284","series":"1.2.826.0.1.3680043.8.498.79687589874326857915156562639454161457","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73480422982315825699847838826691577320","series":"1.2.826.0.1.3680043.8.498.13397538542811398051893510196201814275","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73527530686853911124431549317032662220","series":"1.2.826.0.1.3680043.8.498.83867942217836552185114474898876718504","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73567807253694728043552196136704821457","series":"1.2.826.0.1.3680043.8.498.75288414517734607232109288896906598492","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73616237718955147476241731466078978343","series":"1.2.826.0.1.3680043.8.498.37956714234760765648151787312014117168","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73748007182569068263280098473557128226","series":"1.2.826.0.1.3680043.8.498.17369042002263807867417749372260807374","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73825252186234755453650759159564715478","series":"1.2.826.0.1.3680043.8.498.79623010268978043382386426192447144533","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73926443729786165628848843707532839995","series":"1.2.826.0.1.3680043.8.498.13310307807795355391194629563231191821","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.73988798714023285209290281409329527637","series":"1.2.826.0.1.3680043.8.498.32758876848192010546254386794245895713","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74095402538454760179339842746572802563","series":"1.2.826.0.1.3680043.8.498.11898384420833535604084204397740001474","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74140806279409854828032496578072554031","series":"1.2.826.0.1.3680043.8.498.42468047346176646806723383776568822544","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74206939384612669977629625310277500302","series":"1.2.826.0.1.3680043.8.498.82604002343043716935044577246517371001","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74215611637606164186239511504256137871","series":"1.2.826.0.1.3680043.8.498.54923252877434865050117984798405301368","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74243110959141459268838217468465880820","series":"1.2.826.0.1.3680043.8.498.51878769015121687535773851518034372470","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74247325306307303036016759070403973520","series":"1.2.826.0.1.3680043.8.498.10645632245634241856701769193358494740","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74332475620045145049628676751586693385","series":"1.2.826.0.1.3680043.8.498.10419539575377580606457311182308169512","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74388661917632804671669952747214084502","series":"1.2.826.0.1.3680043.8.498.10142979507556274668701186450874144647","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74461421748415064477531528884191890661","series":"1.2.826.0.1.3680043.8.498.10589226182454647046237348496111145953","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74471724874857406578316075456414902954","series":"1.2.826.0.1.3680043.8.498.59618969021803916015411240457336186735","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74688733778107958348746767768064374883","series":"1.2.826.0.1.3680043.8.498.30772577039582820846831651666223468252","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74725806565437665548056410415178280121","series":"1.2.826.0.1.3680043.8.498.10494122886053965022843613169457293631","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74864145321761783409935288797327597738","series":"1.2.826.0.1.3680043.8.498.19053900654987022819607937654264131430","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74914646210033032631540802996162814268","series":"1.2.826.0.1.3680043.8.498.14600188276380252737041609765700390202","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74940598515597814600837126537274199312","series":"1.2.826.0.1.3680043.8.498.67168798528877486177186948363871075091","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75126165133796914746114117844697263697","series":"1.2.826.0.1.3680043.8.498.13374688719942034873189408239639465168","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75187434248356774277526985329346125190","series":"1.2.826.0.1.3680043.8.498.98676746789639205361719427943243688577","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75276563505367629340309025030148269450","series":"1.2.826.0.1.3680043.8.498.69391363317394060435742378729425064328","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75384600248630012126841890313149674691","series":"1.2.826.0.1.3680043.8.498.20302014879450805640242625933356578752","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75419034794059601820999512200601915669","series":"1.2.826.0.1.3680043.8.498.54422428606076815002627800687853212610","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75518253718111135068488417046278011303","series":"1.2.826.0.1.3680043.8.498.10333322560331603498105150550729550099","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75523541256169382504121567002441615798","series":"1.2.826.0.1.3680043.8.498.46269834127609015467270657907438748873","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75577846246542841539613557336025271434","series":"1.2.826.0.1.3680043.8.498.37858349027110402156712482300293802004","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75627574440860942023321071382766359827","series":"1.2.826.0.1.3680043.8.498.19489455999041108696059183161149695708","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75670996863043899245632198549551296292","series":"1.2.826.0.1.3680043.8.498.10992579851144563451897980905004711466","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75742195884147990442829552992649204078","series":"1.2.826.0.1.3680043.8.498.19353510663897568411975540462369234511","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75751784078102748563815298069799859542","series":"1.2.826.0.1.3680043.8.498.25629711491788111298760978850310575737","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75800418125597325281807386716279941068","series":"1.2.826.0.1.3680043.8.498.13126107841302014540721530859911444701","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75924957745868212343758911856809118662","series":"1.2.826.0.1.3680043.8.498.71716043270946877942552990601049217961","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75980374267002793040832157122311396174","series":"1.2.826.0.1.3680043.8.498.41966663344817964262517489160178828985","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76027799502777251766328820405966979386","series":"1.2.826.0.1.3680043.8.498.67081809714744987222299814204281428874","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76107330932071966465869847071480517541","series":"1.2.826.0.1.3680043.8.498.12096524812156691246397432185236076914","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76125027090596913549224250979076533606","series":"1.2.826.0.1.3680043.8.498.48426726530689952421734667409070533031","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76222770668660274805993655800271653712","series":"1.2.826.0.1.3680043.8.498.77072244963833729976409350390605844287","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76344826994380951498651448092980303697","series":"1.2.826.0.1.3680043.8.498.11860779438106497940397168578618923774","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76514087677277144682713362609812138150","series":"1.2.826.0.1.3680043.8.498.91261676975645843839589532329872252512","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76538737802277241976971950690663079376","series":"1.2.826.0.1.3680043.8.498.83739669276840253909276093827204578184","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76620177268652814425535081607587091903","series":"1.2.826.0.1.3680043.8.498.74927290647005440467944858919815515366","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76662146598924492573212477881527658730","series":"1.2.826.0.1.3680043.8.498.70748244153625227727448008666820535585","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76757251008849492985513280752011229119","series":"1.2.826.0.1.3680043.8.498.11547407456161106069356660473620261730","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76862911430418639277808760660745763387","series":"1.2.826.0.1.3680043.8.498.91518098787455822841310828350510463892","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76909970028125318280995002129970165723","series":"1.2.826.0.1.3680043.8.498.81864066776056997031582376229787910612","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76950154099427419631807259601983648254","series":"1.2.826.0.1.3680043.8.498.94763594962751055616661757677768562278","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77037121474756969001683126693104350509","series":"1.2.826.0.1.3680043.8.498.42817014427462541055603085262215678501","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.77060858802024387619459645469871885412","series":"1.2.826.0.1.3680043.8.498.74280316017614256150976533990811874040","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77155817997584903787597001328868124855","series":"1.2.826.0.1.3680043.8.498.13024185018413342450410785918898978916","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77183368292104509335580564463404842046","series":"1.2.826.0.1.3680043.8.498.64959527953980504438486410868365101593","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77244685031928454051409193776790348219","series":"1.2.826.0.1.3680043.8.498.38055324680995007577016982409668386428","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77283213969268676660453937180363826917","series":"1.2.826.0.1.3680043.8.498.95142882916811441599188772974186522094","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77290477043075828139630734968573461136","series":"1.2.826.0.1.3680043.8.498.11862182417115808261363648191255568957","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77306755058630855443644908572157134062","series":"1.2.826.0.1.3680043.8.498.37517630599795813273641482286641802629","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77370119801633353780677378448761665685","series":"1.2.826.0.1.3680043.8.498.10662950094198943801989371127107938499","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77391801789875882109036004511558148658","series":"1.2.826.0.1.3680043.8.498.34124628228530377420847179929892885679","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77421709644087006134426085284321515809","series":"1.2.826.0.1.3680043.8.498.10481273038277676434540710475171443483","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77442647074650043391818472068835112806","series":"1.2.826.0.1.3680043.8.498.11771465904671161876702110989737784474","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77513348456432770689938407624768928320","series":"1.2.826.0.1.3680043.8.498.74320581611607556152341638886351871996","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77541648519018799027301684184453151471","series":"1.2.826.0.1.3680043.8.498.80066498469155292524612571279684847812","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77584925163022886785284486387921312146","series":"1.2.826.0.1.3680043.8.498.11411355690154594842109448030381722806","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77611073647079414772859496272387571059","series":"1.2.826.0.1.3680043.8.498.21499065257996154631340582415902196665","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77716260980321491689913730053309892120","series":"1.2.826.0.1.3680043.8.498.20710747438268167100545746514486530197","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77740672313593763051900703940745812581","series":"1.2.826.0.1.3680043.8.498.18796958603702754179242560413378582661","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77760453966349959552328150395676117417","series":"1.2.826.0.1.3680043.8.498.36633071235874264651358590427889895271","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77823946901144772346251781172043480398","series":"1.2.826.0.1.3680043.8.498.78026098128190533902094476149941036339","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77889800516853576877948593628420228178","series":"1.2.826.0.1.3680043.8.498.43771949025233930570308639017732401814","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.77911973519830915742655727368286737079","series":"1.2.826.0.1.3680043.8.498.10381516266852869060729203694142371144","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78012314847102407375138809615977711653","series":"1.2.826.0.1.3680043.8.498.12864130145438783289220218698412679635","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78079900750417018394321006978343945239","series":"1.2.826.0.1.3680043.8.498.40265108862142434444300410465727606265","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78119223228067919465880137763593966456","series":"1.2.826.0.1.3680043.8.498.82760066589762165956140175500320921691","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78198993378521219839327914679211902952","series":"1.2.826.0.1.3680043.8.498.11729085983490912301599739111491602632","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78282552802075313852355443512555224586","series":"1.2.826.0.1.3680043.8.498.82796636624622405898410124943286029882","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78296478954955171036523274995555716268","series":"1.2.826.0.1.3680043.8.498.36034725753785217155826748283435244535","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78465765647012043478539489340002371960","series":"1.2.826.0.1.3680043.8.498.80052493400385613698303767997832859802","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78512215519177850923279235346968676828","series":"1.2.826.0.1.3680043.8.498.84591830035101573535572853408950903979","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78574437345603052437892513165832297960","series":"1.2.826.0.1.3680043.8.498.69677556967186646291703850849372269554","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78667999747216717251638754803697959701","series":"1.2.826.0.1.3680043.8.498.19850662897046069114430818816434670688","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78685661415919658264222803849222055515","series":"1.2.826.0.1.3680043.8.498.71091880550684161536107556239035095669","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78800983923143929298359366572020370031","series":"1.2.826.0.1.3680043.8.498.23436631133559531957928875856906486490","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78812661021494224637586904573732040856","series":"1.2.826.0.1.3680043.8.498.45490330124936723092522761935501669960","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78972123731732056524948076388825106061","series":"1.2.826.0.1.3680043.8.498.70278343190042597290117533584927022671","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79077219037094025366585138112899760966","series":"1.2.826.0.1.3680043.8.498.31192935754359658533330525207459438975","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79087227780630716132147615846840128474","series":"1.2.826.0.1.3680043.8.498.85582417322455278440976689298338127361","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79122189963582241618519333445055548347","series":"1.2.826.0.1.3680043.8.498.10747597510837331028563090319082969570","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79225094031423353857449795712402404317","series":"1.2.826.0.1.3680043.8.498.56835901813451580718954890916139736898","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79247871688929479411597859576284194165","series":"1.2.826.0.1.3680043.8.498.10050957726706119832570776691164651252","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79267465462008228906382361587050826784","series":"1.2.826.0.1.3680043.8.498.84444930145606007789863222413007303353","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79290157182784080975822485697879660942","series":"1.2.826.0.1.3680043.8.498.11084846054384130226086063280550288949","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79341955397030543255969676347554165142","series":"1.2.826.0.1.3680043.8.498.65937890220283208915190874425448523983","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79551456190098389081551755482042135791","series":"1.2.826.0.1.3680043.8.498.57199670342237790773329025127516008717","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79565393323323007178523850093527250842","series":"1.2.826.0.1.3680043.8.498.95632898051489716256821477137569830075","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79636443202100988025435536780567446582","series":"1.2.826.0.1.3680043.8.498.20056031138153976553925027306885391221","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79672421957132993896448375311641998688","series":"1.2.826.0.1.3680043.8.498.12761510647193350186242263674795003589","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79741629769872115526056483024289600492","series":"1.2.826.0.1.3680043.8.498.11268175773650872982106552618964857562","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79977436705480715175480363175316375576","series":"1.2.826.0.1.3680043.8.498.11943878544195223720125728159332576721","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80058869639900582578987760750483296668","series":"1.2.826.0.1.3680043.8.498.10588917377939566997586643267371828699","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80161127293548940216292842707683784128","series":"1.2.826.0.1.3680043.8.498.66575274022096427023114096393239017305","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.80192355832283305029770183988436644845","series":"1.2.826.0.1.3680043.8.498.12482592639458698046283084140023909908","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80247974926646352443043954025450139721","series":"1.2.826.0.1.3680043.8.498.83026362127414036756153611857852660177","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80290749690597272632725356393056746299","series":"1.2.826.0.1.3680043.8.498.11093395704768549936728399414209129860","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80313580693550912024103221258669390487","series":"1.2.826.0.1.3680043.8.498.12246136327344103063305670037410584907","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80325205657974094483016625702390834774","series":"1.2.826.0.1.3680043.8.498.13440113385742052438825080774223005042","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80458507774711142534182397228254444268","series":"1.2.826.0.1.3680043.8.498.18506326813514587243205621531675125727","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.80514754572353458107286859773300865138","series":"1.2.826.0.1.3680043.8.498.92460782379645841709740457742862937768","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80589796918823112368758999678369292754","series":"1.2.826.0.1.3680043.8.498.51756568865180670831693738263063729578","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80627605463642274852766102874822833365","series":"1.2.826.0.1.3680043.8.498.25023486423727646639568227572805447997","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80643480554795312280333659144941545925","series":"1.2.826.0.1.3680043.8.498.42505400513559006205620186687750349650","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80661392139668653552110941896475066204","series":"1.2.826.0.1.3680043.8.498.13083935522965801117655424555281172808","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80726797831040227285328789648680519664","series":"1.2.826.0.1.3680043.8.498.45294569668171151388727899585328523765","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80741358365199721720335324482078866565","series":"1.2.826.0.1.3680043.8.498.80887075976294105173413099311427340875","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80861108320341696163987804408524915999","series":"1.2.826.0.1.3680043.8.498.11549414486246601493762704401929639423","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80878057166752747059190481131529937715","series":"1.2.826.0.1.3680043.8.498.76744579049961398688591923476744717329","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81012548926053584036386949658594985125","series":"1.2.826.0.1.3680043.8.498.43929694762321128245364343160788152883","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81024897463656728358703040759074583549","series":"1.2.826.0.1.3680043.8.498.22867523073088024569797963519830813605","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81103809943451790668501089584752866199","series":"1.2.826.0.1.3680043.8.498.42953918609108110048595797921064615231","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81142440700884513521007992957945568924","series":"1.2.826.0.1.3680043.8.498.49418228016389465086283707762818739263","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81327951282145199298572980264557740321","series":"1.2.826.0.1.3680043.8.498.75975212989136336141437992258735308459","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81347841116019279082226143308588516787","series":"1.2.826.0.1.3680043.8.498.97401130869258802407023740895127605747","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81407504276375197122586620633759174421","series":"1.2.826.0.1.3680043.8.498.64041724234071162940693285326987425613","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81464322239381998685473520085495957135","series":"1.2.826.0.1.3680043.8.498.55596251753981674405469877182707014407","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81495303185504163367736663647489556517","series":"1.2.826.0.1.3680043.8.498.78036487512048515187944324628335336007","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81590494282169434250974061642267472166","series":"1.2.826.0.1.3680043.8.498.13393964471939826984952526637990470018","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81631925197682464058124413934958934086","series":"1.2.826.0.1.3680043.8.498.33689799421804981666639983067373825241","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81651455914141365028015556083080638757","series":"1.2.826.0.1.3680043.8.498.26154891202521382052388114276610699798","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81738571920712657790861120957086010772","series":"1.2.826.0.1.3680043.8.498.12880424037978185735451415011968681431","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81790990482142045640192729399175106255","series":"1.2.826.0.1.3680043.8.498.11306329132962003688857114435399592921","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81797437102217585216278133168092577952","series":"1.2.826.0.1.3680043.8.498.77135962813481602391316766278504533319","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81822709772232914298414700148454511661","series":"1.2.826.0.1.3680043.8.498.11702515374651186967783159509292724692","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81927409938015158663782061022054347955","series":"1.2.826.0.1.3680043.8.498.87387960713177017295047637202695262119","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81964000393640770822199916465038759069","series":"1.2.826.0.1.3680043.8.498.11108614151446859362431203806636694098","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81981151447274200345866874172324295172","series":"1.2.826.0.1.3680043.8.498.11166039692943608546313783574423080657","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.82053853036886620816821481139413594407","series":"1.2.826.0.1.3680043.8.498.18237240739110230147459915439256045496","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82143276307898132314496735515204481508","series":"1.2.826.0.1.3680043.8.498.36547692707069600409188767497119216659","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.82166943552764439138333504456139890254","series":"1.2.826.0.1.3680043.8.498.10257912747645500412136630543476045285","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82272793832748838882752586678639564181","series":"1.2.826.0.1.3680043.8.498.49393368848911194679760795688712765548","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82414828460683289235183375457850096746","series":"1.2.826.0.1.3680043.8.498.93180075927063854796730727096356069949","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.82481056562260046738834822905709207128","series":"1.2.826.0.1.3680043.8.498.42044433244566047724871948291814347673","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82523073559792523830827952754346276051","series":"1.2.826.0.1.3680043.8.498.13360151738991191576777566606514314077","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82579118531566493365068788437906656162","series":"1.2.826.0.1.3680043.8.498.10143582080669478742449088132005011355","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.82591599611540849522609405219902904134","series":"1.2.826.0.1.3680043.8.498.64199585535974785721828534370500436305","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82744514729782879778828512295184306538","series":"1.2.826.0.1.3680043.8.498.12387134595034698802253274764339894498","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82858981777405241601610241542173548956","series":"1.2.826.0.1.3680043.8.498.10056911984646058856766964243376225790","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82925474533853765570570726554869886925","series":"1.2.826.0.1.3680043.8.498.58177959005902972223809723217788007880","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83028939839767027342482365275441078777","series":"1.2.826.0.1.3680043.8.498.11352234729049474561959965338198541746","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.83054899303871276184985881638619543362","series":"1.2.826.0.1.3680043.8.498.19228362518329494810933394899177839922","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.83108474538388106770920972885900596508","series":"1.2.826.0.1.3680043.8.498.38258710149865022330586971628784697401","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83152040259754415849193046299744699239","series":"1.2.826.0.1.3680043.8.498.12332514564499345614885277982353894196","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83237706274929081175032033288272528919","series":"1.2.826.0.1.3680043.8.498.77859867397652172947682702041396472478","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83271280040777227673434567660749430744","series":"1.2.826.0.1.3680043.8.498.17013071648340311279125105329350542073","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.83456649088980869429062498344457162631","series":"1.2.826.0.1.3680043.8.498.40768171464309765740100787764282618438","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83495448827553002152296206085110883602","series":"1.2.826.0.1.3680043.8.498.10944340202361378325569067262421989952","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.83602205818477323486169938182117122488","series":"1.2.826.0.1.3680043.8.498.33112858713567680925226487210703377974","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.83740112248863750162633901687556705057","series":"1.2.826.0.1.3680043.8.498.21299361034347693693234804653483130347","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83956129905762530953192730040209719910","series":"1.2.826.0.1.3680043.8.498.48643208686338828949676098265984820123","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.84047134932018909998648460546980598926","series":"1.2.826.0.1.3680043.8.498.12566901385310692651379835554025193445","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84062602050757262792437893691591415560","series":"1.2.826.0.1.3680043.8.498.10068759518307928574955716960374794324","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84069767112794890061709127671908414399","series":"1.2.826.0.1.3680043.8.498.50410332078734985089244201312759980236","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84230432137191939390680841389370206819","series":"1.2.826.0.1.3680043.8.498.11818753464177599632240822229721907695","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84347051904829814300575579670995018991","series":"1.2.826.0.1.3680043.8.498.10284510083934099899208428271285726780","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.84483080389344581106129884869678629612","series":"1.2.826.0.1.3680043.8.498.66570387598124574988425843587913198872","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.84612593835203397696029057940345763682","series":"1.2.826.0.1.3680043.8.498.94308264521650371420354775195246912857","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.84646554968523210021448483220679710652","series":"1.2.826.0.1.3680043.8.498.77250226918273983013886006601824740191","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84720476187201404764508451536959273676","series":"1.2.826.0.1.3680043.8.498.90989008257337682331934177908300819844","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84827027113422423894439539536275453132","series":"1.2.826.0.1.3680043.8.498.22510106959775324133556985118007172626","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84896756080952941537556818257863912066","series":"1.2.826.0.1.3680043.8.498.10815112581716407910129044433300788938","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.85004373913191467860228868467924401547","series":"1.2.826.0.1.3680043.8.498.68725701861319800818467043279957544680","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85036616529391787087213173897728060798","series":"1.2.826.0.1.3680043.8.498.16160709881243768245679045646679888708","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85137704768728261642637391657270006448","series":"1.2.826.0.1.3680043.8.498.72583908982779943074515269274543092597","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85212353344613461454490252928274988058","series":"1.2.826.0.1.3680043.8.498.20427843058379076315107495328311369746","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85333821516157983299252782678788406019","series":"1.2.826.0.1.3680043.8.498.10487289003624216548700013894542051190","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85347263026430684291021533170619978114","series":"1.2.826.0.1.3680043.8.498.13143748714212150271026710545922101017","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85416107585703628549041626609272707431","series":"1.2.826.0.1.3680043.8.498.80215639199376642839010644984277126871","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85454884397223412543050876117416146750","series":"1.2.826.0.1.3680043.8.498.26336187892891729509358333881304901977","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85475892602877115812059264719695919613","series":"1.2.826.0.1.3680043.8.498.18455026283868130134431137564260104125","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85566900782101872452970957532250214590","series":"1.2.826.0.1.3680043.8.498.30697992540418304903004738130621300282","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85624441072689653093363046226467141762","series":"1.2.826.0.1.3680043.8.498.23264327834381853551150516832788847474","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85663162480758065951465617999516770647","series":"1.2.826.0.1.3680043.8.498.25375002123472423549508987139158841477","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85715956158218184426400151797364834203","series":"1.2.826.0.1.3680043.8.498.12917936255671669542682732390945100438","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85735729179216019275685564112535593988","series":"1.2.826.0.1.3680043.8.498.12671737682498734380038302326362408463","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85760360279564672425828346507309104394","series":"1.2.826.0.1.3680043.8.498.86799217346184630916176607040613240282","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85879572979834912642178551098379145607","series":"1.2.826.0.1.3680043.8.498.58043922732742073102908198107432155035","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85883562680953255833567595509408888410","series":"1.2.826.0.1.3680043.8.498.79157154267215787714719578714290430833","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85943737587102032370951456635167782343","series":"1.2.826.0.1.3680043.8.498.37736749983920882670017881342632846085","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85976668143927055109184433855426232055","series":"1.2.826.0.1.3680043.8.498.66495562601847189130089461141940187316","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86020966816754359840982061252012700091","series":"1.2.826.0.1.3680043.8.498.75826408036332092725036860659327884107","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86042942187650835661744645315369706668","series":"1.2.826.0.1.3680043.8.498.10310337730495402385062482991832190467","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86117639434986594266345773277419881144","series":"1.2.826.0.1.3680043.8.498.10991293877741730420590250142542024504","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86239793008693746155478763153154040128","series":"1.2.826.0.1.3680043.8.498.12184668708313568058137676903746717511","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86263264208309369957750915520115390972","series":"1.2.826.0.1.3680043.8.498.42236067113834078851510594723004389101","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86280033511814602594654628677479528508","series":"1.2.826.0.1.3680043.8.498.92831160101233309784561836469873233259","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86332035495941070455209594003499488889","series":"1.2.826.0.1.3680043.8.498.54787257581258866147034143601046699491","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86351141114604781108401133410934252655","series":"1.2.826.0.1.3680043.8.498.76336692579737814590064197315169360082","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86416984885007871128151024515780322767","series":"1.2.826.0.1.3680043.8.498.30831373209361300681076983772458163716","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86435271359723045834079326733478538148","series":"1.2.826.0.1.3680043.8.498.12684827032404236097081537840562991711","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86532643576908316499388520854820002133","series":"1.2.826.0.1.3680043.8.498.10719542742951339426447456548657303544","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86583531066383395270546012510184348251","series":"1.2.826.0.1.3680043.8.498.21879147278561166181680771772603025457","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86715997235054599024919893656486735623","series":"1.2.826.0.1.3680043.8.498.33759013890081538596843036263794517108","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86876540394528304678398111135600894869","series":"1.2.826.0.1.3680043.8.498.36383389603025526418776595479619756995","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86974853023888933977090539174537083409","series":"1.2.826.0.1.3680043.8.498.25872842891458611520795225250780020533","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87018440624359538250658207994111333051","series":"1.2.826.0.1.3680043.8.498.50860453608411420537497526494827726562","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87101276883427329522365304528449003589","series":"1.2.826.0.1.3680043.8.498.59953403217579609880550459195910961102","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87185817772180109135714642184874643715","series":"1.2.826.0.1.3680043.8.498.28440420063788826557436379434608534669","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87214831397290763080243320084625847148","series":"1.2.826.0.1.3680043.8.498.84164037140591586924588849491841533433","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87223930502803944549504594723314220105","series":"1.2.826.0.1.3680043.8.498.88303089780616907740560493705653054656","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87262205711949358943515326577886508585","series":"1.2.826.0.1.3680043.8.498.11096533524356085374503425843265592090","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87404730020015145302086817782843205171","series":"1.2.826.0.1.3680043.8.498.28850472064390007219544553699610736785","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87415625824966124550466651648244553892","series":"1.2.826.0.1.3680043.8.498.10734740444286697358374662445206817474","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87476514600989622078259281213313867927","series":"1.2.826.0.1.3680043.8.498.79023207846688288220714717632826123168","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87560916423885591029120178291827033868","series":"1.2.826.0.1.3680043.8.498.12015137736990116879784221943376710737","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87645071186049036230936726742129446188","series":"1.2.826.0.1.3680043.8.498.10612495620366729920185060744350000952","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87680825247567493080163504511649789348","series":"1.2.826.0.1.3680043.8.498.18589904250714057997255095730844908138","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87761573479019057997334508266888503487","series":"1.2.826.0.1.3680043.8.498.11344387242926958155398330435722316019","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87797202230328868466634480198668365648","series":"1.2.826.0.1.3680043.8.498.63616002567537198534029979718315934495","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87876800169766592603600598167358214111","series":"1.2.826.0.1.3680043.8.498.77364410760924238625772219554159225192","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87998832540756872371878613339628615880","series":"1.2.826.0.1.3680043.8.498.33627572104831229213918688442983003359","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88064086448060280325724227239585797761","series":"1.2.826.0.1.3680043.8.498.97996301605593518037989249810257134053","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88077418639301174409926781329613570435","series":"1.2.826.0.1.3680043.8.498.10326918552454411368494543557296881553","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88211355128040927831407007101167868685","series":"1.2.826.0.1.3680043.8.498.12325301348514730248310885402792094805","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88368930449171804348215887291291274615","series":"1.2.826.0.1.3680043.8.498.15385069676650354826785882816994321903","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88466268868654468582661984391239692466","series":"1.2.826.0.1.3680043.8.498.64955293682204330582309234445604132190","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.88505044370271842163477475980178015566","series":"1.2.826.0.1.3680043.8.498.11068975592208352443550064872524582315","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88570871026022705696415758052703394453","series":"1.2.826.0.1.3680043.8.498.12566570596162310065286868004634196710","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88574487918084130981335189023288704909","series":"1.2.826.0.1.3680043.8.498.60274406271931223709165295499070253775","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88653085234639096911379842849193874077","series":"1.2.826.0.1.3680043.8.498.93891256835023917428418585186064066615","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88679328523895187500714090963655363787","series":"1.2.826.0.1.3680043.8.498.11324892745208824905414088535430058090","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88711356400820562340756013729440417015","series":"1.2.826.0.1.3680043.8.498.63084042729896574765368438900654640193","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88752039058074948027322975455607258941","series":"1.2.826.0.1.3680043.8.498.12641746587935043601248025178223749355","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88827276474694165071267273766912871459","series":"1.2.826.0.1.3680043.8.498.11442259209811063375724021362501719781","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88872155446935518174304990308380906112","series":"1.2.826.0.1.3680043.8.498.50962363226949753622288847652705843630","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88896246957673838955012057210098034177","series":"1.2.826.0.1.3680043.8.498.29201324514865661613425369374864768597","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88964371740619947193957393682095448256","series":"1.2.826.0.1.3680043.8.498.71295458033586516143961363964582788146","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89045629523485194086986785222960864883","series":"1.2.826.0.1.3680043.8.498.10738615554495651969273840697101997866","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89074359486713340585710357399403393294","series":"1.2.826.0.1.3680043.8.498.12251357981901236972978462462010487796","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89190511197564127079697151213577962649","series":"1.2.826.0.1.3680043.8.498.69301336093010744038118739180275122346","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89240184178186399632216878563608101260","series":"1.2.826.0.1.3680043.8.498.12502614539784980038999452822651388719","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89307235192406983202868051697765144920","series":"1.2.826.0.1.3680043.8.498.12398066320372604476402860670101041595","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89332108074590559691047812196991368746","series":"1.2.826.0.1.3680043.8.498.62587060935153132124869762200728811294","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89350648596674998203116057439419132072","series":"1.2.826.0.1.3680043.8.498.91766341146474400735482430464857374955","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89423600201830685256510321877396547497","series":"1.2.826.0.1.3680043.8.498.10589745320065837911291950319305115772","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89464179959474561696319051639748844244","series":"1.2.826.0.1.3680043.8.498.12015477547983192542037004053325577056","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89584627944947813918297925813120247226","series":"1.2.826.0.1.3680043.8.498.38007056445391990411118693577753641610","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89720428711655200442262503508409163392","series":"1.2.826.0.1.3680043.8.498.42571843274625662661580860672667895369","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89829843399218627892046362324057240535","series":"1.2.826.0.1.3680043.8.498.43457954885687406389039624863541799005","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89857801102910604766959864464865910585","series":"1.2.826.0.1.3680043.8.498.53768381688209024189441425568266899557","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89943815722422273305517145510644432987","series":"1.2.826.0.1.3680043.8.498.10522068282711648719452619495487462444","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89987866402625167201267936974120824712","series":"1.2.826.0.1.3680043.8.498.26734399825953161799314758283877773723","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90037507233074511524602897439257102032","series":"1.2.826.0.1.3680043.8.498.40067434011026199339239205594745973862","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90159195278632080234412316065469499332","series":"1.2.826.0.1.3680043.8.498.12536444744419917374830973158644190039","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90193654806390401129887447441469319421","series":"1.2.826.0.1.3680043.8.498.66943405321712888739772195776708329037","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90223492515368389331845681759355261030","series":"1.2.826.0.1.3680043.8.498.28774766327349854162021165446752150805","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90264300606047579088362539636050224478","series":"1.2.826.0.1.3680043.8.498.94334518868826031483858219591503243463","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90274711315692820326973277297753398634","series":"1.2.826.0.1.3680043.8.498.35120233532625957870823386713228557073","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90299615112155255774026824400461236845","series":"1.2.826.0.1.3680043.8.498.85694068865072121608550827235309941027","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90331082101665002495739626395247831768","series":"1.2.826.0.1.3680043.8.498.10287904273310059382196983779019849923","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90433488790413321102007716267340900056","series":"1.2.826.0.1.3680043.8.498.12119038963128830836301952224302332475","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90466423642678253933409966898120881010","series":"1.2.826.0.1.3680043.8.498.31571964375162057252627261836369855488","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90508461713737190527021531642732695353","series":"1.2.826.0.1.3680043.8.498.82336907833258960402552049839958466480","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90551187439998209641370107095424494085","series":"1.2.826.0.1.3680043.8.498.27202349446880739901012551875103177690","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90668646188784648688693620159738574680","series":"1.2.826.0.1.3680043.8.498.12409892482374345179487343060194274096","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90751504670572401790063555695835439317","series":"1.2.826.0.1.3680043.8.498.12852588449348473736480870680107792997","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90764713690393618577990540005126771712","series":"1.2.826.0.1.3680043.8.498.39875925137244170078812776650286887284","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90893882071188409362296099956462540506","series":"1.2.826.0.1.3680043.8.498.12022518859856516082117258153340875222","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.90985037792656682958281230441654772668","series":"1.2.826.0.1.3680043.8.498.11878906764820811885693613340273113254","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91106275065657355304239097295699332135","series":"1.2.826.0.1.3680043.8.498.34240568295613109154073649943868060464","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91125742927944703615970502025043944089","series":"1.2.826.0.1.3680043.8.498.12910498651826331031475616194753134934","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91168270990167204047014161851340054645","series":"1.2.826.0.1.3680043.8.498.61714542637260006751784138573154146673","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91220255464004258635083940797356822178","series":"1.2.826.0.1.3680043.8.498.11331030925683328530131264199274511998","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91262503278090288446189575519187653125","series":"1.2.826.0.1.3680043.8.498.30318316055303527409908554247827063178","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91328985601265188961626433080766431089","series":"1.2.826.0.1.3680043.8.498.47216529209319239766049015244210287898","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91380337141071062166571333367919885984","series":"1.2.826.0.1.3680043.8.498.12478123454414158041727684963779889726","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91391446167553025113367459399831345295","series":"1.2.826.0.1.3680043.8.498.10492934695106070473586113335057668771","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91468214041781427810665883790357021637","series":"1.2.826.0.1.3680043.8.498.13216954773180942494384925295101991672","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91581717063042044543371480238854954692","series":"1.2.826.0.1.3680043.8.498.12720446269848583994594051203781652655","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.91660278231317256353374962767408532443","series":"1.2.826.0.1.3680043.8.498.12994544507226494081806823532032282635","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91727248305411866145131034915485728037","series":"1.2.826.0.1.3680043.8.498.11419252392993469962489494636073049989","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91839696756303525362262251587171486273","series":"1.2.826.0.1.3680043.8.498.40879565942163525475165218156906150613","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91897419008563027309752122760214275182","series":"1.2.826.0.1.3680043.8.498.37073922896790395471963917242589320140","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91913894091933783677568170210064807665","series":"1.2.826.0.1.3680043.8.498.91183435538311217603173771504829522820","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91979363712708769014277114061159398016","series":"1.2.826.0.1.3680043.8.498.10861530093817007276885541128011419259","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92063841381005962285592592672311044745","series":"1.2.826.0.1.3680043.8.498.10188583795836242313580054811560915546","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92127426417720107708158713612888411025","series":"1.2.826.0.1.3680043.8.498.30454058688879663733499121814966530573","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92249560662718438332916289759028859881","series":"1.2.826.0.1.3680043.8.498.92876485138135191834565474666079081389","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92326467051364762529119362944559591665","series":"1.2.826.0.1.3680043.8.498.12235590047123467403376060342697758976","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.92407834172018110142589158281575319154","series":"1.2.826.0.1.3680043.8.498.64601143075069745747142170114061945215","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92445779773398371452908052305534240654","series":"1.2.826.0.1.3680043.8.498.10860239975665863403901480567858158955","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92581137608969847756039955144478716506","series":"1.2.826.0.1.3680043.8.498.39616911482773847334483864196956612360","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92621720172216795135762712303124932801","series":"1.2.826.0.1.3680043.8.498.63029637306701392977599281418926454476","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92650426727723557280117118587545413917","series":"1.2.826.0.1.3680043.8.498.60150369640380990349018796057122572392","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92715232057122971236003034574127234433","series":"1.2.826.0.1.3680043.8.498.56378715682471945391665834852179070610","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.92758635328468215409701857842011454608","series":"1.2.826.0.1.3680043.8.498.96880994689129267608778107193580452694","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92902692647544828045768164058889405369","series":"1.2.826.0.1.3680043.8.498.78952078195216273168934673800535151763","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93026010065162534828966791702007067882","series":"1.2.826.0.1.3680043.8.498.77409936052632731154235988904445634547","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.93096918345463865518663434763610016449","series":"1.2.826.0.1.3680043.8.498.26780190154070334554798461099341280161","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93141871723058796160969424330424882014","series":"1.2.826.0.1.3680043.8.498.17387702716052444502960188001983671593","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93188773180403601524488403580611534053","series":"1.2.826.0.1.3680043.8.498.10462861552274779462085902907133032946","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.93230313557251171119475173996942942855","series":"1.2.826.0.1.3680043.8.498.11669500280258340843308529272687106050","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93318293884671598055223778443354732793","series":"1.2.826.0.1.3680043.8.498.74351734149892909946193381827191043868","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93385433312281352893457225415812880948","series":"1.2.826.0.1.3680043.8.498.10237229023825351555210653255045815105","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93555726419626687118723083838655757151","series":"1.2.826.0.1.3680043.8.498.12693268757392525776680152427006433922","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.93655619137163635289231840294252859840","series":"1.2.826.0.1.3680043.8.498.11331806363781749357146091827556275272","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93722352425168280299901366261374293722","series":"1.2.826.0.1.3680043.8.498.81242294043273524300559462842162359854","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93747304211180517979657679136059719120","series":"1.2.826.0.1.3680043.8.498.89838218270296919569506325910081384106","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93902628119199416703899320994283313812","series":"1.2.826.0.1.3680043.8.498.20005079387053724659384022570799045034","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.93976819306514730278115540117878688308","series":"1.2.826.0.1.3680043.8.498.92326256981209525105850367169487903897","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94027547274004055924993074754507802303","series":"1.2.826.0.1.3680043.8.498.27900031312266832214759955230276924517","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94069670011071682445520414370532541612","series":"1.2.826.0.1.3680043.8.498.10583486887151541643865616228282630043","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94252884278314596961426629318911909075","series":"1.2.826.0.1.3680043.8.498.14048739529189334917456148279764342644","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94296943940827583101208235208463322140","series":"1.2.826.0.1.3680043.8.498.61705598950968884906023037241165242865","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94335780236370744781443631231792792083","series":"1.2.826.0.1.3680043.8.498.11813590797358826168071384923647519224","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94393348753936295687030819683853459606","series":"1.2.826.0.1.3680043.8.498.74054401169745871279349788573318605641","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94448720962054029575571654250994769402","series":"1.2.826.0.1.3680043.8.498.27708621160035305002517660383748422115","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94594009911808653224418094594012527944","series":"1.2.826.0.1.3680043.8.498.32095128776383222926579006484518595808","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94607524665407029611795868588964734480","series":"1.2.826.0.1.3680043.8.498.86323983109760288648631570588443443071","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94658317716039594639343437780115140531","series":"1.2.826.0.1.3680043.8.498.95163245277735548733981151135031982245","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94722363227977785887442194094888460046","series":"1.2.826.0.1.3680043.8.498.80255358491007732464688896616990144000","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94753144127920558221268610279343359449","series":"1.2.826.0.1.3680043.8.498.13195715050039922706666982192144277807","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94798723424795346945893598929864335472","series":"1.2.826.0.1.3680043.8.498.40381006719171014254712895009096245410","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94921929374845793599006362190119159537","series":"1.2.826.0.1.3680043.8.498.12537635565524327225060416566529811398","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94945670991495145462605178471390637325","series":"1.2.826.0.1.3680043.8.498.47773230262935221719479755379297098362","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95026524886584356226576936309513335214","series":"1.2.826.0.1.3680043.8.498.88902621436530968263859551428264613399","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95060380238431209646600378585422128745","series":"1.2.826.0.1.3680043.8.498.12227192362506794861154851902276096280","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95113114010658169441709314948308683982","series":"1.2.826.0.1.3680043.8.498.48329902560833288392664207520302990494","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95193282490279394602564266574551123732","series":"1.2.826.0.1.3680043.8.498.81913143601503092212103259616135739041","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95256518245659249144569318023469597946","series":"1.2.826.0.1.3680043.8.498.85946952445475531854637635468626268874","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95321750682026667237717880394536750835","series":"1.2.826.0.1.3680043.8.498.64425868387574836405235962285499746234","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95351925765761617117057799846763403963","series":"1.2.826.0.1.3680043.8.498.68547278990366168272004289944361631728","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.95433650759419050864053270236035947981","series":"1.2.826.0.1.3680043.8.498.70898761350574043294619049588986027504","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95497572581157858069692837655072209020","series":"1.2.826.0.1.3680043.8.498.52971622212873559893330660731584749635","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95555755747044090477918262640906805885","series":"1.2.826.0.1.3680043.8.498.11483677168258899170848566960067657706","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95570207430778332247200880037915693375","series":"1.2.826.0.1.3680043.8.498.55530892703542407100099851990354408530","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95601371640669355936785637162110963091","series":"1.2.826.0.1.3680043.8.498.35917065377312667007823807222555506364","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.95638601079527306640829424921455011698","series":"1.2.826.0.1.3680043.8.498.52724725573185441368489078557707456000","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95705755527332814775935856896567782522","series":"1.2.826.0.1.3680043.8.498.17838838322562066958793056051484755448","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95728724960383464194568391302477328450","series":"1.2.826.0.1.3680043.8.498.10223419801348329759168499848602665108","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95796009319232592137564145023096016872","series":"1.2.826.0.1.3680043.8.498.59924606869030669868100489238218872714","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95864678405973312042815189771902289608","series":"1.2.826.0.1.3680043.8.498.22000374963790021209598041815098752946","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95918254432056542701730614049432316728","series":"1.2.826.0.1.3680043.8.498.11507446989622545459192974104729976530","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.96054287426681123062488435620352206601","series":"1.2.826.0.1.3680043.8.498.67289155594166065369897308196837782236","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.96117880374445894729658092255088269716","series":"1.2.826.0.1.3680043.8.498.33604439583801851533129833115328570650","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96152894373838410779390571000523933753","series":"1.2.826.0.1.3680043.8.498.12524046139486476803740432065693520062","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96293531288862652713666056362328893428","series":"1.2.826.0.1.3680043.8.498.12188976291649736560844977088537160560","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96323485839061204012967074568679195142","series":"1.2.826.0.1.3680043.8.498.52975632608437116858271260721328626493","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96483513352751590579274577125356217169","series":"1.2.826.0.1.3680043.8.498.14911469069401190028189389696914459601","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96559819562317669526803257835398470484","series":"1.2.826.0.1.3680043.8.498.11771964033775692989829371057701639364","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96564709711765982786409987515735443801","series":"1.2.826.0.1.3680043.8.498.11580514061851868515280886976091860954","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96614828054790815108190478795539254570","series":"1.2.826.0.1.3680043.8.498.87996768261527369042819027522754992646","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96791010280967772394523567484937888186","series":"1.2.826.0.1.3680043.8.498.35897781428610932417951277677923504378","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96846796615932368166905347054245013501","series":"1.2.826.0.1.3680043.8.498.43072334231375784974622543881256743862","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96944926470854551064948200936284887742","series":"1.2.826.0.1.3680043.8.498.11943702781442335298324980564801849425","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96950011369238410949579095568317214261","series":"1.2.826.0.1.3680043.8.498.97090483887946466855462791848704357716","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97020520267554584684160936968569132254","series":"1.2.826.0.1.3680043.8.498.40948537314701794163447318727607064471","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.97055845147740306780908051236353219644","series":"1.2.826.0.1.3680043.8.498.16249121177238417008267739907938555632","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97135552871905937467775630206570358085","series":"1.2.826.0.1.3680043.8.498.95334590582655798992228516336719312305","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.97234348688223610934999923021953923903","series":"1.2.826.0.1.3680043.8.498.12140089643557337382050968574358024261","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97259766233250760343109933426776941754","series":"1.2.826.0.1.3680043.8.498.76766139843939510386288040782837629682","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97274720257634584071500649275217521662","series":"1.2.826.0.1.3680043.8.498.92909718893560498250889678421934340850","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97303893645051596044169857585815133097","series":"1.2.826.0.1.3680043.8.498.89474065123862757820053428811522971164","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97400307568415006244714024769069514497","series":"1.2.826.0.1.3680043.8.498.10329982709663049723290741517468026687","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97485476951995325021464683747847126869","series":"1.2.826.0.1.3680043.8.498.77602249307980609671518391501178702118","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97496319463700021339637597270948309369","series":"1.2.826.0.1.3680043.8.498.12390557107074601054949108332059463306","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97536230315600409983600607358927775480","series":"1.2.826.0.1.3680043.8.498.61076905898886879356733937498378580350","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97579738268290461886850261405719501384","series":"1.2.826.0.1.3680043.8.498.31707842424523736647918356189120437541","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97689748704772735695349883980413529091","series":"1.2.826.0.1.3680043.8.498.17849635286336272721484421402445446380","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97696436660690920809826140686268769509","series":"1.2.826.0.1.3680043.8.498.65372813058816598652981625894812173133","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97731968872354496470096174559789586056","series":"1.2.826.0.1.3680043.8.498.10480598902269102194793415984550182138","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97830923192341890652952842494018952920","series":"1.2.826.0.1.3680043.8.498.50760913717752277603383565319073382718","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97944261713902372188894456162512065499","series":"1.2.826.0.1.3680043.8.498.94952391341613100274561311787243427144","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98022738778461179495257509215244053198","series":"1.2.826.0.1.3680043.8.498.95230572059739717778825860265167252625","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98028018520101798863795620289151810642","series":"1.2.826.0.1.3680043.8.498.10007238824192154774338796293608787949","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98081486779628580946589119341754353354","series":"1.2.826.0.1.3680043.8.498.10675069614001338061474015629930962252","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98107112397588450345153336193341117589","series":"1.2.826.0.1.3680043.8.498.76214928423226389474420438461539908908","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98146155044178593810385943365143374814","series":"1.2.826.0.1.3680043.8.498.24369221689960904265392044201717032503","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98181040478767526925214101502401448074","series":"1.2.826.0.1.3680043.8.498.10302285978260813179259685440784145651","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98255454187594895604610283922744869720","series":"1.2.826.0.1.3680043.8.498.36533637428145850361701572294871351322","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98343185597651455245996667389303915779","series":"1.2.826.0.1.3680043.8.498.12552521993950691720845072107469257294","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98400072082366198211931728015692551635","series":"1.2.826.0.1.3680043.8.498.10211064535249799311312481891082682039","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98541661035340329000785741171799559104","series":"1.2.826.0.1.3680043.8.498.10305911785606743174477254639064677682","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98676079331571304021387833410440136393","series":"1.2.826.0.1.3680043.8.498.91132362994230440055419142012417114120","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.98781117432577714521231232440503778261","series":"1.2.826.0.1.3680043.8.498.54477836812200948029283139446614584038","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98824066826946884510314135448574304233","series":"1.2.826.0.1.3680043.8.498.47111351312581112918495089241188013799","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98883718766501717651651876978607853803","series":"1.2.826.0.1.3680043.8.498.34826478144533389669765917265995583157","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98914528123216365374152816424181510781","series":"1.2.826.0.1.3680043.8.498.11635002551691881354524444251146559471","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99077241529686301278201179461896593659","series":"1.2.826.0.1.3680043.8.498.12976264252535354068376587854559718055","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.99108147913292332418681050058312852494","series":"1.2.826.0.1.3680043.8.498.73471697634001716106178426613812784656","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99237666579371266866432031468578336182","series":"1.2.826.0.1.3680043.8.498.85303304220241979260397593175937268377","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99303079292320085374389086609761974820","series":"1.2.826.0.1.3680043.8.498.94282168265164672922193172760791728924","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99413398303724030328059886749330413303","series":"1.2.826.0.1.3680043.8.498.13329668906682170836591446597164105100","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99441773159476751698123795913150660766","series":"1.2.826.0.1.3680043.8.498.12349296235158026277043011655097410744","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99448612525108835020450967944392129536","series":"1.2.826.0.1.3680043.8.498.63045660500409978193936897248497748366","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99493257395588095270208592176887619190","series":"1.2.826.0.1.3680043.8.498.87755678221224118916955956056613583766","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.99577045720075471058329330051900767206","series":"1.2.826.0.1.3680043.8.498.50119154113720113688072955806113931765","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99651725851820083142440965238449655973","series":"1.2.826.0.1.3680043.8.498.10436847332245692116065840429873897831","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99725523560254732094432244556623376711","series":"1.2.826.0.1.3680043.8.498.55316725501309459145680357156167637579","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99768154649604643975127490783989114506","series":"1.2.826.0.1.3680043.8.498.21045528243146351718412097869011750587","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99827460918477409069820779642161448917","series":"1.2.826.0.1.3680043.8.498.97277894371823332098253770637560755024","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99864558747514747686151806909794288262","series":"1.2.826.0.1.3680043.8.498.10651063225290069440597914902724437752","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99926624968240681772735102303531613209","series":"1.2.826.0.1.3680043.8.498.40678480454302526362872555318918248500","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99957611207313098696146908027229590966","series":"1.2.826.0.1.3680043.8.498.32916764040885408297619051929589831373","plane":"Sagittal","fluid":0,"slot":"SAG_T1"}]')
print('series 320 rgb, UID figes depuis crop130_rgb25d :', len(jobs))
assert len(jobs) == 1778
assert jobs and {'study', 'series', 'plane', 'fluid', 'slot'} <= set(jobs[0])


In [ ]:

def _sort_key(ds) -> float:
    inst = getattr(ds, "InstanceNumber", None)
    if inst is not None:
        try:
            return float(inst)
        except (TypeError, ValueError):
            pass
    ipp = getattr(ds, "ImagePositionPatient", None)
    iop = getattr(ds, "ImageOrientationPatient", None)
    if ipp is not None and iop is not None:
        row = np.array(iop[:3], dtype=np.float64)
        col = np.array(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        return float(np.dot(np.array(ipp, dtype=np.float64), normal))
    if ipp is not None:
        return float(ipp[2])
    return 0.0


def _spacing_mm(ds):
    ps = getattr(ds, "PixelSpacing", None) or getattr(ds, "ImagerPixelSpacing", None)
    if ps is None:
        return None
    try:
        return float(ps[0]), float(ps[1])
    except (TypeError, ValueError, IndexError):
        return None


def _center_crop(img: np.ndarray, row_mm, col_mm) -> tuple[np.ndarray, str]:
    h, w = img.shape[:2]
    if row_mm is not None and col_mm is not None and row_mm > 0 and col_mm > 0:
        ch = int(round(CROP_MM / row_mm))
        cw = int(round(CROP_MM / col_mm))
        ch = min(h, max(8, ch))
        cw = min(w, max(8, cw))
        mode = "mm"
    else:
        ch = max(8, int(round(h * CROP_REL)))
        cw = max(8, int(round(w * CROP_REL)))
        ch, cw = min(h, ch), min(w, cw)
        mode = "rel"
    y0 = (h - ch) // 2
    x0 = (w - cw) // 2
    return img[y0:y0 + ch, x0:x0 + cw], mode


def _rescale(ds, arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32, copy=False)
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    arr = arr * slope + intercept
    if getattr(ds, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    return arr


def _to_u8(sl: np.ndarray) -> np.ndarray:
    sl = cv2.resize(sl, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return np.clip(sl * 255.0, 0, 255).round().astype(np.uint8)


def convert_series(series_dir: Path) -> tuple[np.ndarray, dict]:
    files = sorted(series_dir.glob("*.dcm"))
    if not files:
        raise FileNotFoundError(f"pas de .dcm dans {series_dir}")
    datasets = []
    for f in files:
        try:
            datasets.append(pydicom.dcmread(str(f), stop_before_pixels=False))
        except Exception:
            continue
    if not datasets:
        raise RuntimeError("aucun DICOM lisible")
    datasets.sort(key=_sort_key)

    frames = []
    sp = None
    for ds in datasets:
        arr = ds.pixel_array
        if sp is None:
            sp = _spacing_mm(ds)
        arr = _rescale(ds, arr)
        if arr.ndim == 2:
            frames.append(arr)
        elif arr.ndim == 3:
            frames.extend([arr[i] for i in range(arr.shape[0])])
        else:
            raise RuntimeError(f"ndim inattendu {arr.ndim}")
    if not frames:
        raise RuntimeError("aucune coupe")

    row_mm, col_mm = (sp if sp is not None else (None, None))
    cropped, crop_mode = [], None
    for im in frames:
        c, crop_mode = _center_crop(im, row_mm, col_mm)
        cropped.append(c)
    h0, w0 = cropped[0].shape[:2]
    stacked = []
    for c in cropped:
        if c.shape[:2] != (h0, w0):
            c = cv2.resize(c, (w0, h0), interpolation=cv2.INTER_AREA)
        stacked.append(c)
    vol = np.stack(stacked, axis=0)  # (n, h, w) volume ORIGINAL trie

    lo, hi = np.percentile(vol, 1.0), np.percentile(vol, 99.0)
    if hi <= lo:
        hi = lo + 1.0
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    n = vol.shape[0]
    a = int(np.floor(BAND[0] * (n - 1)))
    b = int(np.ceil(BAND[1] * (n - 1)))
    if b - a + 1 < 8:
        a, b = 0, n - 1
    # memes 24 centres que crop130 (linspace dans la bande),
    # mais voisins ±1 dans le volume original, pas dans les 24
    idx_band = np.linspace(0, b - a, N_SLICES).round().astype(int)
    idx_orig = np.clip(a + idx_band, 0, n - 1)

    out = np.empty((N_SLICES, 3, IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    last = n - 1
    n_clamp = 0
    for i, k in enumerate(idx_orig):
        k = int(k)
        km = 0 if k <= 0 else k - 1
        kp = last if k >= last else k + 1
        if km == k or kp == k:
            n_clamp += 1
        out[i, 0] = _to_u8(vol[km])
        out[i, 1] = _to_u8(vol[k])
        out[i, 2] = _to_u8(vol[kp])

    meta = {
        "n_raw": int(n),
        "band": [a, b],
        "crop_mode": crop_mode,
        "spacing_mm": None if sp is None else [float(sp[0]), float(sp[1])],
        "src_hw": [int(h0), int(w0)],
        "n_files": len(datasets),
        "n_clamp": int(n_clamp),
        "rgb": "dicom_neighbors",
    }
    return out, meta


def series_path(study_uid: str, series_uid: str) -> Path:
    p = TRAIN_SERIES_DIR / study_uid / series_uid
    if p.exists():
        return p
    p2 = TRAIN_SERIES_DIR / series_uid
    return p2


In [ ]:

import zipfile

zip_path = OUT_DIR / f"volumes_crop130_rgb25d_320_part{SHARD_ID}.zip"
manifest_path = OUT_DIR / f"manifest_rgb25d_320_part{SHARD_ID}.csv"
recipe_path = OUT_DIR / "RECIPE.json"

recipe = {
    "crop_mm": CROP_MM,
    "crop_rel": CROP_REL,
    "band": list(BAND),
    "n_slices": N_SLICES,
    "img_size": IMG_SIZE,
    "planes": PLANES,
    "shard_id": SHARD_ID,
    "n_shards": N_SHARDS,
    "n_jobs": len(jobs),
    "shape": [N_SLICES, 3, IMG_SIZE, IMG_SIZE],
    "rgb": "dicom_neighbors",
    "note": (
        "1 serie fluid-prioritaire par plan. Crop 130 mm (PixelSpacing) sinon "
        "centre 70 %. Bande 20-80 %, linspace 24 centres. Pour chaque centre k "
        "dans le volume ORIGINAL : R=k-1, G=k, B=k+1 (1 coupe DICOM d'ecart, "
        "clamp aux bords). Resize 320, uint8. PAS les voisins des 24 linspace."
    ),
}
recipe_path.write_text(json.dumps(recipe, indent=2), encoding="utf-8")
print(json.dumps(recipe, indent=2))

def _one(job):
    sid = job["series"]
    t0 = time.time()
    try:
        p = series_path(job["study"], sid)
        vol, meta = convert_series(p)
        buf = io.BytesIO()
        np.savez_compressed(buf, data=vol)
        payload = buf.getvalue()
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 1,
            "error": "",
            "nbytes": len(payload),
            "dt": round(time.time() - t0, 3),
            **meta,
        }
        return sid, payload, rec
    except Exception as e:
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 0,
            "error": f"{type(e).__name__}: {e}"[:300],
            "nbytes": 0,
            "dt": round(time.time() - t0, 3),
            "n_raw": None, "band": None, "crop_mode": None,
            "spacing_mm": None, "src_hw": None, "n_files": None,
            "n_clamp": None, "rgb": None,
        }
        return sid, None, rec

t_all = time.time()
rows = []
n_ok = n_fail = 0
bytes_sum = 0

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = [ex.submit(_one, job) for job in jobs]
        for i, fut in enumerate(as_completed(futs), 1):
            sid, payload, rec = fut.result()
            rows.append(rec)
            if payload is not None:
                zf.writestr(f"{sid}.npz", payload)
                n_ok += 1
                bytes_sum += rec["nbytes"]
            else:
                n_fail += 1
            if i % 25 == 0 or i == len(jobs):
                elapsed = time.time() - t_all
                rate = i / max(elapsed, 1e-6)
                eta = (len(jobs) - i) / max(rate, 1e-6)
                print(
                    f"[{i}/{len(jobs)}] ok={n_ok} fail={n_fail} "
                    f"zip={bytes_sum/1e9:.2f}GB  {rate:.2f} s/serie  ETA {eta/60:.0f} min",
                    flush=True,
                )
                pd.DataFrame(rows).to_csv(manifest_path, index=False)

pd.DataFrame(rows).to_csv(manifest_path, index=False)
print("termine", zip_path, "size", zip_path.stat().st_size / 1e9, "GB")
print("ok", n_ok, "fail", n_fail, "en", (time.time() - t_all) / 60, "min")
print("manifest", manifest_path)
if n_fail:
    fails = [r for r in rows if not r["ok"]]
    print("exemples d'erreurs :")
    for r in fails[:8]:
        print(" ", r["SeriesInstanceUID"], r["error"])


In [ ]:

import zipfile as zfmod
with zfmod.ZipFile(zip_path) as zf:
    names = [n for n in zf.namelist() if n.endswith(".npz")]
    print("npz dans le zip :", len(names))
    for name in names[:3]:
        with zf.open(name) as f:
            vol = np.load(io.BytesIO(f.read()))["data"]
        assert vol.shape == (N_SLICES, 3, IMG_SIZE, IMG_SIZE), vol.shape
        assert vol.dtype == np.uint8, vol.dtype
        rg = np.abs(vol[:, 0].astype(np.float32) - vol[:, 1].astype(np.float32)).mean()
        bg = np.abs(vol[:, 2].astype(np.float32) - vol[:, 1].astype(np.float32)).mean()
        print(
            name, vol.shape, vol.dtype,
            "min", int(vol.min()), "max", int(vol.max()),
            "mean", float(vol.mean()),
            "|R-G|", round(float(rg), 2), "|B-G|", round(float(bg), 2),
        )
print("attendu : shape (24, 3, 320, 320), |R-G| et |B-G| > 0 (voisins DICOM reels)")
print("fichiers working :", sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))
